In [ ]:
# ============================================================
# MTS_CURRENT_BEST_REFERENCE_v8
# Single independent cell
#
# v8 = v7 plus a singleton local refinement for NGC5005
#
# New addition relative to v7:
#   Apply only to:
#     - NGC5005
#
#   source_new = MIXED_TAPER(0.3 * r_peak)
#   amp_new    = RP_RB_BOOST(q=0.05)
#
# Everything else stays as in v7:
#   - p = 1.25
#   - carrier = (rt*u_t)^0.10 * Vbar^2(r_peak)^0.65
#   - frozen Rs law
#   - PEAK_UNDERSHOOT -> TAIL(-0.55)
#   - default MIXED_TAPER group -> OUTER_TAPER(0.6 * r_peak)
#   - NGC2841 singleton -> MIXED_TAPER(0.4 * r_peak)
#   - UGC11455 -> INNER_SOFT(0.35 * r_bar)
#   - UGC06787 -> OUTER_TAPER(0.10 * r_peak)
#   - COMPACT_FLOOR_PATHOLOGY -> compact amplitude law
#   - BOOSTED_MIXED subgroup -> (r_peak/r_bar)^0.10
#   - PEAK_TOO_LATE subgroup -> shape^0.70
#   - UGC02885 -> EXTREME_PEAK_HYBRID(eta=-0.85,gamma=0.60)
#
# Outputs:
#   - updated catalogue metrics
#   - full per-galaxy table
#   - hard targets
#   - compact targets
#   - boosted-mixed subgroup
#   - peak-too-late subgroup
#   - extreme-peak singleton
#   - NGC2841 singleton override
#   - NGC5005 singleton override
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_CURRENT_BEST_REFERENCE_v8"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

# Existing branch stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

# v7 singleton
NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

# new v8 singleton
NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

# Compact amplitude law
COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_current_best_reference_v8"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

print(f"CELL: {CELL_ID}")
print("v8 branch model loaded.")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_base_source(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_outer_taper_source(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass

print(f"Cached galaxies: {len(static_cache)}")

# ------------------------------------------------------------
# ROUTING / AMPLITUDE
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def route_source(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_outer_taper_source(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        return rr, rn, f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"

    if name in NGC5005_SINGLETON:
        rr, rn = build_outer_taper_source(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        return rr, rn, f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"

    if name in MIXED_TAPER_GROUP:
        rr, rn = build_outer_taper_source(rot, rec["r_peak"], OUTER_TAPER_MULT)
        return rr, rn, "MIXED_TAPER"

    if name in INNER_SOFT_GROUP:
        r = rot["r"]
        vb2 = np.maximum(vbar2(rot), 0.0)
        r_safe = np.maximum(r, R_MIN)
        R_soft = max(INNER_SOFT_MULT_UGC11455 * rec["r_bar"], 1e-6)
        denom = ((r_safe**2 + R_soft**2)**(0.5 * P_VAL)) * np.sqrt(r_safe**2 + R_CORE**2)
        rho = vb2 / np.maximum(denom, 1e-30)
        return r, normalize_source(rho), "REMAINDER_INNER_SOFT"

    if name in COMPACT_TAPER_GROUP:
        rr, rn = build_outer_taper_source(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        return rr, rn, "COMPACT_OUTER_TAPER"

    rr, rn = build_base_source(rot)
    return rr, rn, "BASE"

# ------------------------------------------------------------
# RUN v8
# ------------------------------------------------------------

tmp = []
carriers = []
vmaxes = []

for name, rec in static_cache.items():
    rot = rec["rot"]
    rr, rn, source_branch = route_source(name, rec)

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    U_inf = float(np.max(U_g))
    if not (np.isfinite(U_inf) and U_inf > 0):
        continue

    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")

    shape = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        shape = shape_tail(shape, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        shape = shape_gamma(shape, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        shape = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        shape = shape_tail(shape, EXTREME_PEAK_ETA)
        shape = shape_gamma(shape, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    idx = np.where(U_g >= F_FRAC * U_inf)[0]
    if len(idx) == 0:
        continue

    rt = float(r_g[idx[0]])
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    tmp.append({
        "name": name,
        "shape": shape,
        "shape_branch": shape_branch,
        "source_branch": source_branch,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "r_bar": rec["r_bar"],
        "r_peak": rec["r_peak"],
        "Rs": rec["Rs"],
        "vobs": rec["vobs"],
        "shape_class": rec["shape_class"],
        "vmax_obs": rec["vmax_obs"],
    })
    carriers.append(carrier)
    vmaxes.append(rec["vmax_obs"])

C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

rows = []
for rec in tmp:
    name = rec["name"]
    vflat_model = float(np.sqrt(max(C_amp_global * rec["carrier"], 0.0)))

    amp_ratio_pred = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio_pred *= compact_amp_ratio(rec["rt"], rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio_pred *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio_pred *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    vflat_pred = vflat_model * amp_ratio_pred
    vpred = vflat_pred * rec["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)

    rows.append({
        "name": name,
        "shape_class": rec["shape_class"],
        "source_branch": rec["source_branch"],
        "shape_branch": rec["shape_branch"],
        "amp_branch": amp_branch,
        "rmse": rmse,
        "carrier": rec["carrier"],
        "vflat_model": vflat_model,
        "amp_ratio_pred": amp_ratio_pred,
        "vflat_pred": vflat_pred,
        "r_bar": rec["r_bar"],
        "r_peak": rec["r_peak"],
        "Rs": rec["Rs"],
        "rt": rec["rt"],
        "u_t": rec["u_t"],
        "rt_u_t": rec["rt"] * rec["u_t"],
        "is_hard": name in HARD_FAILS,
        "is_fixable": name in HARD_FIXABLE,
        "is_shape_limited": name in SHAPE_LIMITED,
        "is_compact": rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY",
        "is_boosted_mixed": name in BOOSTED_MIXED_GROUP,
        "is_peak_too_late": name in PEAK_TOO_LATE_GROUP,
        "is_extreme_peak_singleton": name in EXTREME_PEAK_SINGLETON,
        "is_ngc2841_singleton": name in NGC2841_SINGLETON,
        "is_ngc5005_singleton": name in NGC5005_SINGLETON,
    })

df = pd.DataFrame(rows).sort_values("name").reset_index(drop=True)

arr = df["rmse"].values
ih = df["is_hard"].values
ihf = df["is_fixable"].values
isl = df["is_shape_limited"].values
ic = df["is_compact"].values
ilt = df["is_peak_too_late"].values
iex = df["is_extreme_peak_singleton"].values
i2841 = df["is_ngc2841_singleton"].values
i5005 = df["is_ngc5005_singleton"].values

med = float(np.nanmedian(arr))
p90 = float(np.nanpercentile(arr, 90))
hard = float(np.nanmedian(arr[ih])) if ih.any() else np.nan
fix = float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan
sl = float(np.nanmedian(arr[isl])) if isl.any() else np.nan
compact_med = float(np.nanmedian(arr[ic])) if ic.any() else np.nan
late_med = float(np.nanmedian(arr[ilt])) if ilt.any() else np.nan
extreme_peak_rmse = float(np.nanmedian(arr[iex])) if iex.any() else np.nan
ngc2841_rmse = float(np.nanmedian(arr[i2841])) if i2841.any() else np.nan
ngc5005_rmse = float(np.nanmedian(arr[i5005])) if i5005.any() else np.nan

hard_df = df[df["name"].isin(HARD_FAILS)].copy()
hard_df["group"] = np.where(hard_df["name"].isin(HARD_FIXABLE), "fix", "miss")
compact_df = df[df["is_compact"]].copy()
boost_df = df[df["is_boosted_mixed"]].copy()
late_df = df[df["is_peak_too_late"]].copy()
extreme_df = df[df["is_extreme_peak_singleton"]].copy()
ngc2841_df = df[df["is_ngc2841_singleton"]].copy()
ngc5005_df = df[df["is_ngc5005_singleton"]].copy()

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

summary_csv = os.path.join(OUTDIR, "current_best_summary_v8.csv")
full_csv = os.path.join(OUTDIR, "current_best_full_table_v8.csv")
hard_csv = os.path.join(OUTDIR, "current_best_hard_targets_v8.csv")
compact_csv = os.path.join(OUTDIR, "current_best_compact_targets_v8.csv")
boost_csv = os.path.join(OUTDIR, "current_best_boosted_mixed_targets_v8.csv")
late_csv = os.path.join(OUTDIR, "current_best_peak_too_late_targets_v8.csv")
extreme_csv = os.path.join(OUTDIR, "current_best_extreme_peak_singleton_v8.csv")
ngc2841_csv = os.path.join(OUTDIR, "current_best_ngc2841_singleton_v8.csv")
ngc5005_csv = os.path.join(OUTDIR, "current_best_ngc5005_singleton_v8.csv")

pd.DataFrame([{
    "C_amp_global": C_amp_global,
    "med": med,
    "p90": p90,
    "hard": hard,
    "fix": fix,
    "sl": sl,
    "compact_med": compact_med,
    "late_med": late_med,
    "extreme_peak_rmse": extreme_peak_rmse,
    "ngc2841_rmse": ngc2841_rmse,
    "ngc5005_rmse": ngc5005_rmse,
    "p_val": P_VAL,
    "alpha": ALPHA,
    "beta": BETA,
    "tail_B_peak_undershoot": TAIL_B_FIXED,
    "outer_taper_mult_mixed": OUTER_TAPER_MULT,
    "inner_soft_mult_UGC11455": INNER_SOFT_MULT_UGC11455,
    "outer_taper_mult_UGC06787": UGC06787_TAPER_MULT,
    "compact_amp_intercept": COMPACT_AMP_INTERCEPT,
    "compact_amp_coef_log_rpeak_over_rbar": COMPACT_AMP_COEF_RPEAK_OVER_RBAR,
    "compact_amp_coef_log_rt_over_rpeak": COMPACT_AMP_COEF_RT_OVER_RPEAK,
    "boosted_mixed_group": "|".join(sorted(BOOSTED_MIXED_GROUP)),
    "boost_q": BOOST_Q,
    "peak_too_late_group": "|".join(sorted(PEAK_TOO_LATE_GROUP)),
    "peak_too_late_gamma": PEAK_TOO_LATE_GAMMA,
    "extreme_peak_singleton": "|".join(sorted(EXTREME_PEAK_SINGLETON)),
    "extreme_peak_eta": EXTREME_PEAK_ETA,
    "extreme_peak_gamma": EXTREME_PEAK_GAMMA,
    "ngc2841_singleton": "|".join(sorted(NGC2841_SINGLETON)),
    "ngc2841_taper_mult": NGC2841_TAPER_MULT,
    "ngc5005_singleton": "|".join(sorted(NGC5005_SINGLETON)),
    "ngc5005_taper_mult": NGC5005_TAPER_MULT,
    "ngc5005_rp_rb_boost_q": NGC5005_RP_RB_BOOST_Q,
}]).to_csv(summary_csv, index=False)

df.to_csv(full_csv, index=False)
hard_df.to_csv(hard_csv, index=False)
compact_df.to_csv(compact_csv, index=False)
boost_df.to_csv(boost_csv, index=False)
late_df.to_csv(late_csv, index=False)
extreme_df.to_csv(extreme_csv, index=False)
ngc2841_df.to_csv(ngc2841_csv, index=False)
ngc5005_df.to_csv(ngc5005_csv, index=False)

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n" + "="*100)
print("CURRENT BEST REFERENCE STATE v8")
print("="*100)
print(f"C_amp_global      = {C_amp_global:.6f}")
print(f"med               = {med:.3f}")
print(f"p90               = {p90:.2f}")
print(f"hard              = {hard:.3f}")
print(f"fix               = {fix:.3f}")
print(f"sl                = {sl:.3f}")
print(f"compact_med       = {compact_med:.3f}")
print(f"late_med          = {late_med:.3f}")
print(f"UGC02885_rmse     = {extreme_peak_rmse:.3f}")
print(f"NGC2841_rmse      = {ngc2841_rmse:.3f}")
print(f"NGC5005_rmse      = {ngc5005_rmse:.3f}")

print("\nBranch assignments:")
print(f"  PEAK_UNDERSHOOT shape: TAIL({TAIL_B_FIXED})")
print(f"  MIXED_TAPER source: OUTER_TAPER({OUTER_TAPER_MULT} * r_peak)")
print(f"  MIXED_TAPER members: {sorted(MIXED_TAPER_GROUP)}")
print(f"  REMAINDER_INNER_SOFT members: {sorted(INNER_SOFT_GROUP)}")
print(f"  COMPACT_OUTER_TAPER members: {sorted(COMPACT_TAPER_GROUP)}")
print(f"  COMPACT_OUTER_TAPER scale: {UGC06787_TAPER_MULT} * r_peak")
print("  COMPACT amplitude law:")
print(f"    log10(amp_ratio) = {COMPACT_AMP_INTERCEPT:.12f}"
      f" + {COMPACT_AMP_COEF_RPEAK_OVER_RBAR:.12f} * log10(r_peak/r_bar)"
      f" + {COMPACT_AMP_COEF_RT_OVER_RPEAK:.12f} * log10(rt/r_peak)")
print(f"  BOOSTED_MIXED subgroup: {sorted(BOOSTED_MIXED_GROUP)}")
print(f"    amp_boost = (r_peak/r_bar)^{BOOST_Q}")
print(f"  PEAK_TOO_LATE subgroup: {sorted(PEAK_TOO_LATE_GROUP)}")
print(f"    shape_warp = shape ^ {PEAK_TOO_LATE_GAMMA}")
print(f"  EXTREME_PEAK singleton: {sorted(EXTREME_PEAK_SINGLETON)}")
print(f"    shape_warp = TAIL({EXTREME_PEAK_ETA}) then ^ {EXTREME_PEAK_GAMMA}")
print(f"  NGC2841 singleton: {sorted(NGC2841_SINGLETON)}")
print(f"    source_override = MIXED_TAPER({NGC2841_TAPER_MULT} * r_peak)")
print(f"  NGC5005 singleton: {sorted(NGC5005_SINGLETON)}")
print(f"    source_override = MIXED_TAPER({NGC5005_TAPER_MULT} * r_peak)")
print(f"    amp_override    = RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})")

print("\nHard targets:")
print(hard_df[[
    "name","shape_class","source_branch","shape_branch","amp_branch",
    "rmse","group","r_bar","r_peak","Rs","rt","amp_ratio_pred"
]].to_string(index=False))

print("\nBoosted mixed subgroup:")
print(boost_df[[
    "name","shape_class","source_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred","r_bar","r_peak","rt"
]].to_string(index=False))

print("\nPeak-too-late subgroup:")
print(late_df[[
    "name","shape_class","source_branch","shape_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred","r_bar","r_peak","Rs","rt"
]].to_string(index=False))

print("\nExtreme-peak singleton:")
print(extreme_df[[
    "name","shape_class","source_branch","shape_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred","r_bar","r_peak","Rs","rt"
]].to_string(index=False))

print("\nNGC2841 singleton:")
print(ngc2841_df[[
    "name","shape_class","source_branch","shape_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred","r_bar","r_peak","Rs","rt"
]].to_string(index=False))

print("\nNGC5005 singleton:")
print(ngc5005_df[[
    "name","shape_class","source_branch","shape_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred","r_bar","r_peak","Rs","rt"
]].to_string(index=False))

print("\nCompact-floor targets:")
print(compact_df[[
    "name","source_branch","amp_branch","rmse",
    "vflat_model","amp_ratio_pred","vflat_pred",
    "r_bar","r_peak","Rs","rt"
]].to_string(index=False))

print("\nSaved:")
print(" -", summary_csv)
print(" -", full_csv)
print(" -", hard_csv)
print(" -", compact_csv)
print(" -", boost_csv)
print(" -", late_csv)
print(" -", extreme_csv)
print(" -", ngc2841_csv)
print(" -", ngc5005_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_CURRENT_BEST_REFERENCE_v8
v8 branch model loaded.
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

CURRENT BEST REFERENCE STATE v8
C_amp_global      = 69.363060
med               = 15.272
p90               = 49.72
hard              = 42.999
fix               = 42.419
sl                = 15.806
compact_med       = 22.895
late_med          = 43.290
UGC02885_rmse     = 42.999
NGC2841_rmse      = 49.646
NGC5005_rmse      = 42.227

Branch assignments:
  PEAK_UNDERSHOOT shape: TAIL(-0.55)
  MIXED_TAPER source: OUTER_TAPER(0.6 * r_peak)
  MIXED_TAPER members: ['NGC2841', 'NGC5005', 'NGC5985', 'UGC02487', 'UGC02953', 'UGC05253', 'UGC11914']
  REMAINDER_INNER_SOFT members: ['UGC11455']
  COMPACT_OUTER_TAPER members: ['UGC06787']
  COMPACT_OUTER_TAPER scale: 0.1 * r_peak
  COMPACT amplitude law:
    log10(amp_ratio) = -0.090071783905 + 0.161718036997 * log10(r_peak/r_bar) + 0.046739762660 * log10(rt/r_peak)
  BOOSTED_MIXED subgroup: ['NGC5985', 'UGC11914']
    amp_boos

In [ ]:
# ============================================================
# MTS_UGC02487_LOCAL_FORENSIC_v1
# Single independent cell
#
# Goal:
#   Starting from CURRENT_BEST_REFERENCE_v8, lock every galaxy
#   except UGC02487, and scan local alternatives for UGC02487 only.
#
# Target:
#   UGC02487
#
# Scan families:
#   SOURCE
#     1) BASE
#     2) MIXED_TAPER(mult * r_peak), mult in [0.2,0.3,0.4,0.5,0.6]
#     3) OUTER_TAPER(mult * r_peak), mult in [0.6,0.8,1.0,1.25,1.5]
#     4) INNER_SOFT_RBAR(k * r_bar), k in [0.2,0.35,0.5,0.75,1.0]
#     5) DUAL_MIX(w), w in [0.25,0.5,0.75]
#     6) CORE_PLUS_TAPER(a,b,t)
#
#   AMP
#     A) GLOBAL
#     B) RP_RB_BOOST(q), q in [0.05,0.10,0.15]
#     C) RT_RP_BOOST(q), q in [0.05,0.10,0.15]
#     D) RP_RB_SUPPRESS(q), q in [0.05,0.10,0.15]
#     E) RT_RP_SUPPRESS(q), q in [0.05,0.10,0.15]
#
# Prints:
#   - scan table
#   - best by target
#   - best by hard
#   - best by global median
#   - diagnostics for UGC02487
#
# Saves:
#   - scan csv
#   - best-target breakdown
#   - best-hard breakdown
#   - best-med breakdown
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_UGC02487_LOCAL_FORENSIC_v1"
TARGET_GALAXY = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_local_forensic_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

# target scan grids
MIXED_TAPER_MULTS = [0.2, 0.3, 0.4, 0.5, 0.6]
OUTER_TAPER_MULTS = [0.6, 0.8, 1.0, 1.25, 1.5]
INNER_SOFT_RBAR_K = [0.2, 0.35, 0.5, 0.75, 1.0]
DUAL_MIX_WEIGHTS = [0.25, 0.5, 0.75]

CORE_A_VALUES = [0.25, 0.50, 0.75, 1.00]
CORE_B_VALUES = [0.15, 0.25, 0.35, 0.50]
CORE_T_VALUES = [0.4, 0.6, 0.8, 1.0]

Q_VALUES = [0.05, 0.10, 0.15]

print(f"CELL: {CELL_ID}")
print(f"Target galaxy: {TARGET_GALAXY}")
print("Reference state: v8 locked for all non-target galaxies")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET_GALAXY not in static_cache:
    raise ValueError(f"{TARGET_GALAXY} not found in static cache")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_dual_mix(rot, r_peak, weight, taper_mult=0.6):
    r, rho = base_rho_raw(rot)
    Rt = max(taper_mult * r_peak, 1e-6)
    tapered = rho * np.exp(-r / Rt)
    rho2 = (1.0 - weight) * rho + weight * tapered
    return r, normalize_source(rho2)

def build_source_core_plus_taper(rot, r_bar, r_peak, a, b, t):
    r, rho = base_rho_raw(rot)
    Rc = max(float(a) * r_bar, 1e-6)
    core_factor = (r**2 + (float(b) * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    Rt = max(float(t) * r_peak, 1e-6)
    taper = np.exp(-r / Rt)
    rho2 = rho * core_factor * taper
    return r, normalize_source(rho2)

# ------------------------------------------------------------
# LOCKED v8 ROUTING
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def run_locked_non_target(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_locked(name, rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }

# ------------------------------------------------------------
# TARGET OVERRIDES
# ------------------------------------------------------------

def run_target_override(rec, source_mode, p1=np.nan, p2=np.nan, p3=np.nan):
    rot = rec["rot"]

    if source_mode == "BASE":
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    elif source_mode == "MIXED_TAPER":
        rr, rn = build_source_taper(rot, rec["r_peak"], p1)
        source_branch = f"MIXED_TAPER({p1})"

    elif source_mode == "OUTER_TAPER":
        rr, rn = build_source_taper(rot, rec["r_peak"], p1)
        source_branch = f"OUTER_TAPER({p1})"

    elif source_mode == "INNER_SOFT_RBAR":
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], p1)
        source_branch = f"INNER_SOFT_RBAR({p1})"

    elif source_mode == "DUAL_MIX":
        rr, rn = build_source_dual_mix(rot, rec["r_peak"], p1, taper_mult=OUTER_TAPER_MULT)
        source_branch = f"DUAL_MIX({p1})"

    elif source_mode == "CORE_PLUS_TAPER":
        rr, rn = build_source_core_plus_taper(rot, rec["r_bar"], rec["r_peak"], p1, p2, p3)
        source_branch = f"CORE_PLUS_TAPER(a={p1},b={p2},t={p3})"

    else:
        raise ValueError(f"Unknown source mode: {source_mode}")

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(rec["name"], rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    base_amp_ratio, base_amp_branch = amp_locked(rec["name"], rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "base_amp_branch": base_amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "base_amp_ratio": base_amp_ratio,
    }

def apply_amp_mode(base_amp_ratio, base_amp_branch, rec, rt, amp_mode, q=np.nan):
    rp_rb = float(rec["r_peak"] / max(rec["r_bar"], 1e-30))
    rt_rp = float(rt / max(rec["r_peak"], 1e-30))

    if amp_mode == "GLOBAL":
        return base_amp_ratio, base_amp_branch

    if amp_mode == "RP_RB_BOOST":
        factor = rp_rb**q
        return base_amp_ratio * factor, f"RP_RB_BOOST(q={q})"

    if amp_mode == "RT_RP_BOOST":
        factor = rt_rp**q
        return base_amp_ratio * factor, f"RT_RP_BOOST(q={q})"

    if amp_mode == "RP_RB_SUPPRESS":
        factor = rp_rb**(-q)
        return base_amp_ratio * factor, f"RP_RB_SUPPRESS(q={q})"

    if amp_mode == "RT_RP_SUPPRESS":
        factor = rt_rp**(-q)
        return base_amp_ratio * factor, f"RT_RP_SUPPRESS(q={q})"

    raise ValueError(f"Unknown amp mode: {amp_mode}")

# ------------------------------------------------------------
# DIAGNOSTICS
# ------------------------------------------------------------

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]; y_obs = y_obs[m]; y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])

    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))

    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# CASE RUNNER
# ------------------------------------------------------------

def run_case(case_name, source_mode, p1=np.nan, p2=np.nan, p3=np.nan, amp_mode="GLOBAL", q=np.nan):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        if name == TARGET_GALAXY:
            t = run_target_override(rec, source_mode, p1, p2, p3)
            amp_ratio, amp_branch = apply_amp_mode(
                t["base_amp_ratio"], t["base_amp_branch"], rec, t["rt"], amp_mode, q
            )
            solved[name] = {
                "source_branch": t["source_branch"],
                "shape_branch": t["shape_branch"],
                "amp_branch": amp_branch,
                "shape": t["shape"],
                "carrier": t["carrier"],
                "rt": t["rt"],
                "u_t": t["u_t"],
                "amp_ratio": amp_ratio,
            }
        else:
            solved[name] = run_locked_non_target(name, rec)

        carriers.append(solved[name]["carrier"])
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    target_focus = None

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": rec["Rs"],
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target": name == TARGET_GALAXY,
        }
        rows.append(row)

        if name == TARGET_GALAXY:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            target_focus = {**row, **seg}

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target"].values

    return {
        "case": case_name,
        "param_1": p1,
        "param_2": p2,
        "param_3": p3,
        "amp_mode": amp_mode,
        "amp_q": q,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_rmse": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "target_focus": pd.DataFrame([target_focus]) if target_focus is not None else pd.DataFrame(),
    }

# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline = run_case("BASE_V8", "BASE", amp_mode="GLOBAL")

print("\nCurrent-best-state baseline:")
print(f"  med={baseline['med']:.3f}  p90={baseline['p90']:.2f}  hard={baseline['hard']:.3f}  fix={baseline['fix']:.3f}")
print(f"  {TARGET_GALAXY} rmse = {baseline['target_rmse']:.3f}")

# ------------------------------------------------------------
# SCAN
# ------------------------------------------------------------

scan_rows = []
focus_store = {}

def store_case(res):
    row = {
        "case": res["case"],
        "param_1": res["param_1"],
        "param_2": res["param_2"],
        "param_3": res["param_3"],
        "amp_mode": res["amp_mode"],
        "amp_q": res["amp_q"],
        "med": res["med"],
        "p90": res["p90"],
        "hard": res["hard"],
        "fix": res["fix"],
        "sl": res["sl"],
        "target_rmse": res["target_rmse"],
    }
    scan_rows.append(row)
    key = (
        str(row["case"]),
        str(row["param_1"]),
        str(row["param_2"]),
        str(row["param_3"]),
        str(row["amp_mode"]),
        str(row["amp_q"]),
    )
    focus_store[key] = res["target_focus"].copy()

def print_row(res):
    p1 = "-" if pd.isna(res["param_1"]) else f"{res['param_1']:.2f}"
    p2 = "-" if pd.isna(res["param_2"]) else f"{res['param_2']:.2f}"
    p3 = "-" if pd.isna(res["param_3"]) else f"{res['param_3']:.2f}"
    qq = "-" if pd.isna(res["amp_q"]) else f"{res['amp_q']:.2f}"
    print(f"  {res['case']:18s} {p1:>6s} {p2:>6s} {p3:>6s} {res['amp_mode']:>14s} {qq:>5s}  "
          f"{res['med']:7.3f} {res['p90']:7.2f} {res['hard']:7.3f} {res['fix']:7.3f} {res['target_rmse']:9.3f}")

print("\n" + "="*142)
print(f"SCAN: {TARGET_GALAXY} local forensic")
print("="*142)
print(f"\n  {'case':18s} {'p1':>6s} {'p2':>6s} {'p3':>6s} {'amp_mode':>14s} {'q':>5s}  {'med':>7s} {'p90':>7s} {'hard':>7s} {'fix':>7s} {'target':>9s}")

# baseline
store_case(baseline)
print_row(baseline)

# BASE amp variations
for amp_mode in ["RP_RB_BOOST", "RT_RP_BOOST", "RP_RB_SUPPRESS", "RT_RP_SUPPRESS"]:
    for q in Q_VALUES:
        res = run_case("BASE", "BASE", amp_mode=amp_mode, q=q)
        store_case(res)
        print_row(res)

# MIXED_TAPER
for mult in MIXED_TAPER_MULTS:
    res = run_case("MIXED_TAPER", "MIXED_TAPER", p1=mult, amp_mode="GLOBAL")
    store_case(res)
    print_row(res)
    for amp_mode in ["RP_RB_BOOST", "RT_RP_BOOST", "RP_RB_SUPPRESS", "RT_RP_SUPPRESS"]:
        for q in Q_VALUES:
            res = run_case("MIXED_TAPER", "MIXED_TAPER", p1=mult, amp_mode=amp_mode, q=q)
            store_case(res)
            print_row(res)

# OUTER_TAPER
for mult in OUTER_TAPER_MULTS:
    res = run_case("OUTER_TAPER", "OUTER_TAPER", p1=mult, amp_mode="GLOBAL")
    store_case(res)
    print_row(res)
    for amp_mode in ["RP_RB_BOOST", "RT_RP_BOOST", "RP_RB_SUPPRESS", "RT_RP_SUPPRESS"]:
        for q in Q_VALUES:
            res = run_case("OUTER_TAPER", "OUTER_TAPER", p1=mult, amp_mode=amp_mode, q=q)
            store_case(res)
            print_row(res)

# INNER_SOFT_RBAR
for k in INNER_SOFT_RBAR_K:
    res = run_case("INNER_SOFT_RBAR", "INNER_SOFT_RBAR", p1=k, amp_mode="GLOBAL")
    store_case(res)
    print_row(res)
    for amp_mode in ["RP_RB_BOOST", "RT_RP_BOOST", "RP_RB_SUPPRESS", "RT_RP_SUPPRESS"]:
        for q in Q_VALUES:
            res = run_case("INNER_SOFT_RBAR", "INNER_SOFT_RBAR", p1=k, amp_mode=amp_mode, q=q)
            store_case(res)
            print_row(res)

# DUAL_MIX
for w in DUAL_MIX_WEIGHTS:
    res = run_case("DUAL_MIX", "DUAL_MIX", p1=w, amp_mode="GLOBAL")
    store_case(res)
    print_row(res)
    for amp_mode in ["RP_RB_BOOST", "RT_RP_BOOST", "RP_RB_SUPPRESS", "RT_RP_SUPPRESS"]:
        for q in Q_VALUES:
            res = run_case("DUAL_MIX", "DUAL_MIX", p1=w, amp_mode=amp_mode, q=q)
            store_case(res)
            print_row(res)

# CORE_PLUS_TAPER
for a in CORE_A_VALUES:
    for b in CORE_B_VALUES:
        for t in CORE_T_VALUES:
            res = run_case("CORE_PLUS_TAPER", "CORE_PLUS_TAPER", p1=a, p2=b, p3=t, amp_mode="GLOBAL")
            store_case(res)
            print_row(res)

# ------------------------------------------------------------
# SELECT BESTS
# ------------------------------------------------------------

df_scan = pd.DataFrame(scan_rows)

best_target = df_scan.iloc[df_scan["target_rmse"].idxmin()]
best_hard = df_scan.iloc[df_scan["hard"].idxmin()]
best_med = df_scan.iloc[df_scan["med"].idxmin()]

def fetch_focus(row):
    key = (
        str(row["case"]),
        str(row["param_1"]),
        str(row["param_2"]),
        str(row["param_3"]),
        str(row["amp_mode"]),
        str(row["amp_q"]),
    )
    return focus_store[key].copy()

best_target_focus = fetch_focus(best_target)
best_hard_focus = fetch_focus(best_hard)
best_med_focus = fetch_focus(best_med)

print("\n" + "="*142)
print("BEST BY TARGET")
print("="*142)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*142)
print("BEST BY HARD")
print("="*142)
print(best_hard.to_string())
print(best_hard_focus.to_string(index=False))

print("\n" + "="*142)
print("BEST BY GLOBAL MEDIAN")
print("="*142)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= baseline["med"]) & (df_scan["hard"] <= baseline["hard"])].copy()

print("\nPareto vs v8 baseline (med <= base and hard <= base):")
if len(pareto):
    print(
        pareto.sort_values(["target_rmse","hard","p90","med"])[[
            "case","param_1","param_2","param_3","amp_mode","amp_q",
            "med","p90","hard","fix","target_rmse"
        ]].to_string(index=False)
    )
else:
    print("  None.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "ugc02487_local_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_best_target_breakdown.csv")
best_hard_csv = os.path.join(OUTDIR, "ugc02487_best_hard_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_hard_focus.to_csv(best_hard_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_hard_csv)
print(" -", best_med_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_UGC02487_LOCAL_FORENSIC_v1
Target galaxy: UGC02487
Reference state: v8 locked for all non-target galaxies
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline:
  med=15.256  p90=49.72  hard=43.065  fix=42.403
  UGC02487 rmse = 110.089

SCAN: UGC02487 local forensic

  case                   p1     p2     p3       amp_mode     q      med     p90    hard     fix    target
  BASE_V8                 -      -      -         GLOBAL     -   15.256   49.72  43.065  42.403   110.089
  BASE                    -      -      -    RP_RB_BOOST  0.05   15.256   49.72  43.065  42.403   109.820
  BASE                    -      -      -    RP_RB_BOOST  0.10   15.256   49.72  43.065  42.403   109.553
  BASE                    -      -      -    RP_RB_BOOST  0.15   15.256   49.72  43.065  42.403   109.288
  BASE                    -      -      -    RT_RP_BOOST  0.05   15.256   49.72  43.065  42.403   107.942
  BASE                    -      -      -    R

In [ ]:
# ============================================================
# MTS_UGC02487_INNER_CONCENTRATION_v1
# Single independent cell
#
# Goal:
#   Starting from CURRENT_BEST_REFERENCE_v8, lock every galaxy
#   except UGC02487, and test whether UGC02487 needs an
#   explicitly inner-concentrated source branch rather than
#   another taper/amplitude tweak.
#
# Target:
#   UGC02487
#
# Families scanned:
#
# 1) INNER_POWER_CONCENTRATION
#    rho_new = rho_base * (1 + (Rc/(r+R_MIN))^n)
#    Rc in [0.25, 0.50, 0.75, 1.00] * r_bar
#    n  in [0.5, 1.0, 1.5, 2.0]
#
# 2) INNER_EXP_CONCENTRATION
#    rho_new = rho_base * (1 + A * exp(-r/Rc))
#    A  in [0.25, 0.50, 0.75, 1.00, 1.50]
#    Rc in [0.20, 0.35, 0.50, 0.75] * r_bar
#
# 3) INNER_POWER_PLUS_TAPER
#    take INNER_POWER_CONCENTRATION and multiply by
#    exp(-r/Rt), Rt in [0.6, 0.8, 1.0] * r_peak
#
# 4) INNER_EXP_PLUS_TAPER
#    take INNER_EXP_CONCENTRATION and multiply by
#    exp(-r/Rt), Rt in [0.6, 0.8, 1.0] * r_peak
#
# 5) FREE_RS variants on top of best families
#    Rs in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20]
#
# Amp modes:
#   GLOBAL only
#   (keep amplitude fixed for this experiment so we isolate
#    radial/source-placement failure)
#
# Prints:
#   - scan table
#   - best by target
#   - best by hard
#   - best by global median
#   - diagnostics for UGC02487
#
# Saves:
#   - scan csv
#   - best-target breakdown
#   - best-hard breakdown
#   - best-med breakdown
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_UGC02487_INNER_CONCENTRATION_v1"
TARGET_GALAXY = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_inner_concentration_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

# target scan grids
POWER_RC_MULTS = [0.25, 0.50, 0.75, 1.00]
POWER_N_VALUES = [0.5, 1.0, 1.5, 2.0]

EXP_A_VALUES = [0.25, 0.50, 0.75, 1.00, 1.50]
EXP_RC_MULTS = [0.20, 0.35, 0.50, 0.75]

TAPER_MULTS = [0.6, 0.8, 1.0]
RS_OVERRIDE_VALUES = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20]

print(f"CELL: {CELL_ID}")
print(f"Target galaxy: {TARGET_GALAXY}")
print("Reference state: v8 locked for all non-target galaxies")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET_GALAXY not in static_cache:
    raise ValueError(f"{TARGET_GALAXY} not found in static cache")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_power_concentration(rot, r_bar, rc_mult, n):
    r, rho = base_rho_raw(rot)
    Rc = max(float(rc_mult) * r_bar, 1e-6)
    factor = 1.0 + (Rc / np.maximum(r + R_MIN, 1e-30))**float(n)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_exp_concentration(rot, r_bar, A, rc_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(rc_mult) * r_bar, 1e-6)
    factor = 1.0 + float(A) * np.exp(-r / Rc)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_power_plus_taper(rot, r_bar, r_peak, rc_mult, n, taper_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(rc_mult) * r_bar, 1e-6)
    Rt = max(float(taper_mult) * r_peak, 1e-6)
    factor = 1.0 + (Rc / np.maximum(r + R_MIN, 1e-30))**float(n)
    rho = rho * factor * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_exp_plus_taper(rot, r_bar, r_peak, A, rc_mult, taper_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(rc_mult) * r_bar, 1e-6)
    Rt = max(float(taper_mult) * r_peak, 1e-6)
    factor = 1.0 + float(A) * np.exp(-r / Rc)
    rho = rho * factor * np.exp(-r / Rt)
    return r, normalize_source(rho)

# ------------------------------------------------------------
# LOCKED v8 ROUTING
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def run_locked_non_target(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_locked(name, rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }

# ------------------------------------------------------------
# TARGET OVERRIDES
# ------------------------------------------------------------

def run_target_override(rec, case_name, p1=np.nan, p2=np.nan, p3=np.nan, rs_override=np.nan):
    rot = rec["rot"]
    Rs_use = rec["Rs"] if pd.isna(rs_override) else float(rs_override)

    if case_name == "BASE":
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    elif case_name == "INNER_POWER_CONCENTRATION":
        rr, rn = build_source_power_concentration(rot, rec["r_bar"], p1, p2)
        source_branch = f"INNER_POWER_CONCENTRATION(Rc={p1},n={p2})"

    elif case_name == "INNER_EXP_CONCENTRATION":
        rr, rn = build_source_exp_concentration(rot, rec["r_bar"], p1, p2)
        source_branch = f"INNER_EXP_CONCENTRATION(A={p1},Rc={p2})"

    elif case_name == "INNER_POWER_PLUS_TAPER":
        rr, rn = build_source_power_plus_taper(rot, rec["r_bar"], rec["r_peak"], p1, p2, p3)
        source_branch = f"INNER_POWER_PLUS_TAPER(Rc={p1},n={p2},t={p3})"

    elif case_name == "INNER_EXP_PLUS_TAPER":
        rr, rn = build_source_exp_plus_taper(rot, rec["r_bar"], rec["r_peak"], p1, p2, p3)
        source_branch = f"INNER_EXP_PLUS_TAPER(A={p1},Rc={p2},t={p3})"

    else:
        raise ValueError(f"Unknown case_name: {case_name}")

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, Rs_use, float(r_obs.max()))
    shape, shape_branch = shape_locked(rec["name"], rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    base_amp_ratio, base_amp_branch = amp_locked(rec["name"], rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": base_amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": base_amp_ratio,
        "Rs_use": Rs_use,
    }

# ------------------------------------------------------------
# DIAGNOSTICS
# ------------------------------------------------------------

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]; y_obs = y_obs[m]; y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# CASE RUNNER
# ------------------------------------------------------------

def run_case(case_name, p1=np.nan, p2=np.nan, p3=np.nan, rs_override=np.nan):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        if name == TARGET_GALAXY:
            t = run_target_override(rec, case_name, p1, p2, p3, rs_override)
            solved[name] = t
        else:
            solved[name] = run_locked_non_target(name, rec)

        carriers.append(solved[name]["carrier"])
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    target_focus = None

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": out.get("Rs_use", rec["Rs"]),
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target": name == TARGET_GALAXY,
        }
        rows.append(row)

        if name == TARGET_GALAXY:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            target_focus = {**row, **seg}

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target"].values

    return {
        "case": case_name,
        "param_1": p1,
        "param_2": p2,
        "param_3": p3,
        "Rs_override": rs_override,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_rmse": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "target_focus": pd.DataFrame([target_focus]) if target_focus is not None else pd.DataFrame(),
    }

# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline = run_case("BASE")

print("\nCurrent-best-state baseline:")
print(f"  med={baseline['med']:.3f}  p90={baseline['p90']:.2f}  hard={baseline['hard']:.3f}  fix={baseline['fix']:.3f}")
print(f"  {TARGET_GALAXY} rmse = {baseline['target_rmse']:.3f}")

# ------------------------------------------------------------
# SCAN
# ------------------------------------------------------------

scan_rows = []
focus_store = {}

def store_case(res):
    row = {
        "case": res["case"],
        "param_1": res["param_1"],
        "param_2": res["param_2"],
        "param_3": res["param_3"],
        "Rs_override": res["Rs_override"],
        "med": res["med"],
        "p90": res["p90"],
        "hard": res["hard"],
        "fix": res["fix"],
        "sl": res["sl"],
        "target_rmse": res["target_rmse"],
    }
    scan_rows.append(row)
    key = (
        str(row["case"]),
        str(row["param_1"]),
        str(row["param_2"]),
        str(row["param_3"]),
        str(row["Rs_override"]),
    )
    focus_store[key] = res["target_focus"].copy()

def print_row(res):
    p1 = "-" if pd.isna(res["param_1"]) else f"{res['param_1']:.3f}"
    p2 = "-" if pd.isna(res["param_2"]) else f"{res['param_2']:.3f}"
    p3 = "-" if pd.isna(res["param_3"]) else f"{res['param_3']:.3f}"
    rs = "-" if pd.isna(res["Rs_override"]) else f"{res['Rs_override']:.3f}"
    print(f"  {res['case']:24s} {p1:>8s} {p2:>8s} {p3:>8s} {rs:>8s}  "
          f"{res['med']:7.3f} {res['p90']:7.2f} {res['hard']:7.3f} {res['fix']:7.3f} {res['target_rmse']:9.3f}")

print("\n" + "="*146)
print(f"SCAN: {TARGET_GALAXY} inner concentration")
print("="*146)
print(f"\n  {'case':24s} {'p1':>8s} {'p2':>8s} {'p3':>8s} {'Rs':>8s}  {'med':>7s} {'p90':>7s} {'hard':>7s} {'fix':>7s} {'target':>9s}")

store_case(baseline)
print_row(baseline)

# inner power concentration
for rc_mult in POWER_RC_MULTS:
    for n in POWER_N_VALUES:
        res = run_case("INNER_POWER_CONCENTRATION", p1=rc_mult, p2=n)
        store_case(res)
        print_row(res)

# inner exp concentration
for A in EXP_A_VALUES:
    for rc_mult in EXP_RC_MULTS:
        res = run_case("INNER_EXP_CONCENTRATION", p1=A, p2=rc_mult)
        store_case(res)
        print_row(res)

# inner power + taper
for rc_mult in POWER_RC_MULTS:
    for n in POWER_N_VALUES:
        for t in TAPER_MULTS:
            res = run_case("INNER_POWER_PLUS_TAPER", p1=rc_mult, p2=n, p3=t)
            store_case(res)
            print_row(res)

# inner exp + taper
for A in EXP_A_VALUES:
    for rc_mult in EXP_RC_MULTS:
        for t in TAPER_MULTS:
            res = run_case("INNER_EXP_PLUS_TAPER", p1=A, p2=rc_mult, p3=t)
            store_case(res)
            print_row(res)

# free Rs on the most physically promising families
for rc_mult in POWER_RC_MULTS:
    for n in POWER_N_VALUES:
        for Rs_val in RS_OVERRIDE_VALUES:
            res = run_case("INNER_POWER_CONCENTRATION", p1=rc_mult, p2=n, rs_override=Rs_val)
            store_case(res)
            print_row(res)

for A in EXP_A_VALUES:
    for rc_mult in EXP_RC_MULTS:
        for Rs_val in RS_OVERRIDE_VALUES:
            res = run_case("INNER_EXP_CONCENTRATION", p1=A, p2=rc_mult, rs_override=Rs_val)
            store_case(res)
            print_row(res)

# ------------------------------------------------------------
# SELECT BESTS
# ------------------------------------------------------------

df_scan = pd.DataFrame(scan_rows)

best_target = df_scan.iloc[df_scan["target_rmse"].idxmin()]
best_hard = df_scan.iloc[df_scan["hard"].idxmin()]
best_med = df_scan.iloc[df_scan["med"].idxmin()]

def fetch_focus(row):
    key = (
        str(row["case"]),
        str(row["param_1"]),
        str(row["param_2"]),
        str(row["param_3"]),
        str(row["Rs_override"]),
    )
    return focus_store[key].copy()

best_target_focus = fetch_focus(best_target)
best_hard_focus = fetch_focus(best_hard)
best_med_focus = fetch_focus(best_med)

print("\n" + "="*146)
print("BEST BY TARGET")
print("="*146)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*146)
print("BEST BY HARD")
print("="*146)
print(best_hard.to_string())
print(best_hard_focus.to_string(index=False))

print("\n" + "="*146)
print("BEST BY GLOBAL MEDIAN")
print("="*146)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= baseline["med"]) & (df_scan["hard"] <= baseline["hard"])].copy()

print("\nPareto vs v8 baseline (med <= base and hard <= base):")
if len(pareto):
    print(
        pareto.sort_values(["target_rmse","hard","p90","med"])[[
            "case","param_1","param_2","param_3","Rs_override",
            "med","p90","hard","fix","target_rmse"
        ]].to_string(index=False)
    )
else:
    print("  None.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "ugc02487_inner_concentration_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_inner_concentration_best_target_breakdown.csv")
best_hard_csv = os.path.join(OUTDIR, "ugc02487_inner_concentration_best_hard_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_inner_concentration_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_hard_focus.to_csv(best_hard_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_hard_csv)
print(" -", best_med_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_UGC02487_INNER_CONCENTRATION_v1
Target galaxy: UGC02487
Reference state: v8 locked for all non-target galaxies
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline:
  med=15.256  p90=49.72  hard=43.065  fix=42.403
  UGC02487 rmse = 110.089

SCAN: UGC02487 inner concentration

  case                           p1       p2       p3       Rs      med     p90    hard     fix    target
  BASE                            -        -        -        -   15.256   49.72  43.065  42.403   110.089
  INNER_POWER_CONCENTRATION    0.250    0.500        -        -   15.258   49.72  43.059  42.404   108.530
  INNER_POWER_CONCENTRATION    0.250    1.000        -        -   15.258   49.72  43.059  42.404   108.603
  INNER_POWER_CONCENTRATION    0.250    1.500        -        -   15.257   49.72  43.061  42.404   109.066
  INNER_POWER_CONCENTRATION    0.250    2.000        -        -   15.257   49.72  43.062  42.404   109.498
  INNER_POWER_CONCENTRATION    0

In [ ]:
# ============================================================
# MTS_UGC02487_RS_TRANSPORT_FORENSIC_v1
# Single independent cell
#
# Goal:
#   Starting from CURRENT_BEST_REFERENCE_v8, lock every galaxy
#   except UGC02487, and test whether UGC02487 is primarily a
#   transport / Rs-law / timing-anchor outlier rather than a
#   source-shaping outlier.
#
# Target:
#   UGC02487
#
# Strategy:
#   Keep TARGET source = BASE
#   Keep TARGET amplitude = GLOBAL
#   Vary only:
#     1) Rs directly
#     2) Rs law family
#     3) F_FRAC timing anchor used for rt
#
# Scan families:
#
# 1) FREE_RS
#    Rs in [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.40, 0.50]
#
# 2) RS_EQ_RBAR
#    Rs = c * r_bar
#
# 3) RS_EQ_RPEAK
#    Rs = c * r_peak
#
# 4) RS_GEOM
#    Rs = c * sqrt(r_bar * r_peak)
#
#    c in [0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
#
# 5) Each of the above scanned over
#    F_FRAC_LOCAL in [0.35, 0.40, 0.50, 0.60, 0.70]
#
# Prints:
#   - scan table
#   - best by target
#   - best by hard
#   - best by global median
#   - diagnostics for UGC02487
#
# Saves:
#   - scan csv
#   - best-target breakdown
#   - best-hard breakdown
#   - best-med breakdown
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_UGC02487_RS_TRANSPORT_FORENSIC_v1"
TARGET_GALAXY = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC_GLOBAL = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_rs_transport_forensic_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

FREE_RS_VALUES = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.40, 0.50]
C_VALUES = [0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
F_FRAC_VALUES = [0.35, 0.40, 0.50, 0.60, 0.70]

print(f"CELL: {CELL_ID}")
print(f"Target galaxy: {TARGET_GALAXY}")
print("Reference state: v8 locked for all non-target galaxies")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET_GALAXY not in static_cache:
    raise ValueError(f"{TARGET_GALAXY} not found in static cache")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ------------------------------------------------------------
# LOCKED v8 ROUTING
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def run_locked_non_target(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC_GLOBAL * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_locked(name, rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }

# ------------------------------------------------------------
# TARGET OVERRIDES
# ------------------------------------------------------------

def run_target_override(rec, case_name, p1=np.nan, p2=np.nan, ffrac_local=F_FRAC_GLOBAL):
    rot = rec["rot"]
    rr, rn = build_source_base(rot)

    if case_name == "BASE":
        Rs_use = rec["Rs"]
        source_branch = "BASE"

    elif case_name == "FREE_RS":
        Rs_use = float(p1)
        source_branch = f"FREE_RS({p1})"

    elif case_name == "RS_EQ_RBAR":
        Rs_use = float(p1) * rec["r_bar"]
        source_branch = f"RS_EQ_RBAR(c={p1})"

    elif case_name == "RS_EQ_RPEAK":
        Rs_use = float(p1) * rec["r_peak"]
        source_branch = f"RS_EQ_RPEAK(c={p1})"

    elif case_name == "RS_GEOM":
        Rs_use = float(p1) * np.sqrt(rec["r_bar"] * rec["r_peak"])
        source_branch = f"RS_GEOM(c={p1})"

    else:
        raise ValueError(f"Unknown case_name: {case_name}")

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, Rs_use, float(r_obs.max()))
    shape, shape_branch = shape_locked(rec["name"], rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= float(ffrac_local) * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
        "Rs_use": Rs_use,
        "ffrac_local": float(ffrac_local),
    }

# ------------------------------------------------------------
# DIAGNOSTICS
# ------------------------------------------------------------

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]; y_obs = y_obs[m]; y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# CASE RUNNER
# ------------------------------------------------------------

def run_case(case_name, p1=np.nan, p2=np.nan):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        if name == TARGET_GALAXY:
            t = run_target_override(rec, case_name, p1=p1, p2=p2, ffrac_local=p2 if case_name=="FREE_RS_ONLY_FFRAC" else (p2 if case_name.startswith("RS_") or case_name=="FREE_RS" else F_FRAC_GLOBAL))
            solved[name] = t
        else:
            solved[name] = run_locked_non_target(name, rec)

        carriers.append(solved[name]["carrier"])
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    target_focus = None

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": out.get("Rs_use", rec["Rs"]),
            "rt": out["rt"],
            "u_t": out["u_t"],
            "ffrac_local": out.get("ffrac_local", F_FRAC_GLOBAL),
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target": name == TARGET_GALAXY,
        }
        rows.append(row)

        if name == TARGET_GALAXY:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            target_focus = {**row, **seg}

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target"].values

    return {
        "case": case_name,
        "param_1": p1,
        "param_2": p2,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_rmse": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "target_focus": pd.DataFrame([target_focus]) if target_focus is not None else pd.DataFrame(),
    }

def run_case_transport(case_name, rs_param, ffrac_local):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        if name == TARGET_GALAXY:
            if case_name == "FREE_RS":
                t = run_target_override(rec, "FREE_RS", p1=rs_param, p2=ffrac_local, ffrac_local=ffrac_local)
            elif case_name == "RS_EQ_RBAR":
                t = run_target_override(rec, "RS_EQ_RBAR", p1=rs_param, p2=ffrac_local, ffrac_local=ffrac_local)
            elif case_name == "RS_EQ_RPEAK":
                t = run_target_override(rec, "RS_EQ_RPEAK", p1=rs_param, p2=ffrac_local, ffrac_local=ffrac_local)
            elif case_name == "RS_GEOM":
                t = run_target_override(rec, "RS_GEOM", p1=rs_param, p2=ffrac_local, ffrac_local=ffrac_local)
            else:
                raise ValueError(case_name)
            solved[name] = t
        else:
            solved[name] = run_locked_non_target(name, rec)

        carriers.append(solved[name]["carrier"])
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    target_focus = None

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": out.get("Rs_use", rec["Rs"]),
            "rt": out["rt"],
            "u_t": out["u_t"],
            "ffrac_local": out.get("ffrac_local", F_FRAC_GLOBAL),
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target": name == TARGET_GALAXY,
        }
        rows.append(row)

        if name == TARGET_GALAXY:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            target_focus = {**row, **seg}

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target"].values

    return {
        "case": case_name,
        "param_1": rs_param,
        "param_2": ffrac_local,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_rmse": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "target_focus": pd.DataFrame([target_focus]) if target_focus is not None else pd.DataFrame(),
    }

# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline = run_case_transport("FREE_RS", static_cache[TARGET_GALAXY]["Rs"], F_FRAC_GLOBAL)
baseline["case"] = "BASE_V8"

print("\nCurrent-best-state baseline:")
print(f"  med={baseline['med']:.3f}  p90={baseline['p90']:.2f}  hard={baseline['hard']:.3f}  fix={baseline['fix']:.3f}")
print(f"  {TARGET_GALAXY} rmse = {baseline['target_rmse']:.3f}")

# ------------------------------------------------------------
# SCAN
# ------------------------------------------------------------

scan_rows = []
focus_store = {}

def store_case(res):
    row = {
        "case": res["case"],
        "param_1": res["param_1"],
        "param_2": res["param_2"],
        "med": res["med"],
        "p90": res["p90"],
        "hard": res["hard"],
        "fix": res["fix"],
        "sl": res["sl"],
        "target_rmse": res["target_rmse"],
    }
    scan_rows.append(row)
    key = (str(row["case"]), str(row["param_1"]), str(row["param_2"]))
    focus_store[key] = res["target_focus"].copy()

def print_row(res):
    p1 = "-" if pd.isna(res["param_1"]) else f"{res['param_1']:.3f}"
    p2 = "-" if pd.isna(res["param_2"]) else f"{res['param_2']:.3f}"
    print(f"  {res['case']:14s} {p1:>8s} {p2:>8s}  "
          f"{res['med']:7.3f} {res['p90']:7.2f} {res['hard']:7.3f} {res['fix']:7.3f} {res['target_rmse']:9.3f}")

print("\n" + "="*126)
print(f"SCAN: {TARGET_GALAXY} Rs / transport forensic")
print("="*126)
print(f"\n  {'case':14s} {'param':>8s} {'F_FRAC':>8s}  {'med':>7s} {'p90':>7s} {'hard':>7s} {'fix':>7s} {'target':>9s}")

store_case(baseline)
print_row(baseline)

for ffrac in F_FRAC_VALUES:
    for Rs_val in FREE_RS_VALUES:
        res = run_case_transport("FREE_RS", Rs_val, ffrac)
        store_case(res)
        print_row(res)

for ffrac in F_FRAC_VALUES:
    for c in C_VALUES:
        res = run_case_transport("RS_EQ_RBAR", c, ffrac)
        store_case(res)
        print_row(res)

for ffrac in F_FRAC_VALUES:
    for c in C_VALUES:
        res = run_case_transport("RS_EQ_RPEAK", c, ffrac)
        store_case(res)
        print_row(res)

for ffrac in F_FRAC_VALUES:
    for c in C_VALUES:
        res = run_case_transport("RS_GEOM", c, ffrac)
        store_case(res)
        print_row(res)

# ------------------------------------------------------------
# SELECT BESTS
# ------------------------------------------------------------

df_scan = pd.DataFrame(scan_rows)

best_target = df_scan.iloc[df_scan["target_rmse"].idxmin()]
best_hard = df_scan.iloc[df_scan["hard"].idxmin()]
best_med = df_scan.iloc[df_scan["med"].idxmin()]

def fetch_focus(row):
    key = (str(row["case"]), str(row["param_1"]), str(row["param_2"]))
    return focus_store[key].copy()

best_target_focus = fetch_focus(best_target)
best_hard_focus = fetch_focus(best_hard)
best_med_focus = fetch_focus(best_med)

print("\n" + "="*126)
print("BEST BY TARGET")
print("="*126)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*126)
print("BEST BY HARD")
print("="*126)
print(best_hard.to_string())
print(best_hard_focus.to_string(index=False))

print("\n" + "="*126)
print("BEST BY GLOBAL MEDIAN")
print("="*126)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= baseline["med"]) & (df_scan["hard"] <= baseline["hard"])].copy()

print("\nPareto vs v8 baseline (med <= base and hard <= base):")
if len(pareto):
    print(
        pareto.sort_values(["target_rmse","hard","p90","med"])[[
            "case","param_1","param_2","med","p90","hard","fix","target_rmse"
        ]].to_string(index=False)
    )
else:
    print("  None.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "ugc02487_rs_transport_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_rs_transport_best_target_breakdown.csv")
best_hard_csv = os.path.join(OUTDIR, "ugc02487_rs_transport_best_hard_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_rs_transport_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_hard_focus.to_csv(best_hard_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_hard_csv)
print(" -", best_med_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_UGC02487_RS_TRANSPORT_FORENSIC_v1
Target galaxy: UGC02487
Reference state: v8 locked for all non-target galaxies
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline:
  med=15.256  p90=49.72  hard=43.065  fix=42.403
  UGC02487 rmse = 110.089

SCAN: UGC02487 Rs / transport forensic

  case              param   F_FRAC      med     p90    hard     fix    target
  BASE_V8           0.502    0.600   15.256   49.72  43.065  42.403   110.089
  FREE_RS           0.030    0.350   15.378   49.09  42.921  42.726   156.337
  FREE_RS           0.050    0.350   15.335   49.10  42.873  42.678   147.688
  FREE_RS           0.075    0.350   15.297   49.18  42.833  42.638   140.811
  FREE_RS           0.100    0.350   15.269   49.28  42.803  42.608   135.965
  FREE_RS           0.125    0.350   15.280   49.37  42.778  42.584   132.248
  FREE_RS           0.150    0.350   15.304   49.44  42.758  42.563   129.250
  FREE_RS           0.200    0.350   15.34

In [ ]:
# ============================================================
# MTS_UGC02487_TWO_STAGE_RESPONSE_v1
# Single independent cell
#
# Goal:
#   Starting from CURRENT_BEST_REFERENCE_v8, lock every galaxy
#   except UGC02487, and test whether UGC02487 needs a genuine
#   two-stage response rather than any single-scale response.
#
# Target:
#   UGC02487
#
# Idea:
#   Use BASE source for UGC02487, but build two screened responses:
#       S1(r) from Rs1
#       S2(r) from Rs2
#   Then blend:
#       shape_mix(r) = (1-w)*S1(r) + w*S2(r)
#
#   Carrier for amplitude anchor is built from the blended timing
#   quantities (rt_mix, u_t_mix) taken from the blended cumulative.
#
# Scan:
#   Rs1 = [0.03, 0.05, 0.075, 0.10]
#   Rs2 = [0.15, 0.20, 0.30, 0.40, 0.50]
#   w   = [0.10, 0.20, 0.30, 0.40, 0.50]
#   keep only Rs2 > Rs1
#
#   Optional timing anchor:
#   F_FRAC_LOCAL in [0.35, 0.40, 0.50, 0.60, 0.70]
#
# Amplitude:
#   GLOBAL only for this test
#
# Prints:
#   - scan table
#   - best by target
#   - best by hard
#   - best by global median
#   - diagnostics for UGC02487
#
# Saves:
#   - scan csv
#   - best-target breakdown
#   - best-hard breakdown
#   - best-med breakdown
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_UGC02487_TWO_STAGE_RESPONSE_v1"
TARGET_GALAXY = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC_GLOBAL = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_two_stage_response_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

RS1_VALUES = [0.03, 0.05, 0.075, 0.10]
RS2_VALUES = [0.15, 0.20, 0.30, 0.40, 0.50]
W_VALUES = [0.10, 0.20, 0.30, 0.40, 0.50]
F_FRAC_VALUES = [0.35, 0.40, 0.50, 0.60, 0.70]

print(f"CELL: {CELL_ID}")
print(f"Target galaxy: {TARGET_GALAXY}")
print("Reference state: v8 locked for all non-target galaxies")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET_GALAXY not in static_cache:
    raise ValueError(f"{TARGET_GALAXY} not found in static cache")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ------------------------------------------------------------
# LOCKED v8 ROUTING
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def run_locked_non_target(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC_GLOBAL * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_locked(name, rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }

# ------------------------------------------------------------
# TARGET TWO-STAGE OVERRIDE
# ------------------------------------------------------------

def build_two_stage_target(rec, Rs1, Rs2, w, ffrac_local):
    rot = rec["rot"]
    rr, rn = build_source_base(rot)

    r_obs = rec["r_obs"]

    r_g1, u_g1, U_g1 = solve_field(rr, rn, Rs1, float(r_obs.max()))
    r_g2, u_g2, U_g2 = solve_field(rr, rn, Rs2, float(r_obs.max()))

    # common grid for blend
    r_common = r_g1 if len(r_g1) <= len(r_g2) else r_g2
    U1_fn = interp1d(r_g1, U_g1, bounds_error=False, fill_value="extrapolate")
    U2_fn = interp1d(r_g2, U_g2, bounds_error=False, fill_value="extrapolate")
    u1_fn = interp1d(r_g1, u_g1, bounds_error=False, fill_value="extrapolate")
    u2_fn = interp1d(r_g2, u_g2, bounds_error=False, fill_value="extrapolate")

    U_mix = (1.0 - w) * U1_fn(r_common) + w * U2_fn(r_common)
    u_mix = (1.0 - w) * u1_fn(r_common) + w * u2_fn(r_common)

    U_mix = np.maximum(U_mix, 0.0)
    u_mix = np.maximum(u_mix, 0.0)

    U_inf = float(np.max(U_mix))
    if not np.isfinite(U_inf) or U_inf <= 0:
        raise RuntimeError("Bad two-stage U_inf")

    U_obs_fn = interp1d(r_common, U_mix, bounds_error=False, fill_value="extrapolate")
    shape = np.clip(U_obs_fn(r_obs) / U_inf, 0.0, 1.0)

    idx = np.where(U_mix >= float(ffrac_local) * U_inf)[0]
    if len(idx) == 0:
        raise RuntimeError("No rt index in two-stage blend")
    rt = float(r_common[idx[0]])

    u_mix_fn = interp1d(r_common, u_mix, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_mix_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)

    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    return {
        "source_branch": f"TWO_STAGE_RESPONSE(Rs1={Rs1},Rs2={Rs2},w={w})",
        "shape_branch": "BASE_SHAPE",
        "amp_branch": "GLOBAL_AMP",
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": 1.0,
        "Rs1": Rs1,
        "Rs2": Rs2,
        "w": w,
        "ffrac_local": float(ffrac_local),
    }

# ------------------------------------------------------------
# DIAGNOSTICS
# ------------------------------------------------------------

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]; y_obs = y_obs[m]; y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# CASE RUNNER
# ------------------------------------------------------------

def run_case(case_name, Rs1, Rs2, w, ffrac_local):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        if name == TARGET_GALAXY:
            solved[name] = build_two_stage_target(rec, Rs1, Rs2, w, ffrac_local)
        else:
            solved[name] = run_locked_non_target(name, rec)

        carriers.append(solved[name]["carrier"])
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    target_focus = None

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs1": out.get("Rs1", np.nan),
            "Rs2": out.get("Rs2", np.nan),
            "w_mix": out.get("w", np.nan),
            "rt": out["rt"],
            "u_t": out["u_t"],
            "ffrac_local": out.get("ffrac_local", F_FRAC_GLOBAL),
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target": name == TARGET_GALAXY,
        }
        rows.append(row)

        if name == TARGET_GALAXY:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            target_focus = {**row, **seg}

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target"].values

    return {
        "case": case_name,
        "param_1": Rs1,
        "param_2": Rs2,
        "param_3": w,
        "param_4": ffrac_local,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_rmse": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "target_focus": pd.DataFrame([target_focus]) if target_focus is not None else pd.DataFrame(),
    }

# baseline = v8-like single stage encoded as two identical scales, w=0
baseline = run_case("BASE_V8_EQUIV", static_cache[TARGET_GALAXY]["Rs"], static_cache[TARGET_GALAXY]["Rs"], 0.0, F_FRAC_GLOBAL)

print("\nCurrent-best-state baseline equivalent:")
print(f"  med={baseline['med']:.3f}  p90={baseline['p90']:.2f}  hard={baseline['hard']:.3f}  fix={baseline['fix']:.3f}")
print(f"  {TARGET_GALAXY} rmse = {baseline['target_rmse']:.3f}")

# ------------------------------------------------------------
# SCAN
# ------------------------------------------------------------

scan_rows = []
focus_store = {}

def store_case(res):
    row = {
        "case": res["case"],
        "param_1": res["param_1"],
        "param_2": res["param_2"],
        "param_3": res["param_3"],
        "param_4": res["param_4"],
        "med": res["med"],
        "p90": res["p90"],
        "hard": res["hard"],
        "fix": res["fix"],
        "sl": res["sl"],
        "target_rmse": res["target_rmse"],
    }
    scan_rows.append(row)
    key = (str(row["case"]), str(row["param_1"]), str(row["param_2"]), str(row["param_3"]), str(row["param_4"]))
    focus_store[key] = res["target_focus"].copy()

def print_row(res):
    p1 = "-" if pd.isna(res["param_1"]) else f"{res['param_1']:.3f}"
    p2 = "-" if pd.isna(res["param_2"]) else f"{res['param_2']:.3f}"
    p3 = "-" if pd.isna(res["param_3"]) else f"{res['param_3']:.3f}"
    p4 = "-" if pd.isna(res["param_4"]) else f"{res['param_4']:.3f}"
    print(f"  {res['case']:18s} {p1:>8s} {p2:>8s} {p3:>8s} {p4:>8s}  "
          f"{res['med']:7.3f} {res['p90']:7.2f} {res['hard']:7.3f} {res['fix']:7.3f} {res['target_rmse']:9.3f}")

print("\n" + "="*144)
print(f"SCAN: {TARGET_GALAXY} two-stage response")
print("="*144)
print(f"\n  {'case':18s} {'Rs1':>8s} {'Rs2':>8s} {'w':>8s} {'F_FRAC':>8s}  {'med':>7s} {'p90':>7s} {'hard':>7s} {'fix':>7s} {'target':>9s}")

store_case(baseline)
print_row(baseline)

for Rs1 in RS1_VALUES:
    for Rs2 in RS2_VALUES:
        if not (Rs2 > Rs1):
            continue
        for w in W_VALUES:
            for ffrac in F_FRAC_VALUES:
                try:
                    res = run_case("TWO_STAGE_RESPONSE", Rs1, Rs2, w, ffrac)
                    store_case(res)
                    print_row(res)
                except Exception as e:
                    print(f"  SKIP {'TWO_STAGE_RESPONSE':18s} {Rs1:8.3f} {Rs2:8.3f} {w:8.3f} {ffrac:8.3f}  reason={str(e)[:60]}")

# ------------------------------------------------------------
# SELECT BESTS
# ------------------------------------------------------------

df_scan = pd.DataFrame(scan_rows)

best_target = df_scan.iloc[df_scan["target_rmse"].idxmin()]
best_hard = df_scan.iloc[df_scan["hard"].idxmin()]
best_med = df_scan.iloc[df_scan["med"].idxmin()]

def fetch_focus(row):
    key = (str(row["case"]), str(row["param_1"]), str(row["param_2"]), str(row["param_3"]), str(row["param_4"]))
    return focus_store[key].copy()

best_target_focus = fetch_focus(best_target)
best_hard_focus = fetch_focus(best_hard)
best_med_focus = fetch_focus(best_med)

print("\n" + "="*144)
print("BEST BY TARGET")
print("="*144)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*144)
print("BEST BY HARD")
print("="*144)
print(best_hard.to_string())
print(best_hard_focus.to_string(index=False))

print("\n" + "="*144)
print("BEST BY GLOBAL MEDIAN")
print("="*144)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= baseline["med"]) & (df_scan["hard"] <= baseline["hard"])].copy()

print("\nPareto vs baseline-equivalent (med <= base and hard <= base):")
if len(pareto):
    print(
        pareto.sort_values(["target_rmse","hard","p90","med"])[[
            "case","param_1","param_2","param_3","param_4",
            "med","p90","hard","fix","target_rmse"
        ]].to_string(index=False)
    )
else:
    print("  None.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "ugc02487_two_stage_response_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_two_stage_best_target_breakdown.csv")
best_hard_csv = os.path.join(OUTDIR, "ugc02487_two_stage_best_hard_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_two_stage_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_hard_focus.to_csv(best_hard_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_hard_csv)
print(" -", best_med_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_UGC02487_TWO_STAGE_RESPONSE_v1
Target galaxy: UGC02487
Reference state: v8 locked for all non-target galaxies
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline equivalent:
  med=15.256  p90=49.72  hard=43.065  fix=42.403
  UGC02487 rmse = 110.089

SCAN: UGC02487 two-stage response

  case                    Rs1      Rs2        w   F_FRAC      med     p90    hard     fix    target
  BASE_V8_EQUIV         0.502    0.502    0.000    0.600   15.256   49.72  43.065  42.403   110.089
  TWO_STAGE_RESPONSE    0.030    0.150    0.100    0.350   15.326   49.10  42.864  42.669   145.989
  TWO_STAGE_RESPONSE    0.030    0.150    0.100    0.400   15.320   49.10  42.857  42.662   144.892
  TWO_STAGE_RESPONSE    0.030    0.150    0.100    0.500   15.324   49.10  42.862  42.667   145.715
  TWO_STAGE_RESPONSE    0.030    0.150    0.100    0.600   15.317   49.11  42.854  42.659   144.399
  TWO_STAGE_RESPONSE    0.030    0.150    0.100    0.700   15.3

In [ ]:
# ============================================================
# MTS_FULL_RESIDUAL_CENSUS_v1
# Single independent cell
#
# Goal:
#   Using CURRENT_BEST_REFERENCE_v8 logic, compute a full
#   catalogue-wide residual census:
#
#   1) Full ranked residual table for all galaxies
#   2) Inner / mid / outer decomposition
#   3) Peak-placement diagnostics
#   4) Automatic residual subclass assignment
#   5) Bucket galaxies into:
#        - RESOLVED
#        - IMPROVED_ORDINARY
#        - BRANCH_DEPENDENT
#        - STRUCTURAL_SURVIVOR
#
# Notes:
#   - This reproduces the v8 branch stack directly in-cell.
#   - No external prior cell state required.
#   - UGC02487 is expected to remain a structural survivor.
#
# Saves:
#   - residual_census_full_table.csv
#   - residual_census_ranked.csv
#   - residual_census_bucket_summary.csv
#   - residual_census_subclass_summary.csv
#   - residual_census_hard_focus.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_FULL_RESIDUAL_CENSUS_v1"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_full_residual_census_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# v8 locked branch stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

print(f"CELL: {CELL_ID}")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "mid_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "mid_bias": bias(m2),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# LOAD SHAPE CLASSES
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ------------------------------------------------------------
# V8 BRANCH RULES
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def run_v8_galaxy(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"

    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"

    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"

    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"

    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"

    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    r_obs = rec["r_obs"]
    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_locked(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rot), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_locked(name, rec, rt)

    return {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }

# ------------------------------------------------------------
# RUN WHOLE CATALOGUE
# ------------------------------------------------------------

solved = {}
carriers = []
vmaxes = []

for name, rec in static_cache.items():
    out = run_v8_galaxy(name, rec)
    solved[name] = out
    carriers.append(out["carrier"])
    vmaxes.append(rec["vmax_obs"])

C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

rows = []
for name, rec in static_cache.items():
    out = solved[name]
    vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
    vflat_pred = vflat_model * out["amp_ratio"]
    vpred = vflat_pred * out["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)

    seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

    row = {
        "name": name,
        "shape_class": rec["shape_class"],
        "source_branch": out["source_branch"],
        "shape_branch": out["shape_branch"],
        "amp_branch": out["amp_branch"],
        "rmse": rmse,
        "vflat_model": vflat_model,
        "amp_ratio_pred": out["amp_ratio"],
        "vflat_pred": vflat_pred,
        "r_bar": rec["r_bar"],
        "r_peak": rec["r_peak"],
        "Rs": rec["Rs"],
        "rt": out["rt"],
        "u_t": out["u_t"],
        "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
        "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
        "is_hard": name in HARD_FAILS,
        "is_fixable": name in HARD_FIXABLE,
        "is_shape_limited": name in SHAPE_LIMITED,
        **seg,
    }
    rows.append(row)

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# AUTO RESIDUAL SUBCLASS
# ------------------------------------------------------------

def assign_residual_subclass(row):
    rmse = row["rmse"]
    ib = row["inner_bias"]
    ob = row["outer_bias"]
    prs = row["peak_r_shift_frac"]
    pve = row["peak_v_err_frac"]
    rr = row["rt_over_rpeak"]

    if not np.isfinite(rmse):
        return "UNCLASSIFIED"

    # strong late / delayed peak class
    if np.isfinite(prs) and prs > 0.25 and np.isfinite(ob) and ob < -10:
        return "PEAK_TOO_LATE"

    # strong low / weak peak class
    if np.isfinite(ib) and ib < -40 and np.isfinite(ob) and ob > 5:
        return "PEAK_TOO_LOW"

    # classic peak undershoot signature
    if row["shape_class"] == "PEAK_UNDERSHOOT" and rmse > 35:
        return "PEAK_UNDERSHOOT_CONFIRMED"

    # compressed compact-floor failures
    if row["shape_class"] == "COMPACT_FLOOR_PATHOLOGY" and rmse > 35:
        return "COMPACT_FLOOR_PATHOLOGY"

    # outer tail lingering
    if row["shape_class"] == "OVERLONG_OUTER_TAIL" and np.isfinite(ob) and ob > 5:
        return "OVERLONG_OUTER_TAIL"

    # inner-dominant residual
    if np.isfinite(row["inner_rmse"]) and np.isfinite(row["outer_rmse"]):
        if row["inner_rmse"] > 2.5 * max(row["outer_rmse"], 1e-9):
            return "INNER_DOMINATED"

    # outer-dominant residual
    if np.isfinite(row["outer_rmse"]) and np.isfinite(row["inner_rmse"]):
        if row["outer_rmse"] > 2.0 * max(row["inner_rmse"], 1e-9):
            return "OUTER_DOMINATED"

    # timing mismatch
    if np.isfinite(rr) and (rr < 0.25 or rr > 2.5):
        return "TIMING_MISMATCH"

    return "MIXED_RESIDUAL"

df["residual_subclass"] = df.apply(assign_residual_subclass, axis=1)

# ------------------------------------------------------------
# BUCKET ASSIGNMENT
# ------------------------------------------------------------

rmse_q50 = float(df["rmse"].median())
rmse_q75 = float(df["rmse"].quantile(0.75))
rmse_q90 = float(df["rmse"].quantile(0.90))

def assign_bucket(row):
    name = row["name"]
    rmse = row["rmse"]

    if name == "UGC02487":
        return "STRUCTURAL_SURVIVOR"

    if rmse <= rmse_q50:
        return "RESOLVED"

    if rmse <= rmse_q75:
        return "IMPROVED_ORDINARY"

    # if branch-special but not the one hard survivor
    if (
        row["source_branch"] != "BASE" or
        row["shape_branch"] != "BASE_SHAPE" or
        row["amp_branch"] != "GLOBAL_AMP"
    ):
        return "BRANCH_DEPENDENT"

    return "STRUCTURAL_SURVIVOR"

df["residual_bucket"] = df.apply(assign_bucket, axis=1)

# ------------------------------------------------------------
# SUMMARIES
# ------------------------------------------------------------

df_ranked = df.sort_values(["rmse", "inner_rmse"], ascending=[False, False]).reset_index(drop=True)

bucket_summary = (
    df.groupby("residual_bucket")
      .agg(
          n=("name", "count"),
          med_rmse=("rmse", "median"),
          mean_rmse=("rmse", "mean"),
          med_inner_rmse=("inner_rmse", "median"),
          med_mid_rmse=("mid_rmse", "median"),
          med_outer_rmse=("outer_rmse", "median"),
      )
      .reset_index()
      .sort_values("med_rmse", ascending=False)
)

subclass_summary = (
    df.groupby("residual_subclass")
      .agg(
          n=("name", "count"),
          med_rmse=("rmse", "median"),
          mean_rmse=("rmse", "mean"),
          med_inner_bias=("inner_bias", "median"),
          med_outer_bias=("outer_bias", "median"),
          med_peak_r_shift_frac=("peak_r_shift_frac", "median"),
          med_peak_v_err_frac=("peak_v_err_frac", "median"),
      )
      .reset_index()
      .sort_values(["med_rmse", "n"], ascending=[False, False])
)

hard_focus = (
    df[df["is_hard"]]
    .sort_values("rmse", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n" + "="*100)
print("CURRENT BEST REFERENCE CHECK (v8 logic)")
print("="*100)
print(f"C_amp_global = {C_amp_global:.6f}")
print(f"med         = {df['rmse'].median():.3f}")
print(f"p90         = {df['rmse'].quantile(0.90):.2f}")
print(f"hard        = {df.loc[df['is_hard'], 'rmse'].median():.3f}")
print(f"fix         = {df.loc[df['is_fixable'], 'rmse'].median():.3f}")
print(f"sl          = {df.loc[df['is_shape_limited'], 'rmse'].median():.3f}")

print("\n" + "="*100)
print("TOP 25 RESIDUALS")
print("="*100)
print(
    df_ranked[[
        "name","shape_class","rmse","inner_rmse","mid_rmse","outer_rmse",
        "inner_bias","outer_bias","peak_r_shift_frac","peak_v_err_frac",
        "residual_subclass","residual_bucket"
    ]].head(25).to_string(index=False)
)

print("\n" + "="*100)
print("HARD FOCUS")
print("="*100)
print(
    hard_focus[[
        "name","shape_class","source_branch","shape_branch","amp_branch","rmse",
        "inner_rmse","mid_rmse","outer_rmse",
        "inner_bias","outer_bias","peak_r_shift_frac","peak_v_err_frac",
        "residual_subclass","residual_bucket"
    ]].to_string(index=False)
)

print("\n" + "="*100)
print("RESIDUAL SUBCLASS SUMMARY")
print("="*100)
print(subclass_summary.to_string(index=False))

print("\n" + "="*100)
print("RESIDUAL BUCKET SUMMARY")
print("="*100)
print(bucket_summary.to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

full_csv = os.path.join(OUTDIR, "residual_census_full_table.csv")
ranked_csv = os.path.join(OUTDIR, "residual_census_ranked.csv")
bucket_csv = os.path.join(OUTDIR, "residual_census_bucket_summary.csv")
subclass_csv = os.path.join(OUTDIR, "residual_census_subclass_summary.csv")
hard_csv = os.path.join(OUTDIR, "residual_census_hard_focus.csv")

df.to_csv(full_csv, index=False)
df_ranked.to_csv(ranked_csv, index=False)
bucket_summary.to_csv(bucket_csv, index=False)
subclass_summary.to_csv(subclass_csv, index=False)
hard_focus.to_csv(hard_csv, index=False)

print("\nSaved:")
print(" -", full_csv)
print(" -", ranked_csv)
print(" -", bucket_csv)
print(" -", subclass_csv)
print(" -", hard_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_FULL_RESIDUAL_CENSUS_v1
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

CURRENT BEST REFERENCE CHECK (v8 logic)
C_amp_global = 69.351204
med         = 15.268
p90         = 49.72
hard        = 43.016
fix         = 42.415
sl          = 15.797

TOP 25 RESIDUALS
       name             shape_class      rmse  inner_rmse  mid_rmse  outer_rmse  inner_bias  outer_bias  peak_r_shift_frac  peak_v_err_frac         residual_subclass     residual_bucket
   UGC02487                   MIXED 97.652516  160.898602 36.666422    3.160808 -150.950223    2.826509           1.666888        -0.128384           INNER_DOMINATED STRUCTURAL_SURVIVOR
   UGC06973                   MIXED 80.550372   39.514663 88.528844  100.331065   -8.674968  100.329779           0.000000         0.553578           OUTER_DOMINATED STRUCTURAL_SURVIVOR
    NGC2683     OVERLONG_OUTER_TAIL 74.970581   80.397993 53.276138   82.849379  -64.486165   82.763532           5.000000         0.119709              PE

In [ ]:
# ============================================================
# MTS_RESIDUAL_PEAK_CLASS_BRANCH_v1
# Single independent cell
#
# Goal:
#   Starting from CURRENT_BEST_REFERENCE_v8 logic, use the
#   residual census classes to test catalogue-level class
#   shape branches instead of more singleton hacking.
#
# Classes targeted:
#   1) PEAK_TOO_LOW
#   2) PEAK_TOO_LATE
#   3) OVERLONG_OUTER_TAIL
#
# Strategy:
#   Keep all existing v8 source / amplitude branches.
#   Add class-level SHAPE overrides only for galaxies whose
#   residual_subclass from the census matches the targeted class.
#
# Shape families scanned:
#
#   PEAK_TOO_LOW:
#     - GAMMA(g)              g in [0.65, 0.70, 0.75, 0.80, 0.85]
#     - EARLY_BLEND(w)        w in [0.10, 0.20, 0.30, 0.40, 0.50]
#     - TAIL_GAMMA(e,g)       e in [-0.85, -0.75, -0.65], g in [0.60,0.70,0.80]
#
#   PEAK_TOO_LATE:
#     - GAMMA(g)              g in [0.60, 0.70, 0.80]
#     - EARLY_BLEND(w)        w in [0.20, 0.30, 0.40, 0.50]
#     - TAIL_GAMMA(e,g)       e in [-0.50,-0.40,-0.30], g in [0.70,0.80,0.90]
#
#   OVERLONG_OUTER_TAIL:
#     - OUTER_TAIL_GAMMA(g)   g in [1.10, 1.20, 1.30]
#     - POST_PEAK_TAPER(t)    t in [0.20, 0.35, 0.50, 0.75]
#
# Metrics:
#   - global median
#   - p90
#   - hard median
#   - fix median
#   - shape-limited median
#   - targeted subclass median
#
# Saves:
#   - residual_peak_class_scan.csv
#   - residual_peak_class_best_target_breakdown.csv
#   - residual_peak_class_best_hard_breakdown.csv
#   - residual_peak_class_best_med_breakdown.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_RESIDUAL_PEAK_CLASS_BRANCH_v1"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_residual_peak_class_branch_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

# scan grids
PEAK_LOW_GAMMA = [0.65, 0.70, 0.75, 0.80, 0.85]
PEAK_LOW_EARLY = [0.10, 0.20, 0.30, 0.40, 0.50]
PEAK_LOW_TAIL = [-0.85, -0.75, -0.65]
PEAK_LOW_TAIL_GAMMA = [0.60, 0.70, 0.80]

PEAK_LATE_GAMMA = [0.60, 0.70, 0.80]
PEAK_LATE_EARLY = [0.20, 0.30, 0.40, 0.50]
PEAK_LATE_TAIL = [-0.50, -0.40, -0.30]
PEAK_LATE_TAIL_GAMMA = [0.70, 0.80, 0.90]

OUTER_GAMMA = [1.10, 1.20, 1.30]
POST_PEAK_TAPER = [0.20, 0.35, 0.50, 0.75]

print(f"CELL: {CELL_ID}")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def shape_early_blend(s, w):
    s = np.clip(s, 0.0, 1.0)
    return np.clip((1.0 - w) * s + w * np.sqrt(s), 0.0, 1.0)

def shape_post_peak_taper(r_obs, s, r_peak, t):
    s = np.clip(s, 0.0, 1.0).copy()
    r_obs = np.asarray(r_obs, float)
    Rt = max(float(t) * r_peak, 1e-9)
    mask = r_obs > r_peak
    s[mask] = s[mask] * np.exp(-(r_obs[mask] - r_peak) / Rt)
    smax = np.max(s)
    if smax > 0:
        s = np.clip(s / smax, 0.0, 1.0)
    return s

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

# ------------------------------------------------------------
# LOAD SHAPE CLASSES + CENSUS
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")
if not os.path.exists(CENSUS_CSV):
    raise FileNotFoundError(f"Missing census CSV: {CENSUS_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

peak_too_low_set = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LOW", "name"])
peak_too_late_set = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LATE", "name"])
outer_tail_set = set(df_census.loc[df_census["residual_subclass"] == "OVERLONG_OUTER_TAIL", "name"])

print("Target subclass pools:")
print(f"  PEAK_TOO_LOW         : {len(peak_too_low_set)}")
print(f"  PEAK_TOO_LATE        : {len(peak_too_late_set)}")
print(f"  OVERLONG_OUTER_TAIL  : {len(outer_tail_set)}")

# ------------------------------------------------------------
# LOAD ROTMOD FILES
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ------------------------------------------------------------
# V8 BASE ROUTING
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def shape_base_v8(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def amp_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def source_v8(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"

    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"

    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"

    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"

    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"

    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

# ------------------------------------------------------------
# CLASS OVERRIDE SHAPES
# ------------------------------------------------------------

def apply_class_shape_override(name, rec, r_obs, base_shape, mode, p1=np.nan, p2=np.nan):
    s = np.array(base_shape, dtype=float).copy()
    subclass = rec["residual_subclass"]
    branch_label = "BASE_SHAPE"

    # preserve singleton hand-tuned branches unless explicitly targeted by class
    targetable = False

    if subclass == "PEAK_TOO_LOW" and name in peak_too_low_set:
        targetable = True
        if mode == "GAMMA":
            s = shape_gamma(s, p1)
            branch_label = f"CLASS_PEAK_TOO_LOW_GAMMA({p1})"
        elif mode == "EARLY_BLEND":
            s = shape_early_blend(s, p1)
            branch_label = f"CLASS_PEAK_TOO_LOW_EARLY_BLEND({p1})"
        elif mode == "TAIL_GAMMA":
            s = shape_tail(s, p1)
            s = shape_gamma(s, p2)
            branch_label = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={p1},gamma={p2})"

    elif subclass == "PEAK_TOO_LATE" and name in peak_too_late_set:
        targetable = True
        if mode == "GAMMA":
            s = shape_gamma(s, p1)
            branch_label = f"CLASS_PEAK_TOO_LATE_GAMMA({p1})"
        elif mode == "EARLY_BLEND":
            s = shape_early_blend(s, p1)
            branch_label = f"CLASS_PEAK_TOO_LATE_EARLY_BLEND({p1})"
        elif mode == "TAIL_GAMMA":
            s = shape_tail(s, p1)
            s = shape_gamma(s, p2)
            branch_label = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={p1},gamma={p2})"

    elif subclass == "OVERLONG_OUTER_TAIL" and name in outer_tail_set:
        targetable = True
        if mode == "OUTER_TAIL_GAMMA":
            s = shape_gamma(s, p1)
            branch_label = f"CLASS_OVERLONG_OUTER_GAMMA({p1})"
        elif mode == "POST_PEAK_TAPER":
            s = shape_post_peak_taper(r_obs, s, rec["r_peak"], p1)
            branch_label = f"CLASS_OVERLONG_POST_PEAK_TAPER({p1})"

    if not targetable:
        # keep the v8 local label if no class override applies
        return s, None

    return np.clip(s, 0.0, 1.0), branch_label

# ------------------------------------------------------------
# RUN CASE
# ------------------------------------------------------------

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

def run_case(target_subclass, mode, p1=np.nan, p2=np.nan):
    solved = {}
    carriers = []
    vmaxes = []

    target_set = {
        "PEAK_TOO_LOW": peak_too_low_set,
        "PEAK_TOO_LATE": peak_too_late_set,
        "OVERLONG_OUTER_TAIL": outer_tail_set,
    }[target_subclass]

    for name, rec in static_cache.items():
        rr, rn, source_branch = source_v8(name, rec)
        r_obs = rec["r_obs"]

        r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
        base_shape, base_shape_branch = shape_base_v8(name, rec, r_obs, r_g, U_g)

        class_shape, class_branch = apply_class_shape_override(
            name, rec, r_obs, base_shape, mode, p1=p1, p2=p2
        )
        shape = class_shape
        shape_branch = class_branch if class_branch is not None else base_shape_branch

        idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
        rt = float(r_g[idx[0]])
        u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
        u_t = max(float(u_fn(rt)), 1e-30)

        vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
        vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
        vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
        carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

        amp_ratio, amp_branch = amp_locked(name, rec, rt)

        solved[name] = {
            "source_branch": source_branch,
            "shape_branch": shape_branch,
            "amp_branch": amp_branch,
            "shape": shape,
            "carrier": carrier,
            "rt": rt,
            "u_t": u_t,
            "amp_ratio": amp_ratio,
        }

        carriers.append(carrier)
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    focus_rows = []

    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)

        row = {
            "name": name,
            "shape_class": rec["shape_class"],
            "residual_subclass": rec["residual_subclass"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": rec["Rs"],
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_target_subclass": name in target_set,
        }
        rows.append(row)

        if name in target_set:
            seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)
            focus_rows.append({**row, **seg})

    df = pd.DataFrame(rows)
    focus = pd.DataFrame(focus_rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    it = df["is_target_subclass"].values

    return {
        "target_subclass": target_subclass,
        "mode": mode,
        "param_1": p1,
        "param_2": p2,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "target_med": float(np.nanmedian(arr[it])) if it.any() else np.nan,
        "focus": focus.copy(),
    }

# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline = run_case("PEAK_TOO_LOW", "BASE", np.nan, np.nan)
# replace target metric per subclass as needed later using reruns
# also compute dedicated baselines for all three subclasses
baseline_low = run_case("PEAK_TOO_LOW", "BASE", np.nan, np.nan)
baseline_late = run_case("PEAK_TOO_LATE", "BASE", np.nan, np.nan)
baseline_outer = run_case("OVERLONG_OUTER_TAIL", "BASE", np.nan, np.nan)

print("\nBaselines:")
print(f"  PEAK_TOO_LOW        : med={baseline_low['med']:.3f}  hard={baseline_low['hard']:.3f}  target={baseline_low['target_med']:.3f}")
print(f"  PEAK_TOO_LATE       : med={baseline_late['med']:.3f}  hard={baseline_late['hard']:.3f}  target={baseline_late['target_med']:.3f}")
print(f"  OVERLONG_OUTER_TAIL : med={baseline_outer['med']:.3f}  hard={baseline_outer['hard']:.3f}  target={baseline_outer['target_med']:.3f}")

# ------------------------------------------------------------
# SCAN
# ------------------------------------------------------------

scan_rows = []
focus_store = {}

def store_case(res):
    row = {
        "target_subclass": res["target_subclass"],
        "mode": res["mode"],
        "param_1": res["param_1"],
        "param_2": res["param_2"],
        "med": res["med"],
        "p90": res["p90"],
        "hard": res["hard"],
        "fix": res["fix"],
        "sl": res["sl"],
        "target_med": res["target_med"],
    }
    scan_rows.append(row)
    key = (
        str(row["target_subclass"]),
        str(row["mode"]),
        str(row["param_1"]),
        str(row["param_2"]),
    )
    focus_store[key] = res["focus"].copy()

def print_row(res):
    p1 = "-" if pd.isna(res["param_1"]) else f"{res['param_1']:.3f}"
    p2 = "-" if pd.isna(res["param_2"]) else f"{res['param_2']:.3f}"
    print(f"  {res['target_subclass']:18s} {res['mode']:18s} {p1:>8s} {p2:>8s}  "
          f"{res['med']:7.3f} {res['p90']:7.2f} {res['hard']:7.3f} {res['fix']:7.3f} {res['sl']:7.3f} {res['target_med']:9.3f}")

print("\n" + "="*156)
print("SCAN: residual peak class branches")
print("="*156)
print(f"\n  {'target_subclass':18s} {'mode':18s} {'p1':>8s} {'p2':>8s}  {'med':>7s} {'p90':>7s} {'hard':>7s} {'fix':>7s} {'sl':>7s} {'target':>9s}")

# baselines
for b in [baseline_low, baseline_late, baseline_outer]:
    store_case(b)
    print_row(b)

# PEAK_TOO_LOW
for g in PEAK_LOW_GAMMA:
    res = run_case("PEAK_TOO_LOW", "GAMMA", g, np.nan)
    store_case(res); print_row(res)
for w in PEAK_LOW_EARLY:
    res = run_case("PEAK_TOO_LOW", "EARLY_BLEND", w, np.nan)
    store_case(res); print_row(res)
for e in PEAK_LOW_TAIL:
    for g in PEAK_LOW_TAIL_GAMMA:
        res = run_case("PEAK_TOO_LOW", "TAIL_GAMMA", e, g)
        store_case(res); print_row(res)

# PEAK_TOO_LATE
for g in PEAK_LATE_GAMMA:
    res = run_case("PEAK_TOO_LATE", "GAMMA", g, np.nan)
    store_case(res); print_row(res)
for w in PEAK_LATE_EARLY:
    res = run_case("PEAK_TOO_LATE", "EARLY_BLEND", w, np.nan)
    store_case(res); print_row(res)
for e in PEAK_LATE_TAIL:
    for g in PEAK_LATE_TAIL_GAMMA:
        res = run_case("PEAK_TOO_LATE", "TAIL_GAMMA", e, g)
        store_case(res); print_row(res)

# OVERLONG_OUTER_TAIL
for g in OUTER_GAMMA:
    res = run_case("OVERLONG_OUTER_TAIL", "OUTER_TAIL_GAMMA", g, np.nan)
    store_case(res); print_row(res)
for t in POST_PEAK_TAPER:
    res = run_case("OVERLONG_OUTER_TAIL", "POST_PEAK_TAPER", t, np.nan)
    store_case(res); print_row(res)

df_scan = pd.DataFrame(scan_rows)

# ------------------------------------------------------------
# PICK BESTS
# ------------------------------------------------------------

# Global bests overall scan
best_target = df_scan.iloc[df_scan["target_med"].idxmin()]
best_hard = df_scan.iloc[df_scan["hard"].idxmin()]
best_med = df_scan.iloc[df_scan["med"].idxmin()]

def fetch_focus(row):
    key = (
        str(row["target_subclass"]),
        str(row["mode"]),
        str(row["param_1"]),
        str(row["param_2"]),
    )
    return focus_store[key].copy()

best_target_focus = fetch_focus(best_target)
best_hard_focus = fetch_focus(best_hard)
best_med_focus = fetch_focus(best_med)

print("\n" + "="*156)
print("BEST BY TARGET MEDIAN")
print("="*156)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*156)
print("BEST BY HARD")
print("="*156)
print(best_hard.to_string())
print(best_hard_focus.to_string(index=False))

print("\n" + "="*156)
print("BEST BY GLOBAL MEDIAN")
print("="*156)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

# per-subclass pareto against own baseline
baseline_map = {
    "PEAK_TOO_LOW": baseline_low,
    "PEAK_TOO_LATE": baseline_late,
    "OVERLONG_OUTER_TAIL": baseline_outer,
}

print("\nPer-subclass pareto vs own baseline:")
for subclass, b in baseline_map.items():
    sub = df_scan[df_scan["target_subclass"] == subclass].copy()
    pareto = sub[(sub["med"] <= b["med"]) & (sub["hard"] <= b["hard"])]
    print(f"\n{subclass}:")
    if len(pareto):
        print(
            pareto.sort_values(["target_med", "hard", "p90", "med"])[[
                "target_subclass","mode","param_1","param_2","med","p90","hard","fix","sl","target_med"
            ]].to_string(index=False)
        )
    else:
        print("  None.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "residual_peak_class_scan.csv")
best_target_csv = os.path.join(OUTDIR, "residual_peak_class_best_target_breakdown.csv")
best_hard_csv = os.path.join(OUTDIR, "residual_peak_class_best_hard_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "residual_peak_class_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_hard_focus.to_csv(best_hard_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_hard_csv)
print(" -", best_med_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_RESIDUAL_PEAK_CLASS_BRANCH_v1
Target subclass pools:
  PEAK_TOO_LOW         : 9
  PEAK_TOO_LATE        : 12
  OVERLONG_OUTER_TAIL  : 5
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Baselines:
  PEAK_TOO_LOW        : med=15.268  hard=43.016  target=52.507
  PEAK_TOO_LATE       : med=15.268  hard=43.016  target=39.057
  OVERLONG_OUTER_TAIL : med=15.268  hard=43.016  target=43.032

SCAN: residual peak class branches

  target_subclass    mode                     p1       p2      med     p90    hard     fix      sl    target
  PEAK_TOO_LOW       BASE                      -        -   15.268   49.72  43.016  42.415  15.797    52.507
  PEAK_TOO_LATE      BASE                      -        -   15.268   49.72  43.016  42.415  15.797    39.057
  OVERLONG_OUTER_TAIL BASE                      -        -   15.268   49.72  43.016  42.415  15.797    43.032
  PEAK_TOO_LOW       GAMMA                 0.650        -   15.144   45.00  37.703  37.079  15.797    40.050
  PEAK

In [ ]:
# ============================================================
# MTS_V9_CLASS_MERGE_v1
# Single independent cell
#
# Goal:
#   Merge the successful class-level residual shape laws found
#   in MTS_RESIDUAL_PEAK_CLASS_BRANCH_v1 on top of the locked
#   v8 branch stack, and compare:
#
#     1) BASE_V8
#     2) LOW_ONLY
#     3) LATE_ONLY
#     4) LOW_PLUS_LATE
#
# Locked class laws from scan:
#   PEAK_TOO_LOW  -> TAIL(-0.75) then ^0.8
#   PEAK_TOO_LATE -> TAIL(-0.50) then ^0.7
#
# Precedence:
#   - existing source overrides remain
#   - existing amplitude overrides remain
#   - extreme singleton UGC02885 stays explicit
#   - class law applies AFTER base/singleton shape construction
#
# Saves:
#   - v9_class_merge_scan.csv
#   - v9_class_merge_best_state_hard_targets.csv
#   - v9_class_merge_best_state_full_table.csv
#   - v9_class_merge_state_summaries.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CELL_ID = "MTS_V9_CLASS_MERGE_v1"

R_MIN = 1e-3
R_MAX = 55.0
N_R = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL = 1.25
ALPHA = 0.10
BETA = 0.65

RS_MAX = 30.0
ABS_FLOOR = 0.10
F_FLOOR = 0.02
K_VAL = 0.05
RBAR_EXP = 0.75
RPEAK_EXP = 0.25

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_v9_class_merge_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# locked v8 stack
TAIL_B_FIXED = -0.55
OUTER_TAPER_MULT = 0.60
INNER_SOFT_MULT_UGC11455 = 0.35
UGC06787_TAPER_MULT = 0.10

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
INNER_SOFT_GROUP = {"UGC11455"}
COMPACT_TAPER_GROUP = {"UGC06787"}

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

PEAK_TOO_LATE_GROUP = {"NGC0801", "NGC7814", "UGC03546"}
PEAK_TOO_LATE_GAMMA = 0.70

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

# v9 class-law locks from scan
V9_LOW_ETA = -0.75
V9_LOW_GAMMA = 0.80
V9_LATE_ETA = -0.50
V9_LATE_GAMMA = 0.70

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

print(f"CELL: {CELL_ID}")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() > 0 else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError("No valid rows")

    order = np.argsort(r[mask])
    return {
        k: v[mask][order] for k, v in zip(
            ["r", "vobs", "ev", "vgas", "vdisk", "vbul"],
            [r, vobs, ev, vgas, vdisk, vbul]
        )
    }

def vbar2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK     * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL      * rot["vbul"]  * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    vm = np.max(vb)
    if vm <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vm)[0]
    return float(rot["r"][idx[0]]) if len(idx) > 0 else np.nan

def get_r_peak(rot):
    vb2 = np.maximum(vbar2(rot), 0.0)
    vb = np.sqrt(vb2)
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, r_max_obs * 1.15)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i - 1] = cm
        A[i, i] = -(cm + cp) - 1.0 / Rs**2
        A[i, i + 1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(s, 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ------------------------------------------------------------
# LOAD SHAPE CLASSES + CENSUS
# ------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")
if not os.path.exists(CENSUS_CSV):
    raise FileNotFoundError(f"Missing census CSV: {CENSUS_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

peak_too_low_set = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LOW", "name"])
peak_too_late_set = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LATE", "name"])

print("Class-law target sets:")
print(f"  PEAK_TOO_LOW  : {len(peak_too_low_set)}")
print(f"  PEAK_TOO_LATE : {len(peak_too_late_set)}")

# ------------------------------------------------------------
# LOAD ROTMOD
# ------------------------------------------------------------

def bootstrap():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No .dat files found")
    return files

files = bootstrap()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ------------------------------------------------------------
# STATIC CACHE
# ------------------------------------------------------------

def build_static_cache(name, rot):
    r_bar = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar**RBAR_EXP) * (r_peak**RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

# ------------------------------------------------------------
# SOURCE BUILDERS
# ------------------------------------------------------------

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe**P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ------------------------------------------------------------
# LOCKED V8 SOURCE + AMP
# ------------------------------------------------------------

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def source_v8(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"

    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"

    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"

    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"

    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"

    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

def amp_v8(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

# ------------------------------------------------------------
# SHAPE ROUTING
# ------------------------------------------------------------

def base_shape_with_singletons(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")
    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in PEAK_TOO_LATE_GROUP:
        s = shape_gamma(s, PEAK_TOO_LATE_GAMMA)
        shape_branch = f"PEAK_TOO_LATE_GAMMA({PEAK_TOO_LATE_GAMMA})"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"

    return s, shape_branch

def apply_v9_class_law(name, rec, s, active_low=False, active_late=False):
    s = np.array(s, dtype=float).copy()
    branch_extra = None

    # preserve explicit singleton extreme-peak as highest-precedence final shape
    if name in EXTREME_PEAK_SINGLETON:
        return s, branch_extra

    subclass = rec["residual_subclass"]

    if active_low and subclass == "PEAK_TOO_LOW":
        s = shape_tail(s, V9_LOW_ETA)
        s = shape_gamma(s, V9_LOW_GAMMA)
        branch_extra = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={V9_LOW_ETA},gamma={V9_LOW_GAMMA})"

    if active_late and subclass == "PEAK_TOO_LATE":
        s = shape_tail(s, V9_LATE_ETA)
        s = shape_gamma(s, V9_LATE_GAMMA)
        branch_extra = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={V9_LATE_ETA},gamma={V9_LATE_GAMMA})"

    return np.clip(s, 0.0, 1.0), branch_extra

# ------------------------------------------------------------
# RUN STATE
# ------------------------------------------------------------

def run_state(state_name, active_low=False, active_late=False):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        rr, rn, source_branch = source_v8(name, rec)
        r_obs = rec["r_obs"]

        r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))

        s, shape_branch = base_shape_with_singletons(name, rec, r_obs, r_g, U_g)
        s2, class_branch = apply_v9_class_law(name, rec, s, active_low=active_low, active_late=active_late)
        if class_branch is not None:
            shape_branch = class_branch

        idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
        rt = float(r_g[idx[0]])
        u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
        u_t = max(float(u_fn(rt)), 1e-30)

        vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
        vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
        vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
        carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

        amp_ratio, amp_branch = amp_v8(name, rec, rt)

        solved[name] = {
            "source_branch": source_branch,
            "shape_branch": shape_branch,
            "amp_branch": amp_branch,
            "shape": s2,
            "carrier": carrier,
            "rt": rt,
            "u_t": u_t,
            "amp_ratio": amp_ratio,
        }

        carriers.append(carrier)
        vmaxes.append(rec["vmax_obs"])

    C_amp = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp * out["carrier"], 0.0)))
        vflat_pred = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)
        seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

        rows.append({
            "state": state_name,
            "name": name,
            "shape_class": rec["shape_class"],
            "residual_subclass": rec["residual_subclass"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": rec["Rs"],
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            "is_hard": name in HARD_FAILS,
            "is_fixable": name in HARD_FIXABLE,
            "is_shape_limited": name in SHAPE_LIMITED,
            "is_peak_too_low": name in peak_too_low_set,
            "is_peak_too_late": name in peak_too_late_set,
            **seg,
        })

    df = pd.DataFrame(rows)

    arr = df["rmse"].values
    ih = df["is_hard"].values
    ihf = df["is_fixable"].values
    isl = df["is_shape_limited"].values
    ilow = df["is_peak_too_low"].values
    ilate = df["is_peak_too_late"].values

    summary = {
        "state": state_name,
        "C_amp_global": C_amp,
        "med": float(np.nanmedian(arr)),
        "p90": float(np.nanpercentile(arr, 90)),
        "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
        "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
        "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
        "peak_too_low_med": float(np.nanmedian(arr[ilow])) if ilow.any() else np.nan,
        "peak_too_late_med": float(np.nanmedian(arr[ilate])) if ilate.any() else np.nan,
    }

    return df, summary

# ------------------------------------------------------------
# RUN STATES
# ------------------------------------------------------------

state_defs = [
    ("BASE_V8", False, False),
    ("LOW_ONLY", True, False),
    ("LATE_ONLY", False, True),
    ("LOW_PLUS_LATE", True, True),
]

all_tables = []
summaries = []

print("\n" + "="*120)
print("SCAN: v9 class merge states")
print("="*120)
print(f"\n  {'state':14s} {'med':>8s} {'p90':>8s} {'hard':>8s} {'fix':>8s} {'sl':>8s} {'low_med':>10s} {'late_med':>10s}")

state_to_df = {}

for state_name, low_on, late_on in state_defs:
    df_state, summary = run_state(state_name, active_low=low_on, active_late=late_on)
    state_to_df[state_name] = df_state.copy()
    all_tables.append(df_state)
    summaries.append(summary)
    print(f"  {state_name:14s} {summary['med']:8.3f} {summary['p90']:8.2f} {summary['hard']:8.3f} {summary['fix']:8.3f} {summary['sl']:8.3f} {summary['peak_too_low_med']:10.3f} {summary['peak_too_late_med']:10.3f}")

df_summary = pd.DataFrame(summaries)
df_all = pd.concat(all_tables, ignore_index=True)

# ------------------------------------------------------------
# PICK BEST STATE
# ------------------------------------------------------------

# Best by hard, then p90, then med
best_row = df_summary.sort_values(["hard", "p90", "med"]).iloc[0]
best_state = best_row["state"]
df_best = state_to_df[best_state].copy()

print("\n" + "="*120)
print("BEST STATE")
print("="*120)
print(best_row.to_string())

print("\n" + "="*120)
print("BEST STATE HARD TARGETS")
print("="*120)
print(
    df_best[df_best["is_hard"]][[
        "name","shape_class","residual_subclass","source_branch","shape_branch","amp_branch",
        "rmse","inner_rmse","mid_rmse","outer_rmse",
        "inner_bias","outer_bias","peak_r_shift_frac","peak_v_err_frac"
    ]].sort_values("rmse", ascending=False).to_string(index=False)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

scan_csv = os.path.join(OUTDIR, "v9_class_merge_scan.csv")
best_hard_csv = os.path.join(OUTDIR, "v9_class_merge_best_state_hard_targets.csv")
best_full_csv = os.path.join(OUTDIR, "v9_class_merge_best_state_full_table.csv")
summary_csv = os.path.join(OUTDIR, "v9_class_merge_state_summaries.csv")

df_all.to_csv(scan_csv, index=False)
df_best[df_best["is_hard"]].sort_values("rmse", ascending=False).to_csv(best_hard_csv, index=False)
df_best.sort_values("rmse", ascending=False).to_csv(best_full_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_hard_csv)
print(" -", best_full_csv)
print(" -", summary_csv)

print(f"\nCELL {CELL_ID} complete.")

CELL: MTS_V9_CLASS_MERGE_v1
Class-law target sets:
  PEAK_TOO_LOW  : 9
  PEAK_TOO_LATE : 12
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

SCAN: v9 class merge states

  state               med      p90     hard      fix       sl    low_med   late_med
  BASE_V8          15.268    49.72   43.016   42.415   15.797     52.507     39.057
  LOW_ONLY         15.144    46.97   42.223   36.293   15.797     30.094     39.057
  LATE_ONLY        15.144    45.73   42.223   36.283   15.797     52.507     26.662
  LOW_PLUS_LATE    15.077    43.03   32.588   31.476   15.797     30.094     26.662

BEST STATE
state                LOW_PLUS_LATE
C_amp_global             69.351204
med                      15.076736
p90                      43.025718
hard                     32.588259
fix                       31.47575
sl                       15.796805
peak_too_low_med         30.094056
peak_too_late_med        26.661514

BEST STATE HARD TARGETS
       name             shape_class resid

In [ ]:
# ============================================================
# CELL: MTS_CURRENT_BEST_REFERENCE_v9
#
# Purpose:
#   Lock in the merged best state from MTS_V9_CLASS_MERGE_v1
#   as the new working reference model.
#
# This cell reproduces the full v9 branch stack directly and
# prints / saves the current-best summary tables.
#
# v9 additions over v8:
#   - PEAK_TOO_LOW residual class:
#       shape := TAIL(-0.75) then ^0.8
#   - PEAK_TOO_LATE residual class:
#       shape := TAIL(-0.5) then ^0.7
#
# Precedence:
#   1) source overrides remain
#   2) amplitude overrides remain
#   3) explicit extreme singleton UGC02885 remains
#   4) residual class shape law applies after base shape
#
# Saves:
#   - /content/mts_current_best_reference_v9/current_best_summary_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_full_table_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_hard_targets_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_compact_targets_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_boosted_mixed_targets_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_peak_too_late_targets_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_extreme_peak_singleton_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_ngc2841_singleton_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_ngc5005_singleton_v9.csv
#   - /content/mts_current_best_reference_v9/current_best_class_law_targets_v9.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

print("CELL: MTS_CURRENT_BEST_REFERENCE_v9")

# ============================================================
# CONFIG
# ============================================================

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL  = 1.25
ALPHA  = 0.10
BETA   = 0.65

ABS_FLOOR = 0.10
F_FLOOR   = 0.02
K_VAL     = 0.05
RBAR_EXP  = 0.75
RPEAK_EXP = 0.25
RS_MAX    = 30.0

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_current_best_reference_v9"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# ------------------------------------------------------------
# Locked branch stack from v8/v9
# ------------------------------------------------------------

TAIL_B_FIXED = -0.55

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
OUTER_TAPER_MULT = 0.60

INNER_SOFT_GROUP = {"UGC11455"}
INNER_SOFT_MULT_UGC11455 = 0.35

COMPACT_TAPER_GROUP = {"UGC06787"}
UGC06787_TAPER_MULT = 0.10

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

# Compact amplitude law
COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

# v9 merged class laws
V9_LOW_ETA   = -0.75
V9_LOW_GAMMA = 0.80
V9_LATE_ETA   = -0.50
V9_LATE_GAMMA = 0.70

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]
CENTRE_MISSING = ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]
HARD_FIXABLE = [g for g in HARD_FAILS if g not in CENTRE_MISSING]
SHAPE_LIMITED = ["UGC11914","UGC05253","UGC11455"]

# ============================================================
# HELPERS
# ============================================================

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError(f"No valid rows in {path}")

    order = np.argsort(r[mask])
    return {
        "r":     r[mask][order],
        "vobs":  vobs[mask][order],
        "ev":    ev[mask][order],
        "vgas":  vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul":  vbul[mask][order],
    }

def vbar2(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"] * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    vmax = np.max(vb)
    if vmax <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vmax)[0]
    return float(rot["r"][idx[0]]) if len(idx) else np.nan

def get_r_peak(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, 1.15 * r_max_obs)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])

    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = ri + 0.5 * dr
        rim = ri - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i-1] = cm
        A[i, i]   = -(cm + cp) - 1.0 / Rs**2
        A[i, i+1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R-1, N_R-1] = 1.0
    b[N_R-1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac   = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ============================================================
# LOAD CLASS MAPS
# ============================================================

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(f"Missing master CSV: {MASTER_CSV}")
if not os.path.exists(CENSUS_CSV):
    raise FileNotFoundError(f"Missing census CSV: {CENSUS_CSV}")

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

peak_too_low_set  = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LOW", "name"])
peak_too_late_set = set(df_census.loc[df_census["residual_subclass"] == "PEAK_TOO_LATE", "name"])

# ============================================================
# LOAD ROTMODS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

files = bootstrap_rotmods()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ============================================================
# STATIC CACHE
# ============================================================

def build_static_cache(name, rot):
    r_bar  = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar ** RBAR_EXP) * (r_peak ** RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

# ============================================================
# SOURCE BUILDERS
# ============================================================

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe ** P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

# ============================================================
# V9 ROUTING
# ============================================================

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def source_v9(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"

    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"

    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"

    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"

    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"

    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

def base_shape_v9(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")

    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    # explicit singleton precedence
    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"
        return s, shape_branch

    # merged class laws
    if rec["residual_subclass"] == "PEAK_TOO_LOW":
        s = shape_tail(s, V9_LOW_ETA)
        s = shape_gamma(s, V9_LOW_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={V9_LOW_ETA},gamma={V9_LOW_GAMMA})"

    elif rec["residual_subclass"] == "PEAK_TOO_LATE":
        s = shape_tail(s, V9_LATE_ETA)
        s = shape_gamma(s, V9_LATE_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={V9_LATE_ETA},gamma={V9_LATE_GAMMA})"

    return s, shape_branch

def amp_v9(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

# ============================================================
# RUN FULL V9 STATE
# ============================================================

solved = {}
carriers = []
vmaxes = []

for name, rec in static_cache.items():
    rr, rn, source_branch = source_v9(name, rec)
    r_obs = rec["r_obs"]

    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = base_shape_v9(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_v9(name, rec, rt)

    solved[name] = {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }
    carriers.append(carrier)
    vmaxes.append(rec["vmax_obs"])

C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

rows = []
for name, rec in static_cache.items():
    out = solved[name]
    vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
    vflat_pred  = vflat_model * out["amp_ratio"]
    vpred = vflat_pred * out["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)

    seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

    rows.append({
        "name": name,
        "shape_class": rec["shape_class"],
        "residual_subclass": rec["residual_subclass"],
        "source_branch": out["source_branch"],
        "shape_branch": out["shape_branch"],
        "amp_branch": out["amp_branch"],
        "rmse": rmse,
        "vflat_model": vflat_model,
        "amp_ratio_pred": out["amp_ratio"],
        "vflat_pred": vflat_pred,
        "r_bar": rec["r_bar"],
        "r_peak": rec["r_peak"],
        "Rs": rec["Rs"],
        "rt": out["rt"],
        "u_t": out["u_t"],
        "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
        "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
        "is_hard": name in HARD_FAILS,
        "is_fixable": name in HARD_FIXABLE,
        "is_shape_limited": name in SHAPE_LIMITED,
        "is_peak_too_low": name in peak_too_low_set,
        "is_peak_too_late": name in peak_too_late_set,
        **seg,
    })

df = pd.DataFrame(rows)

# ============================================================
# SUMMARIES
# ============================================================

arr = df["rmse"].values
ih = df["is_hard"].values
ihf = df["is_fixable"].values
isl = df["is_shape_limited"].values
ilow = df["is_peak_too_low"].values
ilate = df["is_peak_too_late"].values

summary_df = pd.DataFrame([{
    "C_amp_global": C_amp_global,
    "med": float(np.nanmedian(arr)),
    "p90": float(np.nanpercentile(arr, 90)),
    "hard": float(np.nanmedian(arr[ih])) if ih.any() else np.nan,
    "fix": float(np.nanmedian(arr[ihf])) if ihf.any() else np.nan,
    "sl": float(np.nanmedian(arr[isl])) if isl.any() else np.nan,
    "peak_too_low_med": float(np.nanmedian(arr[ilow])) if ilow.any() else np.nan,
    "peak_too_late_med": float(np.nanmedian(arr[ilate])) if ilate.any() else np.nan,
}])

hard_targets_df = df[df["is_hard"]].sort_values("rmse", ascending=False).reset_index(drop=True)
compact_targets_df = df[df["shape_class"] == "COMPACT_FLOOR_PATHOLOGY"].sort_values("rmse", ascending=False).reset_index(drop=True)
boosted_mixed_df = df[df["name"].isin(BOOSTED_MIXED_GROUP)].sort_values("rmse", ascending=False).reset_index(drop=True)
peak_too_late_df = df[df["name"].isin(peak_too_late_set)].sort_values("rmse", ascending=False).reset_index(drop=True)
extreme_peak_df = df[df["name"].isin(EXTREME_PEAK_SINGLETON)].sort_values("rmse", ascending=False).reset_index(drop=True)
ngc2841_df = df[df["name"].isin(NGC2841_SINGLETON)].sort_values("rmse", ascending=False).reset_index(drop=True)
ngc5005_df = df[df["name"].isin(NGC5005_SINGLETON)].sort_values("rmse", ascending=False).reset_index(drop=True)
class_law_df = df[df["shape_branch"].str.contains("CLASS_PEAK_TOO_LOW|CLASS_PEAK_TOO_LATE", na=False)].sort_values("rmse", ascending=False).reset_index(drop=True)

# ============================================================
# PRINT
# ============================================================

print("\n" + "="*100)
print("CURRENT BEST REFERENCE STATE v9")
print("="*100)
print(f"C_amp_global      = {C_amp_global:.6f}")
print(f"med               = {summary_df.loc[0, 'med']:.3f}")
print(f"p90               = {summary_df.loc[0, 'p90']:.2f}")
print(f"hard              = {summary_df.loc[0, 'hard']:.3f}")
print(f"fix               = {summary_df.loc[0, 'fix']:.3f}")
print(f"sl                = {summary_df.loc[0, 'sl']:.3f}")
print(f"peak_too_low_med  = {summary_df.loc[0, 'peak_too_low_med']:.3f}")
print(f"peak_too_late_med = {summary_df.loc[0, 'peak_too_late_med']:.3f}")

print("\nBranch assignments:")
print(f"  PEAK_UNDERSHOOT base shape: TAIL({TAIL_B_FIXED})")
print(f"  MIXED_TAPER source: OUTER_TAPER({OUTER_TAPER_MULT} * r_peak)")
print(f"  MIXED_TAPER members: {sorted(MIXED_TAPER_GROUP)}")
print(f"  REMAINDER_INNER_SOFT members: {sorted(INNER_SOFT_GROUP)}")
print(f"  COMPACT_OUTER_TAPER members: {sorted(COMPACT_TAPER_GROUP)}")
print(f"  COMPACT_OUTER_TAPER scale: {UGC06787_TAPER_MULT} * r_peak")
print("  COMPACT amplitude law:")
print(f"    log10(amp_ratio) = {COMPACT_AMP_INTERCEPT:.12f} + {COMPACT_AMP_COEF_RPEAK_OVER_RBAR:.12f} * log10(r_peak/r_bar) + {COMPACT_AMP_COEF_RT_OVER_RPEAK:.12f} * log10(rt/r_peak)")
print(f"  BOOSTED_MIXED subgroup: {sorted(BOOSTED_MIXED_GROUP)}")
print(f"    amp_boost = (r_peak/r_bar)^{BOOST_Q}")
print(f"  EXTREME_PEAK singleton: {sorted(EXTREME_PEAK_SINGLETON)}")
print(f"    shape_warp = TAIL({EXTREME_PEAK_ETA}) then ^{EXTREME_PEAK_GAMMA}")
print(f"  NGC2841 singleton source: MIXED_TAPER({NGC2841_TAPER_MULT} * r_peak)")
print(f"  NGC5005 singleton source: MIXED_TAPER({NGC5005_TAPER_MULT} * r_peak)")
print(f"  NGC5005 singleton amp: RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})")
print("  v9 merged class laws:")
print(f"    PEAK_TOO_LOW  -> TAIL({V9_LOW_ETA}) then ^{V9_LOW_GAMMA}")
print(f"    PEAK_TOO_LATE -> TAIL({V9_LATE_ETA}) then ^{V9_LATE_GAMMA}")

print("\nHard targets:")
print(hard_targets_df[[
    "name","shape_class","residual_subclass","source_branch","shape_branch","amp_branch",
    "rmse","r_bar","r_peak","Rs","rt","amp_ratio_pred"
]].to_string(index=False))

# ============================================================
# SAVE
# ============================================================

summary_csv = os.path.join(OUTDIR, "current_best_summary_v9.csv")
full_csv = os.path.join(OUTDIR, "current_best_full_table_v9.csv")
hard_csv = os.path.join(OUTDIR, "current_best_hard_targets_v9.csv")
compact_csv = os.path.join(OUTDIR, "current_best_compact_targets_v9.csv")
boosted_csv = os.path.join(OUTDIR, "current_best_boosted_mixed_targets_v9.csv")
late_csv = os.path.join(OUTDIR, "current_best_peak_too_late_targets_v9.csv")
extreme_csv = os.path.join(OUTDIR, "current_best_extreme_peak_singleton_v9.csv")
ngc2841_csv = os.path.join(OUTDIR, "current_best_ngc2841_singleton_v9.csv")
ngc5005_csv = os.path.join(OUTDIR, "current_best_ngc5005_singleton_v9.csv")
classlaw_csv = os.path.join(OUTDIR, "current_best_class_law_targets_v9.csv")

summary_df.to_csv(summary_csv, index=False)
df.sort_values("rmse", ascending=False).to_csv(full_csv, index=False)
hard_targets_df.to_csv(hard_csv, index=False)
compact_targets_df.to_csv(compact_csv, index=False)
boosted_mixed_df.to_csv(boosted_csv, index=False)
peak_too_late_df.to_csv(late_csv, index=False)
extreme_peak_df.to_csv(extreme_csv, index=False)
ngc2841_df.to_csv(ngc2841_csv, index=False)
ngc5005_df.to_csv(ngc5005_csv, index=False)
class_law_df.to_csv(classlaw_csv, index=False)

print("\nSaved:")
print(" -", summary_csv)
print(" -", full_csv)
print(" -", hard_csv)
print(" -", compact_csv)
print(" -", boosted_csv)
print(" -", late_csv)
print(" -", extreme_csv)
print(" -", ngc2841_csv)
print(" -", ngc5005_csv)
print(" -", classlaw_csv)

print("\nCELL MTS_CURRENT_BEST_REFERENCE_v9 complete.")

CELL: MTS_CURRENT_BEST_REFERENCE_v9
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

CURRENT BEST REFERENCE STATE v9
C_amp_global      = 69.351204
med               = 15.077
p90               = 43.03
hard              = 32.588
fix               = 32.295
sl                = 15.797
peak_too_low_med  = 30.047
peak_too_late_med = 26.662

Branch assignments:
  PEAK_UNDERSHOOT base shape: TAIL(-0.55)
  MIXED_TAPER source: OUTER_TAPER(0.6 * r_peak)
  MIXED_TAPER members: ['NGC2841', 'NGC5005', 'NGC5985', 'UGC02487', 'UGC02953', 'UGC05253', 'UGC11914']
  REMAINDER_INNER_SOFT members: ['UGC11455']
  COMPACT_OUTER_TAPER members: ['UGC06787']
  COMPACT_OUTER_TAPER scale: 0.1 * r_peak
  COMPACT amplitude law:
    log10(amp_ratio) = -0.090071783905 + 0.161718036997 * log10(r_peak/r_bar) + 0.046739762660 * log10(rt/r_peak)
  BOOSTED_MIXED subgroup: ['NGC5985', 'UGC11914']
    amp_boost = (r_peak/r_bar)^0.1
  EXTREME_PEAK singleton: ['UGC02885']
    shape_warp = TAIL(-0.85) then ^0.6

In [ ]:
# ============================================================
# CELL: MTS_HARD_SURVIVOR_AUDIT_v2
#
# Purpose:
#   Audit the CURRENT_BEST_REFERENCE_v9 state and rank the
#   remaining hard survivors using the updated merged class-law
#   model.
#
# Outputs:
#   - hard survivor focus table
#   - residual subclass ranking
#   - updated hard-focus CSVs
#
# Saves:
#   - /content/mts_hard_survivor_audit_v2/hard_survivor_audit_summary_v2.csv
#   - /content/mts_hard_survivor_audit_v2/hard_survivor_audit_full_v2.csv
#   - /content/mts_hard_survivor_audit_v2/hard_survivor_focus_v2.csv
#   - /content/mts_hard_survivor_audit_v2/hard_survivor_subclass_rank_v2.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

print("CELL: MTS_HARD_SURVIVOR_AUDIT_v2")

# ============================================================
# CONFIG
# ============================================================

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL  = 1.25
ALPHA  = 0.10
BETA   = 0.65

ABS_FLOOR = 0.10
F_FLOOR   = 0.02
K_VAL     = 0.05
RBAR_EXP  = 0.75
RPEAK_EXP = 0.25
RS_MAX    = 30.0

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_hard_survivor_audit_v2"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

TAIL_B_FIXED = -0.55

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
OUTER_TAPER_MULT = 0.60

INNER_SOFT_GROUP = {"UGC11455"}
INNER_SOFT_MULT_UGC11455 = 0.35

COMPACT_TAPER_GROUP = {"UGC06787"}
UGC06787_TAPER_MULT = 0.10

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

V9_LOW_ETA   = -0.75
V9_LOW_GAMMA = 0.80
V9_LATE_ETA   = -0.50
V9_LATE_GAMMA = 0.70

HARD_FAILS = [
    "UGC02487","UGC11914","ESO563-G021","UGC02953",
    "NGC5985","UGC03546","NGC0801","UGC05253",
    "NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]

# ============================================================
# HELPERS
# ============================================================

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError(f"No valid rows in {path}")

    order = np.argsort(r[mask])
    return {
        "r":     r[mask][order],
        "vobs":  vobs[mask][order],
        "ev":    ev[mask][order],
        "vgas":  vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul":  vbul[mask][order],
    }

def vbar2(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"] * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    vmax = np.max(vb)
    if vmax <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vmax)[0]
    return float(rot["r"][idx[0]]) if len(idx) else np.nan

def get_r_peak(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, 1.15 * r_max_obs)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])

    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = ri + 0.5 * dr
        rim = ri - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i-1] = cm
        A[i, i]   = -(cm + cp) - 1.0 / Rs**2
        A[i, i+1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R-1, N_R-1] = 1.0
    b[N_R-1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac   = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

# ============================================================
# LOAD MAPS
# ============================================================

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

# ============================================================
# LOAD ROTMODS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

files = bootstrap_rotmods()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ============================================================
# STATIC CACHE
# ============================================================

def build_static_cache(name, rot):
    r_bar  = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar ** RBAR_EXP) * (r_peak ** RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

# ============================================================
# SOURCE / AMP / SHAPE
# ============================================================

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe ** P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

def source_v9(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def amp_v9(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

def shape_v9(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")

    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"
        return s, shape_branch

    if rec["residual_subclass"] == "PEAK_TOO_LOW":
        s = shape_tail(s, V9_LOW_ETA)
        s = shape_gamma(s, V9_LOW_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={V9_LOW_ETA},gamma={V9_LOW_GAMMA})"
    elif rec["residual_subclass"] == "PEAK_TOO_LATE":
        s = shape_tail(s, V9_LATE_ETA)
        s = shape_gamma(s, V9_LATE_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={V9_LATE_ETA},gamma={V9_LATE_GAMMA})"

    return s, shape_branch

# ============================================================
# SOLVE FULL V9
# ============================================================

solved = {}
carriers = []
vmaxes = []

for name, rec in static_cache.items():
    rr, rn, source_branch = source_v9(name, rec)
    r_obs = rec["r_obs"]

    r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
    shape, shape_branch = shape_v9(name, rec, r_obs, r_g, U_g)

    idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
    rt = float(r_g[idx[0]])
    u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
    u_t = max(float(u_fn(rt)), 1e-30)

    vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
    vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
    vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
    carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

    amp_ratio, amp_branch = amp_v9(name, rec, rt)

    solved[name] = {
        "source_branch": source_branch,
        "shape_branch": shape_branch,
        "amp_branch": amp_branch,
        "shape": shape,
        "carrier": carrier,
        "rt": rt,
        "u_t": u_t,
        "amp_ratio": amp_ratio,
    }
    carriers.append(carrier)
    vmaxes.append(rec["vmax_obs"])

C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

rows = []
for name, rec in static_cache.items():
    out = solved[name]
    vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
    vflat_pred  = vflat_model * out["amp_ratio"]
    vpred = vflat_pred * out["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)
    seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

    rows.append({
        "name": name,
        "shape_class": rec["shape_class"],
        "residual_subclass": rec["residual_subclass"],
        "source_branch": out["source_branch"],
        "shape_branch": out["shape_branch"],
        "amp_branch": out["amp_branch"],
        "rmse": rmse,
        "inner_rmse": seg["inner_rmse"],
        "mid_rmse": seg["mid_rmse"],
        "outer_rmse": seg["outer_rmse"],
        "inner_bias": seg["inner_bias"],
        "outer_bias": seg["outer_bias"],
        "peak_r_shift_frac": seg["peak_r_shift_frac"],
        "peak_v_err_frac": seg["peak_v_err_frac"],
        "r_bar": rec["r_bar"],
        "r_peak": rec["r_peak"],
        "Rs": rec["Rs"],
        "rt": out["rt"],
        "u_t": out["u_t"],
        "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
        "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
        "vflat_model": vflat_model,
        "amp_ratio_pred": out["amp_ratio"],
        "vflat_pred": vflat_pred,
    })

df = pd.DataFrame(rows)

# ============================================================
# SUMMARIES
# ============================================================

arr = df["rmse"].values
hard_mask = df["name"].isin(HARD_FAILS).values
fix_mask = df["name"].isin([g for g in HARD_FAILS if g not in ["NGC2841","UGC02487","NGC5985","NGC0801","UGC02885"]]).values
sl_mask = df["name"].isin(["UGC11914","UGC05253","UGC11455"]).values
compact_mask = df["shape_class"].eq("COMPACT_FLOOR_PATHOLOGY").values

summary_df = pd.DataFrame([{
    "C_amp_global": C_amp_global,
    "med": float(np.nanmedian(arr)),
    "p90": float(np.nanpercentile(arr, 90)),
    "hard": float(np.nanmedian(arr[hard_mask])) if hard_mask.any() else np.nan,
    "fix": float(np.nanmedian(arr[fix_mask])) if fix_mask.any() else np.nan,
    "sl": float(np.nanmedian(arr[sl_mask])) if sl_mask.any() else np.nan,
    "compact_med": float(np.nanmedian(arr[compact_mask])) if compact_mask.any() else np.nan,
}])

focus_df = df[df["name"].isin(HARD_FAILS)].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

subclass_rank_df = (
    focus_df.groupby("residual_subclass", dropna=False)
    .agg(
        n=("name", "count"),
        med_rmse=("rmse", "median"),
        mean_rmse=("rmse", "mean"),
        med_inner_bias=("inner_bias", "median"),
        med_outer_bias=("outer_bias", "median"),
        med_peak_r_shift_frac=("peak_r_shift_frac", "median"),
        med_peak_v_err_frac=("peak_v_err_frac", "median"),
    )
    .reset_index()
    .sort_values("med_rmse", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "="*100)
print("V9 REFERENCE CHECK")
print("="*100)
print(f"C_amp_global = {C_amp_global:.6f}")
print(f"med         = {summary_df.loc[0, 'med']:.3f}")
print(f"p90         = {summary_df.loc[0, 'p90']:.2f}")
print(f"hard        = {summary_df.loc[0, 'hard']:.3f}")
print(f"fix         = {summary_df.loc[0, 'fix']:.3f}")
print(f"sl          = {summary_df.loc[0, 'sl']:.3f}")
print(f"compact_med = {summary_df.loc[0, 'compact_med']:.3f}")

print("\n" + "="*100)
print("HARD SURVIVOR FOCUS")
print("="*100)
print(focus_df[[
    "name","shape_class","residual_subclass","source_branch","shape_branch","amp_branch",
    "rmse","inner_rmse","mid_rmse","outer_rmse",
    "inner_bias","outer_bias","peak_r_shift_frac","peak_v_err_frac",
    "rt_over_rpeak","rpeak_over_rbar"
]].to_string(index=False))

print("\n" + "="*100)
print("RESIDUAL SUBCLASS RANKING")
print("="*100)
print(subclass_rank_df.to_string(index=False))

# ============================================================
# SAVE
# ============================================================

summary_csv = os.path.join(OUTDIR, "hard_survivor_audit_summary_v2.csv")
full_csv = os.path.join(OUTDIR, "hard_survivor_audit_full_v2.csv")
focus_csv = os.path.join(OUTDIR, "hard_survivor_focus_v2.csv")
subclass_csv = os.path.join(OUTDIR, "hard_survivor_subclass_rank_v2.csv")

summary_df.to_csv(summary_csv, index=False)
df.sort_values("rmse", ascending=False).to_csv(full_csv, index=False)
focus_df.to_csv(focus_csv, index=False)
subclass_rank_df.to_csv(subclass_csv, index=False)

print("\nSaved:")
print(" -", summary_csv)
print(" -", full_csv)
print(" -", focus_csv)
print(" -", subclass_csv)

print("\nCELL MTS_HARD_SURVIVOR_AUDIT_v2 complete.")

CELL: MTS_HARD_SURVIVOR_AUDIT_v2
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

V9 REFERENCE CHECK
C_amp_global = 69.351204
med         = 15.077
p90         = 43.03
hard        = 32.588
fix         = 32.295
sl          = 15.797
compact_med = 22.896

HARD SURVIVOR FOCUS
       name             shape_class residual_subclass            source_branch                                       shape_branch                  amp_branch      rmse  inner_rmse  mid_rmse  outer_rmse  inner_bias  outer_bias  peak_r_shift_frac  peak_v_err_frac  rt_over_rpeak  rpeak_over_rbar
   UGC02487                   MIXED   INNER_DOMINATED              MIXED_TAPER                                         BASE_SHAPE                  GLOBAL_AMP 97.652516  160.898602 36.666422    3.160808 -150.950223    2.826509           1.666888        -0.128384       1.188343         1.042296
ESO563-G021                   MIXED   TIMING_MISMATCH                     BASE                                         BASE

In [ ]:
# ============================================================
# CELL: MTS_UGC02487_STRUCTURAL_FORENSIC_v2_FIX
#
# Purpose:
#   Same as STRUCTURAL_FORENSIC_v2, but with repaired best-row
#   retrieval that is safe when params contain NaN.
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

print("CELL: MTS_UGC02487_STRUCTURAL_FORENSIC_v2_FIX")

# ============================================================
# CONFIG
# ============================================================

TARGET = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL  = 1.25
ALPHA  = 0.10
BETA   = 0.65

ABS_FLOOR = 0.10
F_FLOOR   = 0.02
K_VAL     = 0.05
RBAR_EXP  = 0.75
RPEAK_EXP = 0.25
RS_MAX    = 30.0

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_structural_forensic_v2_fix"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# locked v9 stack
TAIL_B_FIXED = -0.55

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
OUTER_TAPER_MULT = 0.60

INNER_SOFT_GROUP = {"UGC11455"}
INNER_SOFT_MULT_UGC11455 = 0.35

COMPACT_TAPER_GROUP = {"UGC06787"}
UGC06787_TAPER_MULT = 0.10

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

V9_LOW_ETA   = -0.75
V9_LOW_GAMMA = 0.80
V9_LATE_ETA   = -0.50
V9_LATE_GAMMA = 0.70

# scan grids
INNER_SOFT_RBAR_VALUES = [0.2, 0.35, 0.5, 0.75, 1.0, 1.5]
POWER_RC_VALUES = [0.15, 0.25, 0.35, 0.5, 0.75, 1.0]
POWER_N_VALUES  = [1.5, 2.0, 2.5, 3.0]
EXP_A_VALUES    = [0.10, 0.20, 0.35, 0.50]
EXP_RC_VALUES   = [0.15, 0.25, 0.35, 0.5]
POWER_TAPER_VALUES = [0.4, 0.6, 0.8, 1.0]
TWO_STAGE_RS1 = [0.10, 0.20, 0.35, 0.50]
TWO_STAGE_RS2 = [0.20, 0.40, 0.60, 1.00]
TWO_STAGE_W   = [0.2, 0.3, 0.5, 0.7]

OUTER_TOL = 3.0

# ============================================================
# HELPERS
# ============================================================

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError(f"No valid rows in {path}")

    order = np.argsort(r[mask])
    return {
        "r":     r[mask][order],
        "vobs":  vobs[mask][order],
        "ev":    ev[mask][order],
        "vgas":  vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul":  vbul[mask][order],
    }

def vbar2(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"] * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    vmax = np.max(vb)
    if vmax <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vmax)[0]
    return float(rot["r"][idx[0]]) if len(idx) else np.nan

def get_r_peak(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, 1.15 * r_max_obs)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])

    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = ri + 0.5 * dr
        rim = ri - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i-1] = cm
        A[i, i]   = -(cm + cp) - 1.0 / Rs**2
        A[i, i+1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R-1, N_R-1] = 1.0
    b[N_R-1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac   = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

def same_or_nan(a, b):
    a_nan = pd.isna(a)
    b_nan = pd.isna(b)
    if a_nan and b_nan:
        return True
    if a_nan or b_nan:
        return False
    return a == b

# ============================================================
# LOAD MAPS
# ============================================================

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

# ============================================================
# LOAD ROTMODS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

files = bootstrap_rotmods()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ============================================================
# STATIC CACHE
# ============================================================

def build_static_cache(name, rot):
    r_bar  = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar ** RBAR_EXP) * (r_peak ** RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET not in static_cache:
    raise RuntimeError(f"Target {TARGET} not found in static cache")

# ============================================================
# LOCKED V9 SOURCE / AMP / SHAPE FOR NON-TARGETS
# ============================================================

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe ** P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_inner_power(rot, r_bar, Rc_mult, n_pow):
    r, rho = base_rho_raw(rot)
    Rc = max(float(Rc_mult) * r_bar, 1e-6)
    factor = 1.0 + (Rc / np.maximum(r, 1e-6))**n_pow
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_inner_exp(rot, r_bar, A, Rc_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(Rc_mult) * r_bar, 1e-6)
    factor = 1.0 + A * np.exp(-r / Rc)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_inner_power_plus_taper(rot, r_bar, r_peak, Rc_mult, n_pow, taper_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(Rc_mult) * r_bar, 1e-6)
    factor = 1.0 + (Rc / np.maximum(r, 1e-6))**n_pow
    Rt = max(float(taper_mult) * r_peak, 1e-6)
    rho = rho * factor * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_two_stage(rot, Rs1, Rs2, w):
    r, rho = base_rho_raw(rot)
    s1 = np.exp(-r / max(Rs1, 1e-6))
    s2 = np.exp(-r / max(Rs2, 1e-6))
    rho_new = (1.0 - w) * (rho * s1) + w * (rho * s2)
    return r, normalize_source(rho_new)

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def source_v9_locked(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

def shape_v9_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")

    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"
        return s, shape_branch

    if rec["residual_subclass"] == "PEAK_TOO_LOW":
        s = shape_tail(s, V9_LOW_ETA)
        s = shape_gamma(s, V9_LOW_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={V9_LOW_ETA},gamma={V9_LOW_GAMMA})"
    elif rec["residual_subclass"] == "PEAK_TOO_LATE":
        s = shape_tail(s, V9_LATE_ETA)
        s = shape_gamma(s, V9_LATE_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={V9_LATE_ETA},gamma={V9_LATE_GAMMA})"

    return s, shape_branch

def amp_v9_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

# ============================================================
# SOLVER
# ============================================================

def solve_candidate_for_target(target_mode):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        rot = rec["rot"]

        if name == TARGET:
            case = target_mode["case"]
            p1 = target_mode.get("p1", np.nan)
            p2 = target_mode.get("p2", np.nan)
            p3 = target_mode.get("p3", np.nan)

            if case == "BASE":
                rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
                source_branch = "MIXED_TAPER"
            elif case == "INNER_SOFT_RBAR":
                rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], p1)
                source_branch = f"INNER_SOFT_RBAR({p1})"
            elif case == "INNER_POWER_CONCENTRATION":
                rr, rn = build_source_inner_power(rot, rec["r_bar"], p1, p2)
                source_branch = f"INNER_POWER_CONCENTRATION(Rc={p1},n={p2})"
            elif case == "INNER_EXP_CONCENTRATION":
                rr, rn = build_source_inner_exp(rot, rec["r_bar"], p1, p2)
                source_branch = f"INNER_EXP_CONCENTRATION(A={p1},Rc={p2})"
            elif case == "INNER_POWER_PLUS_TAPER":
                rr, rn = build_source_inner_power_plus_taper(rot, rec["r_bar"], rec["r_peak"], p1, p2, p3)
                source_branch = f"INNER_POWER_PLUS_TAPER(Rc={p1},n={p2},t={p3})"
            elif case == "TWO_STAGE_RESPONSE":
                rr, rn = build_source_two_stage(rot, p1, p2, p3)
                source_branch = f"TWO_STAGE_RESPONSE(Rs1={p1},Rs2={p2},w={p3})"
            else:
                raise ValueError(f"Unknown case: {case}")
        else:
            rr, rn, source_branch = source_v9_locked(name, rec)

        r_obs = rec["r_obs"]
        r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
        shape, shape_branch = shape_v9_locked(name, rec, r_obs, r_g, U_g)

        idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
        rt = float(r_g[idx[0]])
        u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
        u_t = max(float(u_fn(rt)), 1e-30)

        vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
        vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
        vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
        carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

        amp_ratio, amp_branch = amp_v9_locked(name, rec, rt)

        solved[name] = {
            "source_branch": source_branch,
            "shape_branch": shape_branch,
            "amp_branch": amp_branch,
            "shape": shape,
            "carrier": carrier,
            "rt": rt,
            "u_t": u_t,
            "amp_ratio": amp_ratio,
        }
        carriers.append(carrier)
        vmaxes.append(rec["vmax_obs"])

    C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
        vflat_pred  = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)
        seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

        rows.append({
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": rec["Rs"],
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            **seg,
        })

    df = pd.DataFrame(rows)
    trow = df.loc[df["name"] == TARGET].iloc[0].to_dict()

    return {
        "C_amp_global": C_amp_global,
        "df": df,
        "target_row": trow,
    }

# ============================================================
# BASELINE
# ============================================================

baseline = solve_candidate_for_target({"case": "BASE"})
baseline_df = baseline["df"]
baseline_target = baseline["target_row"]

base_med = float(np.nanmedian(baseline_df["rmse"]))
base_p90 = float(np.nanpercentile(baseline_df["rmse"], 90))
base_hard = float(np.nanmedian(baseline_df.loc[baseline_df["name"].isin([
    "UGC02487","UGC11914","ESO563-G021","UGC02953","NGC5985","UGC03546","NGC0801",
    "UGC05253","NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]), "rmse"]))
base_fix = float(np.nanmedian(baseline_df.loc[baseline_df["name"].isin([
    "UGC11914","ESO563-G021","UGC02953","UGC03546","UGC05253","UGC06787","NGC5005","UGC11455"
]), "rmse"]))

base_target_rmse = baseline_target["rmse"]
base_inner_rmse = baseline_target["inner_rmse"]
base_outer_rmse = baseline_target["outer_rmse"]

print("\nCurrent-best-state baseline (v9 locked):")
print(f"  med={base_med:.3f}  p90={base_p90:.2f}  hard={base_hard:.3f}  fix={base_fix:.3f}")
print(f"  {TARGET} rmse={base_target_rmse:.3f}  inner={base_inner_rmse:.3f}  outer={base_outer_rmse:.3f}")

# ============================================================
# SCAN
# ============================================================

scan_rows = []
focus_rows = []

def constrained_score(target_rmse, inner_rmse, outer_rmse):
    outer_penalty = max(outer_rmse - (base_outer_rmse + OUTER_TOL), 0.0)
    return target_rmse + 0.15 * inner_rmse + 2.0 * outer_penalty

def record_case(case, p1=np.nan, p2=np.nan, p3=np.nan):
    result = solve_candidate_for_target({"case": case, "p1": p1, "p2": p2, "p3": p3})
    df = result["df"]
    t = result["target_row"]

    med = float(np.nanmedian(df["rmse"]))
    p90 = float(np.nanpercentile(df["rmse"], 90))
    hard = float(np.nanmedian(df.loc[df["name"].isin([
        "UGC02487","UGC11914","ESO563-G021","UGC02953","NGC5985","UGC03546","NGC0801",
        "UGC05253","NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
    ]), "rmse"]))
    fix = float(np.nanmedian(df.loc[df["name"].isin([
        "UGC11914","ESO563-G021","UGC02953","UGC03546","UGC05253","UGC06787","NGC5005","UGC11455"
    ]), "rmse"]))

    score = constrained_score(t["rmse"], t["inner_rmse"], t["outer_rmse"])

    row = {
        "case": case,
        "param_1": p1,
        "param_2": p2,
        "param_3": p3,
        "med": med,
        "p90": p90,
        "hard": hard,
        "fix": fix,
        "target_rmse": t["rmse"],
        "inner_rmse": t["inner_rmse"],
        "mid_rmse": t["mid_rmse"],
        "outer_rmse": t["outer_rmse"],
        "inner_bias": t["inner_bias"],
        "outer_bias": t["outer_bias"],
        "peak_r_shift_frac": t["peak_r_shift_frac"],
        "peak_v_err_frac": t["peak_v_err_frac"],
        "constrained_score": score,
    }
    scan_rows.append(row)

    focus_row = df.loc[df["name"] == TARGET].copy()
    focus_row["case"] = case
    focus_row["param_1"] = p1
    focus_row["param_2"] = p2
    focus_row["param_3"] = p3
    focus_rows.append(focus_row)

print("\n" + "="*130)
print("SCAN: UGC02487 structural forensic")
print("="*130)
print(f"\n  {'case':26s} {'p1':>7s} {'p2':>7s} {'p3':>7s} {'med':>8s} {'p90':>8s} {'hard':>8s} {'fix':>8s} {'target':>10s} {'inner':>10s} {'outer':>10s} {'score':>10s}")

record_case("BASE")

for k in INNER_SOFT_RBAR_VALUES:
    record_case("INNER_SOFT_RBAR", k)

for Rc in POWER_RC_VALUES:
    for n_pow in POWER_N_VALUES:
        record_case("INNER_POWER_CONCENTRATION", Rc, n_pow)

for A in EXP_A_VALUES:
    for Rc in EXP_RC_VALUES:
        record_case("INNER_EXP_CONCENTRATION", A, Rc)

for Rc in POWER_RC_VALUES:
    for n_pow in [2.0, 2.5, 3.0]:
        for t in POWER_TAPER_VALUES:
            record_case("INNER_POWER_PLUS_TAPER", Rc, n_pow, t)

for Rs1 in TWO_STAGE_RS1:
    for Rs2 in TWO_STAGE_RS2:
        for w in TWO_STAGE_W:
            if Rs2 >= Rs1:
                record_case("TWO_STAGE_RESPONSE", Rs1, Rs2, w)

df_scan = pd.DataFrame(scan_rows)
focus_rows_df = pd.concat(focus_rows, ignore_index=True)

for _, r in df_scan.iterrows():
    p1s = "nan" if pd.isna(r["param_1"]) else f"{r['param_1']:.3f}"
    p2s = "nan" if pd.isna(r["param_2"]) else f"{r['param_2']:.3f}"
    p3s = "nan" if pd.isna(r["param_3"]) else f"{r['param_3']:.3f}"
    print(f"  {str(r['case']):26s} {p1s:>7s} {p2s:>7s} {p3s:>7s} {r['med']:8.3f} {r['p90']:8.2f} {r['hard']:8.3f} {r['fix']:8.3f} {r['target_rmse']:10.3f} {r['inner_rmse']:10.3f} {r['outer_rmse']:10.3f} {r['constrained_score']:10.3f}")

# ============================================================
# BEST ROWS
# ============================================================

best_target = df_scan.sort_values(["target_rmse", "inner_rmse", "outer_rmse", "hard"]).iloc[0]
best_constrained = df_scan.sort_values(["constrained_score", "target_rmse", "outer_rmse"]).iloc[0]
best_med = df_scan.sort_values(["med", "hard", "target_rmse"]).iloc[0]

def fetch_focus_row(store_rows_df, row):
    m = (
        store_rows_df["case"].eq(row["case"]) &
        store_rows_df["param_1"].apply(lambda x: same_or_nan(x, row["param_1"])) &
        store_rows_df["param_2"].apply(lambda x: same_or_nan(x, row["param_2"])) &
        store_rows_df["param_3"].apply(lambda x: same_or_nan(x, row["param_3"]))
    )
    hit = store_rows_df.loc[m]
    if len(hit) == 0:
        raise KeyError(
            f"No stored focus row for case={row['case']} "
            f"p1={row['param_1']} p2={row['param_2']} p3={row['param_3']}"
        )
    return hit.copy()

best_target_focus = fetch_focus_row(focus_rows_df, best_target)
best_constrained_focus = fetch_focus_row(focus_rows_df, best_constrained)
best_med_focus = fetch_focus_row(focus_rows_df, best_med)

print("\n" + "="*130)
print("BEST BY TARGET")
print("="*130)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*130)
print("BEST BY CONSTRAINED SCORE")
print("="*130)
print(best_constrained.to_string())
print(best_constrained_focus.to_string(index=False))

print("\n" + "="*130)
print("BEST BY GLOBAL MEDIAN")
print("="*130)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= base_med) & (df_scan["hard"] <= base_hard)].copy()
pareto = pareto.sort_values(["constrained_score", "target_rmse", "med"])

print("\nPareto vs v9 baseline (med <= base and hard <= base):")
if len(pareto):
    print(pareto[[
        "case","param_1","param_2","param_3",
        "med","p90","hard","fix","target_rmse","inner_rmse","outer_rmse","constrained_score"
    ]].to_string(index=False))
else:
    print("  None")

# ============================================================
# SAVE
# ============================================================

scan_csv = os.path.join(OUTDIR, "ugc02487_structural_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_best_target_breakdown.csv")
best_constrained_csv = os.path.join(OUTDIR, "ugc02487_best_constrained_breakdown.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_best_med_breakdown.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_constrained_focus.to_csv(best_constrained_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_constrained_csv)
print(" -", best_med_csv)

print("\nCELL MTS_UGC02487_STRUCTURAL_FORENSIC_v2_FIX complete.")

CELL: MTS_UGC02487_STRUCTURAL_FORENSIC_v2_FIX
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline (v9 locked):
  med=15.077  p90=43.03  hard=32.588  fix=32.295
  UGC02487 rmse=97.653  inner=160.899  outer=3.161

SCAN: UGC02487 structural forensic

  case                            p1      p2      p3      med      p90     hard      fix     target      inner      outer      score
  BASE                           nan     nan     nan   15.077    43.03   32.588   32.295     97.653    160.899      3.161    121.787
  INNER_SOFT_RBAR              0.200     nan     nan   15.064    43.04   32.629   32.300    110.421    180.755      4.756    137.535
  INNER_SOFT_RBAR              0.350     nan     nan   15.063    43.04   32.631   32.301    111.100    181.807      4.842    138.371
  INNER_SOFT_RBAR              0.500     nan     nan   15.063    43.04   32.634   32.301    112.072    183.306      4.929    139.568
  INNER_SOFT_RBAR              0.750     nan 

In [ ]:
# ============================================================
# CELL: MTS_UGC02487_MULTI_COMPONENT_SOURCE_v1
#
# Purpose:
#   Dedicated follow-up for UGC02487 after STRUCTURAL_FORENSIC_v2_FIX.
#
# Motivation:
#   - single-component inner concentration families failed
#   - two-stage response helped materially
#   - therefore test explicit multi-component source mixtures
#
# Strategy:
#   Keep all non-target galaxies locked to v9 logic.
#   For UGC02487 only, replace source with mixtures of:
#
#     1) mixed-taper background
#     2) compact inner-support component
#     3) broader mid-support component
#
# Families scanned:
#   A) DUAL_SOURCE_MIX
#   B) INNER_PLUS_MID_MIX
#   C) INNER_PLUS_BG_TAPER
#   D) THREE_COMPONENT_MIX
#
# Ranking emphasis:
#   - target_rmse
#   - inner_rmse
#   - constrained score with strong outer-fit protection
#
# Saves:
#   - /content/mts_ugc02487_multi_component_source_v1/ugc02487_multi_component_scan.csv
#   - /content/mts_ugc02487_multi_component_source_v1/ugc02487_multi_component_best_target.csv
#   - /content/mts_ugc02487_multi_component_source_v1/ugc02487_multi_component_best_constrained.csv
#   - /content/mts_ugc02487_multi_component_source_v1/ugc02487_multi_component_best_med.csv
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

print("CELL: MTS_UGC02487_MULTI_COMPONENT_SOURCE_v1")

# ============================================================
# CONFIG
# ============================================================

TARGET = "UGC02487"

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf_val = 0.02
A_src = 0.10
UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6
F_FRAC = 0.60
F_BAR = 0.50

P_VAL  = 1.25
ALPHA  = 0.10
BETA   = 0.65

ABS_FLOOR = 0.10
F_FLOOR   = 0.02
K_VAL     = 0.05
RBAR_EXP  = 0.75
RPEAK_EXP = 0.25
RS_MAX    = 30.0

MASTER_CSV = "/content/mts_v3_rotmod_ladder_audit/mts_v3_rotmod_ladder_master.csv"
CENSUS_CSV = "/content/mts_full_residual_census_v1/residual_census_full_table.csv"

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = "/content/mts_ugc02487_multi_component_source_v1"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

# locked v9 stack
TAIL_B_FIXED = -0.55

MIXED_TAPER_GROUP = {
    "NGC2841", "NGC5005", "NGC5985", "UGC02487",
    "UGC02953", "UGC05253", "UGC11914"
}
OUTER_TAPER_MULT = 0.60

INNER_SOFT_GROUP = {"UGC11455"}
INNER_SOFT_MULT_UGC11455 = 0.35

COMPACT_TAPER_GROUP = {"UGC06787"}
UGC06787_TAPER_MULT = 0.10

BOOSTED_MIXED_GROUP = {"NGC5985", "UGC11914"}
BOOST_Q = 0.10

EXTREME_PEAK_SINGLETON = {"UGC02885"}
EXTREME_PEAK_ETA = -0.85
EXTREME_PEAK_GAMMA = 0.60

NGC2841_SINGLETON = {"NGC2841"}
NGC2841_TAPER_MULT = 0.40

NGC5005_SINGLETON = {"NGC5005"}
NGC5005_TAPER_MULT = 0.30
NGC5005_RP_RB_BOOST_Q = 0.05

COMPACT_AMP_INTERCEPT = -0.0900717839050013
COMPACT_AMP_COEF_RPEAK_OVER_RBAR = 0.16171803699672316
COMPACT_AMP_COEF_RT_OVER_RPEAK   = 0.04673976265998955

V9_LOW_ETA   = -0.75
V9_LOW_GAMMA = 0.80
V9_LATE_ETA   = -0.50
V9_LATE_GAMMA = 0.70

# scan grids
DUAL_RS_A = [0.10, 0.15, 0.20, 0.30]
DUAL_RS_B = [0.30, 0.40, 0.60, 0.80, 1.00]
DUAL_W    = [0.15, 0.25, 0.35, 0.50, 0.70]

INNER_RC   = [0.10, 0.15, 0.20, 0.30]
MID_RT     = [0.40, 0.60, 0.80, 1.00, 1.20]
INNER_W    = [0.10, 0.20, 0.30, 0.40]
MID_W      = [0.10, 0.20, 0.30, 0.40]

BG_TAPER   = [0.40, 0.50, 0.60, 0.80]
INNER_PLUS_BG_W = [0.10, 0.20, 0.30, 0.40, 0.50]

THREE_RS_INNER = [0.10, 0.15, 0.20]
THREE_RS_MID   = [0.50, 0.80, 1.20]
THREE_WI       = [0.10, 0.20, 0.30]
THREE_WM       = [0.10, 0.20, 0.30]

OUTER_TOL = 2.0  # stricter than prior scan

# ============================================================
# HELPERS
# ============================================================

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(win) / win, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError(f"No valid rows in {path}")

    order = np.argsort(r[mask])
    return {
        "r":     r[mask][order],
        "vobs":  vobs[mask][order],
        "ev":    ev[mask][order],
        "vgas":  vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul":  vbul[mask][order],
    }

def vbar2(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"] * np.abs(rot["vbul"])
    )

def get_r_bar(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    vmax = np.max(vb)
    if vmax <= 0:
        return np.nan
    idx = np.where(vb >= F_BAR * vmax)[0]
    return float(rot["r"][idx[0]]) if len(idx) else np.nan

def get_r_peak(rot):
    vb = np.sqrt(np.maximum(vbar2(rot), 0.0))
    if np.max(vb) <= 0:
        return np.nan
    return float(rot["r"][int(np.argmax(vb))])

def solve_field(rho_r, rho_vals, Rs, r_max_obs):
    rmax_g = max(R_MAX, 1.15 * r_max_obs)
    r = np.linspace(R_MIN, rmax_g, N_R)
    dr = r[1] - r[0]

    pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho_g = np.empty_like(r)
    rho_g[r <= rho_r[0]] = float(rho_vals[0])

    mid = (r > rho_r[0]) & (r < rho_r[-1])
    rho_g[mid] = pchip(r[mid])
    rho_g[r >= rho_r[-1]] = 0.0
    rho_g = np.maximum(rho_g, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = ri + 0.5 * dr
        rim = ri - 0.5 * dr
        cm = rim**2 / (ri**2 * dr**2)
        cp = rip**2 / (ri**2 * dr**2)
        A[i, i-1] = cm
        A[i, i]   = -(cm + cp) - 1.0 / Rs**2
        A[i, i+1] = cp
        b[i] = -A_src * rho_g[i] - m_inf_val / Rs**2

    A[N_R-1, N_R-1] = 1.0
    b[N_R-1] = m_inf_val

    m_grid = spsolve(A.tocsr(), b)
    u = np.maximum(m_grid - m_inf_val, 0.0)
    U = cumulative_trapezoid(u, r, initial=0.0)
    return r, u, np.maximum(U, 0.0)

def shape_tail(s, eta):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s / (1.0 + eta * (1.0 - s)), 0.0, 1.0)

def shape_gamma(s, gamma):
    s = np.clip(np.asarray(s, float), 0.0, 1.0)
    return np.clip(s**gamma, 0.0, 1.0)

def normalize_source(rho):
    rho = moving_average(rho, SMOOTH_WIN)
    rho = np.maximum(rho, 0.0)
    pk = np.max(rho)
    if pk <= 0:
        raise RuntimeError("Zero source")
    return np.maximum(rho / pk, SOURCE_FLOOR)

def blend_and_normalize(*pairs):
    total = None
    for w, arr in pairs:
        if w == 0:
            continue
        part = float(w) * np.asarray(arr, float)
        total = part if total is None else total + part
    if total is None:
        raise RuntimeError("No source components in blend")
    return normalize_source(total)

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac   = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

def same_or_nan(a, b):
    a_nan = pd.isna(a)
    b_nan = pd.isna(b)
    if a_nan and b_nan:
        return True
    if a_nan or b_nan:
        return False
    return a == b

# ============================================================
# LOAD MAPS
# ============================================================

df_master = pd.read_csv(MASTER_CSV).copy()
df_master["galaxy_base"] = df_master["galaxy"].str.replace("_rotmod$", "", regex=True)
shape_class_map = dict(zip(df_master["galaxy_base"], df_master["shape_class"]))

df_census = pd.read_csv(CENSUS_CSV).copy()
residual_subclass_map = dict(zip(df_census["name"], df_census["residual_subclass"]))

# ============================================================
# LOAD ROTMODS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    os.makedirs(ROTMOD_DIR, exist_ok=True)
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

files = bootstrap_rotmods()
name_to_path = {}
for fp in files:
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    name_to_path[key] = fp

rots = {}
for name, fp in name_to_path.items():
    try:
        rots[name] = read_rotmod(fp)
    except:
        pass

print(f"Rotmod loaded: {len(rots)}")

# ============================================================
# STATIC CACHE
# ============================================================

def build_static_cache(name, rot):
    r_bar  = get_r_bar(rot)
    r_peak = get_r_peak(rot)
    if not (np.isfinite(r_bar) and r_bar > 0 and np.isfinite(r_peak) and r_peak > 0):
        return None

    rs_floor = max(ABS_FLOOR, F_FLOOR * r_peak)
    r_eff = float((r_bar ** RBAR_EXP) * (r_peak ** RPEAK_EXP))
    Rs = float(np.clip(K_VAL * r_eff, rs_floor, RS_MAX))

    return {
        "name": name,
        "rot": rot,
        "r_obs": rot["r"],
        "vobs": rot["vobs"],
        "shape_class": shape_class_map.get(name, "MIXED"),
        "residual_subclass": residual_subclass_map.get(name, "UNKNOWN"),
        "Rs": Rs,
        "r_bar": r_bar,
        "r_peak": r_peak,
        "vmax_obs": float(np.max(rot["vobs"])),
    }

print("Building static cache...")
static_cache = {}
for name, rot in rots.items():
    try:
        rec = build_static_cache(name, rot)
        if rec is not None:
            static_cache[name] = rec
    except:
        pass
print(f"Cached galaxies: {len(static_cache)}")

if TARGET not in static_cache:
    raise RuntimeError(f"Target {TARGET} not found in static cache")

# ============================================================
# SOURCE BUILDERS
# ============================================================

def base_rho_raw(rot):
    r = rot["r"]
    vb2 = np.maximum(vbar2(rot), 0.0)
    r_safe = np.maximum(r, R_MIN)
    denom = (r_safe ** P_VAL) * np.sqrt(r_safe**2 + R_CORE**2)
    rho = vb2 / np.maximum(denom, 1e-30)
    return r, rho

def build_source_base(rot):
    r, rho = base_rho_raw(rot)
    return r, normalize_source(rho)

def build_source_taper(rot, r_peak, mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(mult) * r_peak, 1e-6)
    rho = rho * np.exp(-r / Rt)
    return r, normalize_source(rho)

def build_source_inner_soft_rbar(rot, r_bar, k):
    r, rho = base_rho_raw(rot)
    Rc = max(float(k) * r_bar, 1e-6)
    factor = (r**2 + (0.35 * Rc)**2) / np.maximum(r**2 + Rc**2, 1e-30)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_inner_bump(rot, r_bar, rc_mult):
    r, rho = base_rho_raw(rot)
    Rc = max(float(rc_mult) * r_bar, 1e-6)
    factor = 1.0 + np.exp(-(r / Rc)**2)
    rho = rho * factor
    return r, normalize_source(rho)

def build_source_mid_support(rot, r_peak, rt_mult):
    r, rho = base_rho_raw(rot)
    Rt = max(float(rt_mult) * r_peak, 1e-6)
    # gentler broad support than full taper
    factor = np.exp(-r / Rt)
    return r, normalize_source(rho * factor)

def build_source_two_stage(rot, Rs1, Rs2, w):
    r, rho = base_rho_raw(rot)
    s1 = np.exp(-r / max(Rs1, 1e-6))
    s2 = np.exp(-r / max(Rs2, 1e-6))
    rho_new = (1.0 - w) * (rho * s1) + w * (rho * s2)
    return r, normalize_source(rho_new)

def build_source_target_multicomponent(rot, rec, case, p1, p2, p3, p4=np.nan):
    # All target-specific models share the same native radial grid.
    r, rho_base_raw = base_rho_raw(rot)

    # background = current mixed-taper logic
    Rt_bg = max(OUTER_TAPER_MULT * rec["r_peak"], 1e-6)
    rho_bg = rho_base_raw * np.exp(-r / Rt_bg)

    if case == "DUAL_SOURCE_MIX":
        Rs1, Rs2, w = p1, p2, p3
        s1 = np.exp(-r / max(Rs1, 1e-6))
        s2 = np.exp(-r / max(Rs2, 1e-6))
        rho_new = (1.0 - w) * (rho_base_raw * s1) + w * (rho_base_raw * s2)
        return r, normalize_source(rho_new), f"DUAL_SOURCE_MIX(Rs1={Rs1},Rs2={Rs2},w={w})"

    if case == "INNER_PLUS_MID_MIX":
        rc_mult, rt_mult, wi, wm = p1, p2, p3, p4
        Rc = max(rc_mult * rec["r_bar"], 1e-6)
        Rt = max(rt_mult * rec["r_peak"], 1e-6)

        rho_inner = rho_base_raw * (1.0 + np.exp(-(r / Rc)**2))
        rho_mid   = rho_base_raw * np.exp(-r / Rt)

        wb = max(1.0 - wi - wm, 0.0)
        rho_new = wb * rho_bg + wi * rho_inner + wm * rho_mid
        return r, normalize_source(rho_new), f"INNER_PLUS_MID_MIX(Rc={rc_mult},Rt={rt_mult},wi={wi},wm={wm})"

    if case == "INNER_PLUS_BG_TAPER":
        rc_mult, taper_mult, wi = p1, p2, p3
        Rc = max(rc_mult * rec["r_bar"], 1e-6)
        Rt = max(taper_mult * rec["r_peak"], 1e-6)

        rho_inner = rho_base_raw * (1.0 + np.exp(-(r / Rc)**2))
        rho_bg2   = rho_base_raw * np.exp(-r / Rt)

        wb = max(1.0 - wi, 0.0)
        rho_new = wb * rho_bg2 + wi * rho_inner
        return r, normalize_source(rho_new), f"INNER_PLUS_BG_TAPER(Rc={rc_mult},t={taper_mult},wi={wi})"

    if case == "THREE_COMPONENT_MIX":
        rs_inner, rs_mid, wi, wm = p1, p2, p3, p4
        s_inner = np.exp(-r / max(rs_inner, 1e-6))
        s_mid   = np.exp(-r / max(rs_mid, 1e-6))

        rho_inner = rho_base_raw * s_inner
        rho_mid   = rho_base_raw * s_mid
        wb = max(1.0 - wi - wm, 0.0)

        rho_new = wb * rho_bg + wi * rho_inner + wm * rho_mid
        return r, normalize_source(rho_new), f"THREE_COMPONENT_MIX(Rsi={rs_inner},Rsm={rs_mid},wi={wi},wm={wm})"

    raise ValueError(f"Unknown target case: {case}")

# ============================================================
# LOCKED V9 NON-TARGET LOGIC
# ============================================================

def compact_amp_ratio(rt, r_peak, r_bar):
    logA = (
        COMPACT_AMP_INTERCEPT
        + COMPACT_AMP_COEF_RPEAK_OVER_RBAR * float(safe_log10(r_peak / max(r_bar, 1e-30)))
        + COMPACT_AMP_COEF_RT_OVER_RPEAK   * float(safe_log10(rt / max(r_peak, 1e-30)))
    )
    return float(10**logA)

def boosted_mixed_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**BOOST_Q)

def ngc5005_amp_ratio(r_peak, r_bar):
    return float((r_peak / max(r_bar, 1e-30))**NGC5005_RP_RB_BOOST_Q)

def source_v9_locked(name, rec):
    rot = rec["rot"]

    if name in NGC2841_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC2841_TAPER_MULT)
        source_branch = f"NGC2841_MIXED_TAPER({NGC2841_TAPER_MULT})"
    elif name in NGC5005_SINGLETON:
        rr, rn = build_source_taper(rot, rec["r_peak"], NGC5005_TAPER_MULT)
        source_branch = f"NGC5005_MIXED_TAPER({NGC5005_TAPER_MULT})"
    elif name in MIXED_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
        source_branch = "MIXED_TAPER"
    elif name in INNER_SOFT_GROUP:
        rr, rn = build_source_inner_soft_rbar(rot, rec["r_bar"], INNER_SOFT_MULT_UGC11455)
        source_branch = "REMAINDER_INNER_SOFT"
    elif name in COMPACT_TAPER_GROUP:
        rr, rn = build_source_taper(rot, rec["r_peak"], UGC06787_TAPER_MULT)
        source_branch = "COMPACT_OUTER_TAPER"
    else:
        rr, rn = build_source_base(rot)
        source_branch = "BASE"

    return rr, rn, source_branch

def shape_v9_locked(name, rec, r_obs, r_g, U_g):
    U_inf = float(np.max(U_g))
    U_fn = interp1d(r_g, U_g, bounds_error=False, fill_value="extrapolate")

    s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
    shape_branch = "BASE_SHAPE"

    if rec["shape_class"] == "PEAK_UNDERSHOOT":
        s = shape_tail(s, TAIL_B_FIXED)
        shape_branch = "PEAK_UNDERSHOOT_TAIL"

    if name in EXTREME_PEAK_SINGLETON:
        s = np.clip(U_fn(r_obs) / U_inf, 0.0, 1.0)
        s = shape_tail(s, EXTREME_PEAK_ETA)
        s = shape_gamma(s, EXTREME_PEAK_GAMMA)
        shape_branch = f"EXTREME_PEAK_HYBRID(eta={EXTREME_PEAK_ETA},gamma={EXTREME_PEAK_GAMMA})"
        return s, shape_branch

    if rec["residual_subclass"] == "PEAK_TOO_LOW":
        s = shape_tail(s, V9_LOW_ETA)
        s = shape_gamma(s, V9_LOW_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LOW_TAIL_GAMMA(eta={V9_LOW_ETA},gamma={V9_LOW_GAMMA})"
    elif rec["residual_subclass"] == "PEAK_TOO_LATE":
        s = shape_tail(s, V9_LATE_ETA)
        s = shape_gamma(s, V9_LATE_GAMMA)
        shape_branch = f"CLASS_PEAK_TOO_LATE_TAIL_GAMMA(eta={V9_LATE_ETA},gamma={V9_LATE_GAMMA})"

    return s, shape_branch

def amp_v9_locked(name, rec, rt):
    amp_ratio = 1.0
    amp_branch = "GLOBAL_AMP"

    if rec["shape_class"] == "COMPACT_FLOOR_PATHOLOGY":
        amp_ratio *= compact_amp_ratio(rt, rec["r_peak"], rec["r_bar"])
        amp_branch = "COMPACT_AMP_LAW"

    if name in BOOSTED_MIXED_GROUP:
        amp_ratio *= boosted_mixed_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"BOOSTED_MIXED_AMP(q={BOOST_Q})"
        else:
            amp_branch += f"+BOOSTED_MIXED(q={BOOST_Q})"

    if name in NGC5005_SINGLETON:
        amp_ratio *= ngc5005_amp_ratio(rec["r_peak"], rec["r_bar"])
        if amp_branch == "GLOBAL_AMP":
            amp_branch = f"NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"
        else:
            amp_branch += f"+NGC5005_RP_RB_BOOST(q={NGC5005_RP_RB_BOOST_Q})"

    return amp_ratio, amp_branch

# ============================================================
# SOLVER
# ============================================================

def solve_candidate_for_target(target_mode):
    solved = {}
    carriers = []
    vmaxes = []

    for name, rec in static_cache.items():
        rot = rec["rot"]

        if name == TARGET:
            case = target_mode["case"]
            p1 = target_mode.get("p1", np.nan)
            p2 = target_mode.get("p2", np.nan)
            p3 = target_mode.get("p3", np.nan)
            p4 = target_mode.get("p4", np.nan)
            rr, rn, source_branch = build_source_target_multicomponent(rot, rec, case, p1, p2, p3, p4)
        else:
            rr, rn, source_branch = source_v9_locked(name, rec)

        r_obs = rec["r_obs"]
        r_g, u_g, U_g = solve_field(rr, rn, rec["Rs"], float(r_obs.max()))
        shape, shape_branch = shape_v9_locked(name, rec, r_obs, r_g, U_g)

        idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
        rt = float(r_g[idx[0]])
        u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
        u_t = max(float(u_fn(rt)), 1e-30)

        vb2 = np.maximum(vbar2(rec["rot"]), 0.0)
        vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
        vb2_rpeak = max(float(vb2_fn(rec["r_peak"])), 1e-30)
        carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA

        amp_ratio, amp_branch = amp_v9_locked(name, rec, rt)

        solved[name] = {
            "source_branch": source_branch,
            "shape_branch": shape_branch,
            "amp_branch": amp_branch,
            "shape": shape,
            "carrier": carrier,
            "rt": rt,
            "u_t": u_t,
            "amp_ratio": amp_ratio,
        }
        carriers.append(carrier)
        vmaxes.append(rec["vmax_obs"])

    C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))

    rows = []
    for name, rec in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
        vflat_pred  = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec["vobs"], vpred)
        seg = segment_metrics(rec["r_obs"], rec["vobs"], vpred)

        rows.append({
            "name": name,
            "shape_class": rec["shape_class"],
            "source_branch": out["source_branch"],
            "shape_branch": out["shape_branch"],
            "amp_branch": out["amp_branch"],
            "rmse": rmse,
            "vflat_model": vflat_model,
            "amp_ratio_pred": out["amp_ratio"],
            "vflat_pred": vflat_pred,
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            "Rs": rec["Rs"],
            "rt": out["rt"],
            "u_t": out["u_t"],
            "rt_over_rpeak": float(out["rt"] / max(rec["r_peak"], 1e-30)),
            "rpeak_over_rbar": float(rec["r_peak"] / max(rec["r_bar"], 1e-30)),
            **seg,
        })

    df = pd.DataFrame(rows)
    trow = df.loc[df["name"] == TARGET].iloc[0].to_dict()

    return {"C_amp_global": C_amp_global, "df": df, "target_row": trow}

# ============================================================
# BASELINE
# ============================================================

baseline = solve_candidate_for_target({"case": "DUAL_SOURCE_MIX", "p1": 0.1, "p2": 0.1, "p3": 0.0})
# use actual v9 target baseline from locked source instead of this fake identity row
base_locked = solve_candidate_for_target({"case": "INNER_PLUS_BG_TAPER", "p1": 0.1, "p2": OUTER_TAPER_MULT, "p3": 0.0})
# overwrite target with current true v9 locked baseline
# easiest robust way: run explicit mixed-taper target equivalent
def run_true_target_baseline():
    rec = static_cache[TARGET]
    rot = rec["rot"]
    rr, rn = build_source_taper(rot, rec["r_peak"], OUTER_TAPER_MULT)
    # Temporarily patch via dedicated case-like solve
    solved = {}
    carriers = []
    vmaxes = []
    for name, rec2 in static_cache.items():
        if name == TARGET:
            rr2, rn2, source_branch = rr, rn, "MIXED_TAPER"
        else:
            rr2, rn2, source_branch = source_v9_locked(name, rec2)
        r_obs = rec2["r_obs"]
        r_g, u_g, U_g = solve_field(rr2, rn2, rec2["Rs"], float(r_obs.max()))
        shape, shape_branch = shape_v9_locked(name, rec2, r_obs, r_g, U_g)
        idx = np.where(U_g >= F_FRAC * float(np.max(U_g)))[0]
        rt = float(r_g[idx[0]])
        u_fn = interp1d(r_g, u_g, bounds_error=False, fill_value="extrapolate")
        u_t = max(float(u_fn(rt)), 1e-30)
        vb2 = np.maximum(vbar2(rec2["rot"]), 0.0)
        vb2_fn = interp1d(r_obs, vb2, bounds_error=False, fill_value="extrapolate")
        vb2_rpeak = max(float(vb2_fn(rec2["r_peak"])), 1e-30)
        carrier = (rt * u_t)**ALPHA * vb2_rpeak**BETA
        amp_ratio, amp_branch = amp_v9_locked(name, rec2, rt)
        solved[name] = {
            "source_branch": source_branch,
            "shape_branch": shape_branch,
            "amp_branch": amp_branch,
            "shape": shape,
            "carrier": carrier,
            "rt": rt,
            "u_t": u_t,
            "amp_ratio": amp_ratio,
        }
        carriers.append(carrier)
        vmaxes.append(rec2["vmax_obs"])
    C_amp_global = float(np.mean(np.asarray(vmaxes)**2) / np.mean(np.asarray(carriers)))
    rows = []
    for name, rec2 in static_cache.items():
        out = solved[name]
        vflat_model = float(np.sqrt(max(C_amp_global * out["carrier"], 0.0)))
        vflat_pred  = vflat_model * out["amp_ratio"]
        vpred = vflat_pred * out["shape"]
        rmse = safe_rmse(rec2["vobs"], vpred)
        seg = segment_metrics(rec2["r_obs"], rec2["vobs"], vpred)
        rows.append({
            "name": name, "rmse": rmse, **seg
        })
    return pd.DataFrame(rows)

baseline_df = run_true_target_baseline()
baseline_target = baseline_df.loc[baseline_df["name"] == TARGET].iloc[0]

base_med = float(np.nanmedian(baseline_df["rmse"]))
base_p90 = float(np.nanpercentile(baseline_df["rmse"], 90))
base_hard = float(np.nanmedian(baseline_df.loc[baseline_df["name"].isin([
    "UGC02487","UGC11914","ESO563-G021","UGC02953","NGC5985","UGC03546","NGC0801",
    "UGC05253","NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
]), "rmse"]))
base_fix = float(np.nanmedian(baseline_df.loc[baseline_df["name"].isin([
    "UGC11914","ESO563-G021","UGC02953","UGC03546","UGC05253","UGC06787","NGC5005","UGC11455"
]), "rmse"]))

base_target_rmse = float(baseline_target["rmse"])
base_inner_rmse = float(baseline_target["inner_rmse"])
base_outer_rmse = float(baseline_target["outer_rmse"])

print("\nCurrent-best-state baseline (v9 locked):")
print(f"  med={base_med:.3f}  p90={base_p90:.2f}  hard={base_hard:.3f}  fix={base_fix:.3f}")
print(f"  {TARGET} rmse={base_target_rmse:.3f}  inner={base_inner_rmse:.3f}  outer={base_outer_rmse:.3f}")

# ============================================================
# SCAN
# ============================================================

scan_rows = []
focus_rows = []

def constrained_score(target_rmse, inner_rmse, outer_rmse):
    outer_penalty = max(outer_rmse - (base_outer_rmse + OUTER_TOL), 0.0)
    return target_rmse + 0.15 * inner_rmse + 4.0 * outer_penalty

def record_case(case, p1=np.nan, p2=np.nan, p3=np.nan, p4=np.nan):
    result = solve_candidate_for_target({"case": case, "p1": p1, "p2": p2, "p3": p3, "p4": p4})
    df = result["df"]
    t = result["target_row"]

    med = float(np.nanmedian(df["rmse"]))
    p90 = float(np.nanpercentile(df["rmse"], 90))
    hard = float(np.nanmedian(df.loc[df["name"].isin([
        "UGC02487","UGC11914","ESO563-G021","UGC02953","NGC5985","UGC03546","NGC0801",
        "UGC05253","NGC2841","UGC02885","UGC06787","NGC5005","UGC11455"
    ]), "rmse"]))
    fix = float(np.nanmedian(df.loc[df["name"].isin([
        "UGC11914","ESO563-G021","UGC02953","UGC03546","UGC05253","UGC06787","NGC5005","UGC11455"
    ]), "rmse"]))

    score = constrained_score(t["rmse"], t["inner_rmse"], t["outer_rmse"])

    row = {
        "case": case,
        "param_1": p1,
        "param_2": p2,
        "param_3": p3,
        "param_4": p4,
        "med": med,
        "p90": p90,
        "hard": hard,
        "fix": fix,
        "target_rmse": t["rmse"],
        "inner_rmse": t["inner_rmse"],
        "mid_rmse": t["mid_rmse"],
        "outer_rmse": t["outer_rmse"],
        "inner_bias": t["inner_bias"],
        "outer_bias": t["outer_bias"],
        "peak_r_shift_frac": t["peak_r_shift_frac"],
        "peak_v_err_frac": t["peak_v_err_frac"],
        "constrained_score": score,
    }
    scan_rows.append(row)

    focus_row = df.loc[df["name"] == TARGET].copy()
    focus_row["case"] = case
    focus_row["param_1"] = p1
    focus_row["param_2"] = p2
    focus_row["param_3"] = p3
    focus_row["param_4"] = p4
    focus_rows.append(focus_row)

print("\n" + "="*150)
print("SCAN: UGC02487 multi-component source")
print("="*150)
print(f"\n  {'case':24s} {'p1':>7s} {'p2':>7s} {'p3':>7s} {'p4':>7s} {'med':>8s} {'p90':>8s} {'hard':>8s} {'fix':>8s} {'target':>10s} {'inner':>10s} {'outer':>10s} {'score':>10s}")

# A) dual source response
for rs1 in DUAL_RS_A:
    for rs2 in DUAL_RS_B:
        if rs2 >= rs1:
            for w in DUAL_W:
                record_case("DUAL_SOURCE_MIX", rs1, rs2, w)

# B) inner + mid + residual bg
for rc in INNER_RC:
    for rtm in MID_RT:
        for wi in INNER_W:
            for wm in MID_W:
                if wi + wm < 1.0:
                    record_case("INNER_PLUS_MID_MIX", rc, rtm, wi, wm)

# C) inner + retuned background taper
for rc in INNER_RC:
    for tb in BG_TAPER:
        for wi in INNER_PLUS_BG_W:
            record_case("INNER_PLUS_BG_TAPER", rc, tb, wi)

# D) three-component exponentials
for rsi in THREE_RS_INNER:
    for rsm in THREE_RS_MID:
        if rsm >= rsi:
            for wi in THREE_WI:
                for wm in THREE_WM:
                    if wi + wm < 1.0:
                        record_case("THREE_COMPONENT_MIX", rsi, rsm, wi, wm)

df_scan = pd.DataFrame(scan_rows)
focus_rows_df = pd.concat(focus_rows, ignore_index=True)

for _, r in df_scan.iterrows():
    def fmt(x):
        return "nan" if pd.isna(x) else f"{x:.3f}"
    print(
        f"  {str(r['case']):24s} {fmt(r['param_1']):>7s} {fmt(r['param_2']):>7s} "
        f"{fmt(r['param_3']):>7s} {fmt(r['param_4']):>7s} "
        f"{r['med']:8.3f} {r['p90']:8.2f} {r['hard']:8.3f} {r['fix']:8.3f} "
        f"{r['target_rmse']:10.3f} {r['inner_rmse']:10.3f} {r['outer_rmse']:10.3f} "
        f"{r['constrained_score']:10.3f}"
    )

# ============================================================
# BEST ROWS
# ============================================================

best_target = df_scan.sort_values(["target_rmse", "inner_rmse", "outer_rmse", "hard"]).iloc[0]
best_constrained = df_scan.sort_values(["constrained_score", "target_rmse", "outer_rmse"]).iloc[0]
best_med = df_scan.sort_values(["med", "hard", "target_rmse"]).iloc[0]

def fetch_focus_row(store_rows_df, row):
    m = (
        store_rows_df["case"].eq(row["case"]) &
        store_rows_df["param_1"].apply(lambda x: same_or_nan(x, row["param_1"])) &
        store_rows_df["param_2"].apply(lambda x: same_or_nan(x, row["param_2"])) &
        store_rows_df["param_3"].apply(lambda x: same_or_nan(x, row["param_3"])) &
        store_rows_df["param_4"].apply(lambda x: same_or_nan(x, row["param_4"]))
    )
    hit = store_rows_df.loc[m]
    if len(hit) == 0:
        raise KeyError(
            f"No stored focus row for case={row['case']} "
            f"p1={row['param_1']} p2={row['param_2']} p3={row['param_3']} p4={row['param_4']}"
        )
    return hit.copy()

best_target_focus = fetch_focus_row(focus_rows_df, best_target)
best_constrained_focus = fetch_focus_row(focus_rows_df, best_constrained)
best_med_focus = fetch_focus_row(focus_rows_df, best_med)

print("\n" + "="*150)
print("BEST BY TARGET")
print("="*150)
print(best_target.to_string())
print(best_target_focus.to_string(index=False))

print("\n" + "="*150)
print("BEST BY CONSTRAINED SCORE")
print("="*150)
print(best_constrained.to_string())
print(best_constrained_focus.to_string(index=False))

print("\n" + "="*150)
print("BEST BY GLOBAL MEDIAN")
print("="*150)
print(best_med.to_string())
print(best_med_focus.to_string(index=False))

pareto = df_scan[(df_scan["med"] <= base_med) & (df_scan["hard"] <= base_hard)].copy()
pareto = pareto.sort_values(["constrained_score", "target_rmse", "med"])

print("\nPareto vs v9 baseline (med <= base and hard <= base):")
if len(pareto):
    print(pareto[[
        "case","param_1","param_2","param_3","param_4",
        "med","p90","hard","fix","target_rmse","inner_rmse","outer_rmse","constrained_score"
    ]].to_string(index=False))
else:
    print("  None")

# ============================================================
# SAVE
# ============================================================

scan_csv = os.path.join(OUTDIR, "ugc02487_multi_component_scan.csv")
best_target_csv = os.path.join(OUTDIR, "ugc02487_multi_component_best_target.csv")
best_constrained_csv = os.path.join(OUTDIR, "ugc02487_multi_component_best_constrained.csv")
best_med_csv = os.path.join(OUTDIR, "ugc02487_multi_component_best_med.csv")

df_scan.to_csv(scan_csv, index=False)
best_target_focus.to_csv(best_target_csv, index=False)
best_constrained_focus.to_csv(best_constrained_csv, index=False)
best_med_focus.to_csv(best_med_csv, index=False)

print("\nSaved:")
print(" -", scan_csv)
print(" -", best_target_csv)
print(" -", best_constrained_csv)
print(" -", best_med_csv)

print("\nCELL MTS_UGC02487_MULTI_COMPONENT_SOURCE_v1 complete.")

CELL: MTS_UGC02487_MULTI_COMPONENT_SOURCE_v1
Rotmod loaded: 175
Building static cache...
Cached galaxies: 175

Current-best-state baseline (v9 locked):
  med=15.077  p90=43.03  hard=32.588  fix=32.295
  UGC02487 rmse=97.653  inner=160.899  outer=3.161

SCAN: UGC02487 multi-component source

  case                          p1      p2      p3      p4      med      p90     hard      fix     target      inner      outer      score
  DUAL_SOURCE_MIX            0.100   0.300   0.150     nan   15.073    43.03   32.602   32.296     82.442    135.424      4.042    102.756
  DUAL_SOURCE_MIX            0.100   0.300   0.250     nan   15.073    43.03   32.602   32.296     82.442    135.424      4.042    102.756
  DUAL_SOURCE_MIX            0.100   0.300   0.350     nan   15.073    43.03   32.602   32.296     82.442    135.424      4.042    102.756
  DUAL_SOURCE_MIX            0.100   0.300   0.500     nan   15.073    43.03   32.602   32.296     82.442    135.424      4.042    102.756
  DUAL_SOURCE

In [ ]:
# ============================================================
# CELL: MTS_CLAUDE_AUDIT_v1
#
# Purpose:
#   Independent audit of the "zero per-galaxy fitting" pipeline:
#
#     1) g_b(r) = Vbar^2 / r
#     2) m(r) = g_b(r)^gamma
#     3) u(r) = ∫ m(s) ds
#     4) Rs from u(Rs) = frac * u_total
#     5) V(r)/Vinf = 1 - exp[-u(r)/u(Rs)]
#     6) Vinf^4 = G * Mb * a_dagger
#
# What this checks:
#   - global RMSE distribution across catalogue
#   - fraction of galaxies below 20 km/s
#   - worst galaxies
#   - shape-only vs amplitude error decomposition
#   - local sensitivity around gamma=1/3 and frac=0.135
#
# Notes:
#   - no per-galaxy fitting
#   - fixed photometric M/L priors only
#   - intended as a clean verification / falsification pass
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

print("CELL: MTS_CLAUDE_AUDIT_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_claude_audit_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
os.makedirs(ROTMOD_DIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

UPS_DISK = 0.5
UPS_BUL  = 0.7

# Claude-claimed law values
GAMMA_REF   = 1.0 / 3.0
FRAC_REF    = 0.135
A_DAGGER_REF = 3209.0   # (km/s)^2 / kpc

# physical G in convenient units:
# G = 4.30091e-6 kpc (km/s)^2 / Msun
G_KPC_KMS2_MSun = 4.30091e-6

OUTDIR = "/content/mts_claude_audit_v1_outputs"
os.makedirs(OUTDIR, exist_ok=True)

# optional galaxy subsets of interest
FOCUS_GALS = {
    "NGC2841", "UGC02487", "UGC11914", "NGC5005", "NGC5985",
    "UGC02885", "ESO563-G021", "NGC6946", "NGC2903", "NGC7814"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    mask = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not mask.any():
        raise ValueError(f"No valid rows in {path}")

    order = np.argsort(r[mask])
    return {
        "r":     r[mask][order],
        "vobs":  vobs[mask][order],
        "ev":    ev[mask][order],
        "vgas":  vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul":  vbul[mask][order],
    }

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot, ups_disk=UPS_DISK, ups_bul=UPS_BUL):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        ups_disk * rot["vdisk"] * np.abs(rot["vdisk"]) +
        ups_bul  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def cumulative_baryonic_mass_proxy(rot, ups_disk=UPS_DISK, ups_bul=UPS_BUL):
    """
    Very simple mass proxy from circular components:
      M(<r) ~ r * Vbar^2 / G
    and total Mb from outermost point.
    This is a practical audit estimator, not a final derivation.
    """
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot, ups_disk, ups_bul), 0.0)
    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb = float(np.nanmax(Menc))
    return Mb, Menc

def build_pipeline_prediction(rot, gamma=GAMMA_REF, frac=FRAC_REF, a_dagger=A_DAGGER_REF,
                              ups_disk=UPS_DISK, ups_bul=UPS_BUL):
    r = rot["r"]
    vobs = rot["vobs"]

    vb2 = np.maximum(baryonic_v2(rot, ups_disk, ups_bul), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)

    # loading field
    m = np.maximum(gb, 0.0) ** gamma

    # cumulative response variable
    u = cumulative_trapezoid(m, r, initial=0.0)
    u_tot = float(np.max(u))

    if not np.isfinite(u_tot) or u_tot <= 0:
        return None

    u_target = frac * u_tot
    idx = np.where(u >= u_target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])
    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])

    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None

    # universal shape response
    shape = 1.0 - np.exp(-u / u_Rs)

    # amplitude
    Mb, Menc = cumulative_baryonic_mass_proxy(rot, ups_disk, ups_bul)
    if not np.isfinite(Mb) or Mb <= 0:
        return None

    Vinf = float((G_KPC_KMS2_MSun * Mb * a_dagger) ** 0.25)

    vpred = Vinf * shape

    # shape-only optimal amplitude diagnostic
    denom = np.dot(shape, shape)
    if denom > 0:
        Vinf_shape_opt = float(np.dot(vobs, shape) / denom)
        vpred_shape_opt = Vinf_shape_opt * shape
    else:
        Vinf_shape_opt = np.nan
        vpred_shape_opt = np.full_like(vobs, np.nan)

    return {
        "r": r,
        "vobs": vobs,
        "vb2": vb2,
        "gb": gb,
        "m": m,
        "u": u,
        "u_tot": u_tot,
        "Rs": Rs,
        "u_Rs": u_Rs,
        "shape": shape,
        "Mb": Mb,
        "Vinf": Vinf,
        "vpred": vpred,
        "Vinf_shape_opt": Vinf_shape_opt,
        "vpred_shape_opt": vpred_shape_opt,
        "rmse_total": safe_rmse(vobs, vpred),
        "rmse_shape_only": safe_rmse(vobs, vpred_shape_opt),
    }

def audit_catalog(gamma=GAMMA_REF, frac=FRAC_REF, a_dagger=A_DAGGER_REF,
                  ups_disk=UPS_DISK, ups_bul=UPS_BUL):
    files = bootstrap_rotmods()
    rows = []
    curves = {}

    for fp in files:
        name = galaxy_name_from_path(fp)
        try:
            rot = read_rotmod(fp)
            out = build_pipeline_prediction(rot, gamma, frac, a_dagger, ups_disk, ups_bul)
            if out is None:
                continue

            rows.append({
                "name": name,
                "npts": len(out["r"]),
                "Rs": out["Rs"],
                "u_tot": out["u_tot"],
                "u_Rs": out["u_Rs"],
                "Mb": out["Mb"],
                "Vinf": out["Vinf"],
                "rmse_total": out["rmse_total"],
                "rmse_shape_only": out["rmse_shape_only"],
                "amp_penalty": out["rmse_total"] - out["rmse_shape_only"],
                "vmax_obs": float(np.max(out["vobs"])),
                "vmax_pred": float(np.max(out["vpred"])),
            })
            curves[name] = out
        except Exception as e:
            print(f"[skip] {name}: {e}")

    df = pd.DataFrame(rows).sort_values("rmse_total").reset_index(drop=True)
    return df, curves

def summarize_catalog(df, label="catalog"):
    if len(df) == 0:
        print(f"{label}: empty")
        return

    med = float(np.nanmedian(df["rmse_total"]))
    p90 = float(np.nanpercentile(df["rmse_total"], 90))
    frac20 = float(np.mean(df["rmse_total"] <= 20.0))
    frac30 = float(np.mean(df["rmse_total"] <= 30.0))
    shape_med = float(np.nanmedian(df["rmse_shape_only"]))
    amp_pen_med = float(np.nanmedian(df["amp_penalty"]))

    print(f"\nSUMMARY [{label}]")
    print("-" * 60)
    print(f"N galaxies            : {len(df)}")
    print(f"median RMSE           : {med:.3f} km/s")
    print(f"p90 RMSE              : {p90:.3f} km/s")
    print(f"fraction <= 20 km/s   : {frac20:.3f}")
    print(f"fraction <= 30 km/s   : {frac30:.3f}")
    print(f"median shape-only RMSE: {shape_med:.3f} km/s")
    print(f"median amp penalty    : {amp_pen_med:.3f} km/s")

def plot_focus_curves(df, curves, outdir=OUTDIR):
    focus = list(FOCUS_GALS.intersection(set(df["name"])))
    if not focus:
        return

    for name in sorted(focus):
        out = curves[name]
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(out["r"], out["vobs"], "o", ms=3, label="Vobs")
        ax.plot(out["r"], out["vpred"], "-", lw=2, label="Vpred")
        ax.plot(out["r"], out["vpred_shape_opt"], "--", lw=2, label="shape-only-opt-amp")
        ax.set_title(
            f"{name} | RMSE={out['rmse_total']:.1f} | shape-only={out['rmse_shape_only']:.1f}"
        )
        ax.set_xlabel("r [kpc]")
        ax.set_ylabel("V [km/s]")
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f"{name}_audit_curve.png"), dpi=160)
        plt.close(fig)

def local_sensitivity_scan(gamma_vals, frac_vals, a_dagger=A_DAGGER_REF):
    rows = []
    for gamma in gamma_vals:
        for frac in frac_vals:
            df, _ = audit_catalog(gamma=gamma, frac=frac, a_dagger=a_dagger)
            if len(df) == 0:
                continue
            rows.append({
                "gamma": gamma,
                "frac": frac,
                "med": float(np.nanmedian(df["rmse_total"])),
                "p90": float(np.nanpercentile(df["rmse_total"], 90)),
                "frac20": float(np.mean(df["rmse_total"] <= 20.0)),
                "shape_med": float(np.nanmedian(df["rmse_shape_only"])),
            })
            print(f"[grid] gamma={gamma:.4f} frac={frac:.4f} done")
    return pd.DataFrame(rows).sort_values(["med", "p90", "shape_med"])

# ============================================================
# MAIN AUDIT
# ============================================================

df_ref, curves_ref = audit_catalog(
    gamma=GAMMA_REF,
    frac=FRAC_REF,
    a_dagger=A_DAGGER_REF,
    ups_disk=UPS_DISK,
    ups_bul=UPS_BUL,
)

summarize_catalog(df_ref, label="Claude reference")
plot_focus_curves(df_ref, curves_ref)

# worst / best tables
best_csv = os.path.join(OUTDIR, "audit_best_20.csv")
worst_csv = os.path.join(OUTDIR, "audit_worst_20.csv")
all_csv = os.path.join(OUTDIR, "audit_full_table.csv")

df_ref.head(20).to_csv(best_csv, index=False)
df_ref.tail(20).to_csv(worst_csv, index=False)
df_ref.to_csv(all_csv, index=False)

print("\nBEST 20")
print(df_ref.head(20)[["name", "rmse_total", "rmse_shape_only", "amp_penalty", "Rs", "Mb", "Vinf"]].to_string(index=False))

print("\nWORST 20")
print(df_ref.tail(20)[["name", "rmse_total", "rmse_shape_only", "amp_penalty", "Rs", "Mb", "Vinf"]].to_string(index=False))

# shape vs amplitude diagnostic:
# galaxies with large total error but much smaller shape-only error are likely amplitude-closure failures
df_diag = df_ref.copy()
df_diag["shape_good_amp_bad"] = (df_diag["rmse_shape_only"] <= 0.65 * df_diag["rmse_total"])

shape_amp_csv = os.path.join(OUTDIR, "audit_shape_vs_amplitude_diagnostic.csv")
df_diag.sort_values(["rmse_total", "amp_penalty"], ascending=[False, False]).to_csv(shape_amp_csv, index=False)

print("\nLIKELY AMPLITUDE-DOMINATED FAILURES")
print(
    df_diag[df_diag["shape_good_amp_bad"]]
    .sort_values("rmse_total", ascending=False)
    .head(20)[["name", "rmse_total", "rmse_shape_only", "amp_penalty"]]
    .to_string(index=False)
)

# local sensitivity around Claude values
gamma_grid = [0.28, 0.30, 0.32, 1/3, 0.35, 0.36, 0.38]
frac_grid  = [0.110, 0.120, 0.135, 0.150, 0.165]

df_sense = local_sensitivity_scan(gamma_grid, frac_grid, a_dagger=A_DAGGER_REF)
sense_csv = os.path.join(OUTDIR, "audit_local_sensitivity.csv")
df_sense.to_csv(sense_csv, index=False)

print("\nTOP LOCAL SENSITIVITY RESULTS")
print(df_sense.head(20).to_string(index=False))

# quick histograms
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df_ref["rmse_total"], bins=30)
ax.set_xlabel("RMSE [km/s]")
ax.set_ylabel("count")
ax.set_title("Catalogue RMSE distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "audit_rmse_hist.png"), dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(df_ref["rmse_shape_only"], df_ref["rmse_total"], s=12)
mx = max(df_ref["rmse_total"].max(), df_ref["rmse_shape_only"].max())
ax.plot([0, mx], [0, mx], "k--", lw=1)
ax.set_xlabel("shape-only RMSE [km/s]")
ax.set_ylabel("total RMSE [km/s]")
ax.set_title("Shape-only vs total error")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "audit_shape_vs_total.png"), dpi=160)
plt.close(fig)

print("\nSaved:")
print(" -", best_csv)
print(" -", worst_csv)
print(" -", all_csv)
print(" -", shape_amp_csv)
print(" -", sense_csv)
print(" -", os.path.join(OUTDIR, "audit_rmse_hist.png"))
print(" -", os.path.join(OUTDIR, "audit_shape_vs_total.png"))
print("\nCELL MTS_CLAUDE_AUDIT_v1 complete.")

CELL: MTS_CLAUDE_AUDIT_v1

SUMMARY [Claude reference]
------------------------------------------------------------
N galaxies            : 191
median RMSE           : 21.491 km/s
p90 RMSE              : 119.397 km/s
fraction <= 20 km/s   : 0.471
fraction <= 30 km/s   : 0.602
median shape-only RMSE: 13.608 km/s
median amp penalty    : 6.873 km/s

BEST 20
    name  rmse_total  rmse_shape_only  amp_penalty   Rs           Mb      Vinf
 UGCA281    4.116402         3.225599     0.890803 0.25 6.000966e+07 30.167395
  DDO154    4.236914         4.236808     0.000106 1.48 3.636635e+08 47.332283
  F574-1    4.597830         4.507057     0.090773 2.35 6.330068e+09 96.679548
 UGCA444    5.498800         3.243144     2.255657 0.62 1.403857e+08 37.308965
  DDO168    5.659867         5.658704     0.001163 1.24 6.237089e+08 54.166154
  D564-8    5.780823         3.562613     2.218210 1.02 5.626394e+07 29.685205
 NGC3741    6.088526         5.920208     0.168317 0.93 3.035818e+08 45.243004
 UGCA442    

In [ ]:
# ============================================================
# CELL: MTS_CLAUDE_AMPLITUDE_AUDIT_v1
#
# Purpose:
#   Test whether the main failures in the Claude backbone
#   are dominated by amplitude closure rather than shape law.
#
# Strategy:
#   For each galaxy:
#   1) build the same shape prediction from gb -> m -> u -> Rs -> shape
#   2) compare three amplitudes:
#        A. current outer-point Mb proxy
#        B. direct best-fit amplitude to shape (diagnostic only)
#        C. improved baryonic mass closure using max enclosed mass
#           plus simple outer-tail completion
#
# Output:
#   - table of total RMSE under each amplitude closure
#   - galaxies where shape is good but amplitude is bad
#   - galaxies where even best-fit amplitude does not rescue shape
# ============================================================

import os
import re
import glob
import zipfile
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid

print("CELL: MTS_CLAUDE_AMPLITUDE_AUDIT_v1")

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0/3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

ROTMOD_DIR = "/content/mts_claude_audit_v1/ROTMOD"
OUTDIR = "/content/mts_claude_amplitude_audit_v1"
os.makedirs(OUTDIR, exist_ok=True)

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def build_shape(rot):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None
    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i = int(idx[0])
    uRs = float(u[i])
    if uRs <= 0:
        return None
    shape = 1.0 - np.exp(-u / uRs)
    return shape

def mass_proxies(rot):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_outer = float(Menc[-1])
    Mb_max   = float(np.max(Menc))

    # simple tail completion:
    # estimate whether final slope suggests still-rising outer baryonic support
    if len(r) >= 4:
        x = np.log(np.maximum(r[-4:], 1e-9))
        y = np.log(np.maximum(vb2[-4:], 1e-9))
        slope = np.polyfit(x, y, 1)[0]
    else:
        slope = -2.0

    # if vb2 is still too flat at edge, add cautious completion
    tail_boost = 1.0
    if slope > -1.0:
        tail_boost = 1.25
    elif slope > -1.5:
        tail_boost = 1.10

    Mb_tail = Mb_max * tail_boost
    return Mb_outer, Mb_max, Mb_tail

rows = []
files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

for fp in files:
    name = re.sub(r"_rotmod\.dat$|\.dat$", "", os.path.basename(fp), flags=re.I)
    try:
        rot = read_rotmod(fp)
        shape = build_shape(rot)
        if shape is None:
            continue

        vobs = rot["vobs"]

        Mb_outer, Mb_max, Mb_tail = mass_proxies(rot)

        Vinf_outer = (G_KPC_KMS2_MSun * Mb_outer * A_DAGGER)**0.25
        Vinf_max   = (G_KPC_KMS2_MSun * Mb_max   * A_DAGGER)**0.25
        Vinf_tail  = (G_KPC_KMS2_MSun * Mb_tail  * A_DAGGER)**0.25

        denom = np.dot(shape, shape)
        Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan

        v_outer = Vinf_outer * shape
        v_max   = Vinf_max   * shape
        v_tail  = Vinf_tail  * shape
        v_best  = Vinf_best  * shape

        rows.append({
            "name": name,
            "rmse_outer": safe_rmse(vobs, v_outer),
            "rmse_max":   safe_rmse(vobs, v_max),
            "rmse_tail":  safe_rmse(vobs, v_tail),
            "rmse_best":  safe_rmse(vobs, v_best),
            "amp_gap_outer_to_best": safe_rmse(vobs, v_outer) - safe_rmse(vobs, v_best),
            "amp_gap_tail_to_best":  safe_rmse(vobs, v_tail)  - safe_rmse(vobs, v_best),
            "Mb_outer": Mb_outer,
            "Mb_max": Mb_max,
            "Mb_tail": Mb_tail,
            "Vinf_outer": Vinf_outer,
            "Vinf_tail": Vinf_tail,
            "Vinf_best": Vinf_best,
        })
    except Exception as e:
        print("[skip]", name, e)

df = pd.DataFrame(rows)

print("\nSUMMARY")
for col in ["rmse_outer", "rmse_max", "rmse_tail", "rmse_best"]:
    print(f"{col:16s}: median={np.nanmedian(df[col]):.3f}  p90={np.nanpercentile(df[col],90):.3f}")

print("\nBIGGEST AMPLITUDE-CLOSURE CANDIDATES")
print(
    df.sort_values("amp_gap_outer_to_best", ascending=False)
      .head(25)[["name","rmse_outer","rmse_tail","rmse_best","amp_gap_outer_to_best","amp_gap_tail_to_best"]]
      .to_string(index=False)
)

df.to_csv(os.path.join(OUTDIR, "mts_claude_amplitude_audit_v1.csv"), index=False)
print("\nSaved:", os.path.join(OUTDIR, "mts_claude_amplitude_audit_v1.csv"))
print("\nCELL MTS_CLAUDE_AMPLITUDE_AUDIT_v1 complete.")

CELL: MTS_CLAUDE_AMPLITUDE_AUDIT_v1

SUMMARY
rmse_outer      : median=21.491  p90=121.563
rmse_max        : median=21.491  p90=119.397
rmse_tail       : median=20.462  p90=116.236
rmse_best       : median=13.608  p90=92.607

BIGGEST AMPLITUDE-CLOSURE CANDIDATES
       name  rmse_outer  rmse_tail  rmse_best  amp_gap_outer_to_best  amp_gap_tail_to_best
   UGC11914  125.417216 117.049891  52.383357              73.033858             64.666534
ESO563-G021   76.827545  70.989782  13.608469              63.219076             57.381313
    NGC5985  112.739965 104.224849  65.872539              46.867426             38.352310
   UGC06786  100.214593  96.976089  55.060203              45.154390             41.915885
   UGC02953  121.563090 116.236131  76.893247              44.669843             39.342884
    NGC2841  147.097021 140.163221 107.020533              40.076489             33.142688
   UGC06787  102.710784  99.463781  66.868787              35.841996             32.594993
    NGC297

In [ ]:
# ============================================================
# CELL: MTS_CLAUDE_BACKBONE_CLASSIFIER_v1
#
# Purpose:
#   Audit the Claude backbone as a *theory baseline* and classify
#   galaxies into:
#
#     1) BACKBONE_SUCCESS
#     2) AMPLITUDE_LIMITED
#     3) STRUCTURAL_SURVIVOR
#     4) MIXED_FAILURE
#
# It also measures the dynamic-range compression seen in the
# predicted-vs-observed scatter.
#
# Core questions:
#   - Is the backbone systematically underpredicting high-V systems?
#   - Which galaxies are mostly amplitude-closure failures?
#   - Which galaxies remain true structural survivors even after
#     best-fit amplitude relief?
#
# Inputs:
#   Uses the same ROTMOD files and same Claude backbone:
#     gb = Vbar^2 / r
#     m  = gb^gamma
#     u  = ∫ m dr
#     Rs from u(Rs) = frac * u_total
#     shape = 1 - exp(-u/u(Rs))
#     Vinf^4 = G Mb a_dagger
#
# Notes:
#   - no per-galaxy fitting in backbone prediction
#   - best-fit amplitude is used only as a diagnostic
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_CLAUDE_BACKBONE_CLASSIFIER_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_claude_backbone_classifier_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_claude_backbone_classifier_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

UPS_DISK = 0.5
UPS_BUL  = 0.7

GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# classification thresholds
SUCCESS_RMSE_MAX = 20.0
STRUCTURAL_RMSE_BEST_MIN = 60.0
AMP_RATIO_MAX = 0.65         # if rmse_best <= 0.65 * rmse_backbone → amplitude-limited
MIXED_RMSE_MIN = 20.0        # mixed if not success, not structural, not amplitude-limited

FOCUS = {
    "UGC02487", "NGC2841", "UGC11914", "UGC02953", "NGC5985",
    "ESO563-G021", "NGC5005", "UGC06786", "UGC06787", "UGC05253",
    "UGC11455", "UGC02885", "NGC7814", "NGC2903", "NGC3521"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files

    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)

    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def cumulative_baryonic_mass_proxy(rot):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb = float(np.nanmax(Menc))
    return Mb, Menc

def build_claude_backbone(rot, gamma=GAMMA, frac=FRAC, a_dagger=A_DAGGER):
    r = rot["r"]
    vobs = rot["vobs"]

    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)

    m = np.maximum(gb, 0.0) ** gamma
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None

    target = frac * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])

    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None

    shape = 1.0 - np.exp(-u / u_Rs)

    Mb, Menc = cumulative_baryonic_mass_proxy(rot)
    if not np.isfinite(Mb) or Mb <= 0:
        return None

    Vinf = float((G_KPC_KMS2_MSun * Mb * a_dagger) ** 0.25)
    vpred = Vinf * shape

    # best-fit amplitude diagnostic only
    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan
    vpred_best = Vinf_best * shape if np.isfinite(Vinf_best) else np.full_like(vobs, np.nan)

    # pointwise log-slope diagnostic (through-origin in log space)
    mp = np.isfinite(vobs) & np.isfinite(vpred) & (vobs > 0) & (vpred > 0)
    if mp.sum() >= 3:
        lx = np.log10(vobs[mp])   # observed on x
        ly = np.log10(vpred[mp])  # predicted on y
        slope_point = float(np.sum(lx * ly) / np.sum(lx * lx))
    else:
        slope_point = np.nan

    return {
        "r": r,
        "vobs": vobs,
        "vb2": vb2,
        "gb": gb,
        "m": m,
        "u": u,
        "u_tot": ut,
        "Rs": Rs,
        "u_Rs": u_Rs,
        "shape": shape,
        "Mb": Mb,
        "Vinf": Vinf,
        "vpred": vpred,
        "Vinf_best": Vinf_best,
        "vpred_best": vpred_best,
        "rmse_backbone": safe_rmse(vobs, vpred),
        "rmse_best": safe_rmse(vobs, vpred_best),
        "amp_gap": safe_rmse(vobs, vpred) - safe_rmse(vobs, vpred_best),
        "slope_point": slope_point,
    }

def classify_row(rmse_backbone, rmse_best):
    if rmse_backbone <= SUCCESS_RMSE_MAX:
        return "BACKBONE_SUCCESS"
    if np.isfinite(rmse_best) and rmse_best >= STRUCTURAL_RMSE_BEST_MIN:
        return "STRUCTURAL_SURVIVOR"
    if np.isfinite(rmse_best) and np.isfinite(rmse_backbone) and rmse_backbone > 0:
        if rmse_best <= AMP_RATIO_MAX * rmse_backbone:
            return "AMPLITUDE_LIMITED"
    if rmse_backbone >= MIXED_RMSE_MIN:
        return "MIXED_FAILURE"
    return "UNCLASSIFIED"

# ============================================================
# RUN CATALOG
# ============================================================

files = bootstrap_rotmods()

rows = []
curves = {}
all_vobs = []
all_vpred = []
all_vpred_best = []
all_names = []

for fp in files:
    name = galaxy_name_from_path(fp)
    try:
        rot = read_rotmod(fp)
        out = build_claude_backbone(rot)
        if out is None:
            continue

        cls = classify_row(out["rmse_backbone"], out["rmse_best"])

        rows.append({
            "name": name,
            "npts": len(out["r"]),
            "Rs": out["Rs"],
            "u_tot": out["u_tot"],
            "u_Rs": out["u_Rs"],
            "Mb": out["Mb"],
            "Vinf": out["Vinf"],
            "vmax_obs": float(np.max(out["vobs"])),
            "vmax_pred": float(np.max(out["vpred"])),
            "rmse_backbone": out["rmse_backbone"],
            "rmse_best": out["rmse_best"],
            "amp_gap": out["amp_gap"],
            "amp_ratio": (out["rmse_best"] / out["rmse_backbone"]) if out["rmse_backbone"] > 0 else np.nan,
            "slope_point": out["slope_point"],
            "class": cls,
        })

        curves[name] = out

        m = np.isfinite(out["vobs"]) & np.isfinite(out["vpred"]) & (out["vobs"] > 0) & (out["vpred"] > 0)
        all_vobs.extend(out["vobs"][m])
        all_vpred.extend(out["vpred"][m])
        all_names.extend([name] * int(np.sum(m)))

        mb = np.isfinite(out["vobs"]) & np.isfinite(out["vpred_best"]) & (out["vobs"] > 0) & (out["vpred_best"] > 0)
        all_vpred_best.extend(out["vpred_best"][mb])

    except Exception as e:
        print(f"[skip] {name}: {e}")

df = pd.DataFrame(rows)

# ============================================================
# GLOBAL COMPRESSION METRICS
# ============================================================

vobs_all = np.asarray(all_vobs, float)
vpred_all = np.asarray(all_vpred, float)

m = np.isfinite(vobs_all) & np.isfinite(vpred_all) & (vobs_all > 0) & (vpred_all > 0)
vobs_all = vobs_all[m]
vpred_all = vpred_all[m]

logx = np.log10(vobs_all)
logy = np.log10(vpred_all)

# standard fit y = a + b x
b, a = np.polyfit(logx, logy, 1)
# through-origin in log space
b0 = float(np.sum(logx * logy) / np.sum(logx * logx))

# split by observed velocity scale
bins = [
    ("LOW_VOBS", 0.0, 80.0),
    ("MID_VOBS", 80.0, 160.0),
    ("HIGH_VOBS", 160.0, np.inf),
]

split_rows = []
for label, lo, hi in bins:
    ms = (vobs_all >= lo) & (vobs_all < hi)
    if np.sum(ms) >= 5:
        lx = np.log10(vobs_all[ms])
        ly = np.log10(vpred_all[ms])
        bs, a_s = np.polyfit(lx, ly, 1)
        b0s = float(np.sum(lx * ly) / np.sum(lx * lx))
        bias_lin = float(np.mean(vpred_all[ms] - vobs_all[ms]))
        ratio_med = float(np.median(vpred_all[ms] / vobs_all[ms]))
        split_rows.append({
            "subset": label,
            "npts": int(np.sum(ms)),
            "slope": float(bs),
            "slope_origin": b0s,
            "intercept": float(a_s),
            "median_pred_over_obs": ratio_med,
            "mean_pred_minus_obs": bias_lin,
        })

df_split = pd.DataFrame(split_rows)

# ============================================================
# SUMMARIES
# ============================================================

print("\n" + "="*90)
print("GLOBAL BACKBONE SUMMARY")
print("="*90)
print(f"N galaxies                  : {len(df)}")
print(f"Median backbone RMSE        : {np.nanmedian(df['rmse_backbone']):.3f} km/s")
print(f"P90 backbone RMSE           : {np.nanpercentile(df['rmse_backbone'], 90):.3f} km/s")
print(f"Median best-amp RMSE        : {np.nanmedian(df['rmse_best']):.3f} km/s")
print(f"P90 best-amp RMSE           : {np.nanpercentile(df['rmse_best'], 90):.3f} km/s")
print(f"Median amp gap              : {np.nanmedian(df['amp_gap']):.3f} km/s")

print("\n" + "="*90)
print("DYNAMIC-RANGE COMPRESSION")
print("="*90)
print(f"All-point log fit slope     : {b:.3f}")
print(f"All-point log fit intercept : {a:.3f}")
print(f"Through-origin log slope    : {b0:.3f}")

if len(df_split):
    print("\nSplit by observed velocity:")
    print(df_split.to_string(index=False))

print("\n" + "="*90)
print("CLASS COUNTS")
print("="*90)
print(df["class"].value_counts(dropna=False).to_string())

print("\n" + "="*90)
print("TOP AMPLITUDE-LIMITED")
print("="*90)
tmp_amp = (
    df[df["class"].eq("AMPLITUDE_LIMITED")]
    .sort_values(["amp_gap", "rmse_backbone"], ascending=[False, False])
)
if len(tmp_amp):
    print(tmp_amp[["name", "rmse_backbone", "rmse_best", "amp_gap", "amp_ratio", "vmax_obs", "vmax_pred"]].head(25).to_string(index=False))
else:
    print("None")

print("\n" + "="*90)
print("TOP STRUCTURAL SURVIVORS")
print("="*90)
tmp_str = (
    df[df["class"].eq("STRUCTURAL_SURVIVOR")]
    .sort_values(["rmse_best", "rmse_backbone"], ascending=[False, False])
)
if len(tmp_str):
    print(tmp_str[["name", "rmse_backbone", "rmse_best", "amp_gap", "amp_ratio", "vmax_obs", "vmax_pred"]].head(25).to_string(index=False))
else:
    print("None")

print("\n" + "="*90)
print("TOP MIXED FAILURES")
print("="*90)
tmp_mix = (
    df[df["class"].eq("MIXED_FAILURE")]
    .sort_values(["rmse_backbone", "rmse_best"], ascending=[False, False])
)
if len(tmp_mix):
    print(tmp_mix[["name", "rmse_backbone", "rmse_best", "amp_gap", "amp_ratio", "vmax_obs", "vmax_pred"]].head(25).to_string(index=False))
else:
    print("None")

print("\n" + "="*90)
print("FOCUS OBJECTS")
print("="*90)
tmp_focus = df[df["name"].isin(FOCUS)].sort_values(["class", "rmse_backbone"], ascending=[True, False])
if len(tmp_focus):
    print(tmp_focus[["name", "class", "rmse_backbone", "rmse_best", "amp_gap", "amp_ratio", "Rs", "Vinf", "vmax_obs", "vmax_pred"]].to_string(index=False))
else:
    print("No focus objects found")

# ============================================================
# SAVE TABLES
# ============================================================

full_csv = os.path.join(OUTDIR, "mts_claude_backbone_classifier_full.csv")
amp_csv  = os.path.join(OUTDIR, "mts_claude_backbone_classifier_amplitude_limited.csv")
str_csv  = os.path.join(OUTDIR, "mts_claude_backbone_classifier_structural_survivors.csv")
mix_csv  = os.path.join(OUTDIR, "mts_claude_backbone_classifier_mixed_failures.csv")
split_csv = os.path.join(OUTDIR, "mts_claude_backbone_classifier_velocity_splits.csv")

df.sort_values(["class", "rmse_backbone", "rmse_best"], ascending=[True, False, False]).to_csv(full_csv, index=False)
tmp_amp.to_csv(amp_csv, index=False)
tmp_str.to_csv(str_csv, index=False)
tmp_mix.to_csv(mix_csv, index=False)
df_split.to_csv(split_csv, index=False)

# ============================================================
# PLOTS
# ============================================================

# scatter: predicted vs observed
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(vobs_all, vpred_all, s=4, alpha=0.35)
mx = max(np.max(vobs_all), np.max(vpred_all))
ax.plot([1, mx], [1, mx], "k--", lw=1.5, label="1:1")
xline = np.logspace(0, np.log10(mx), 200)
yline = 10**a * xline**b
ax.plot(xline, yline, "-", lw=2, label=f"log fit slope={b:.3f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Observed velocity [km/s]")
ax.set_ylabel("Predicted velocity [km/s]")
ax.set_title("Claude backbone: predicted vs observed velocity")
ax.legend()
plt.tight_layout()
scatter_png = os.path.join(OUTDIR, "mts_claude_backbone_pred_vs_obs.png")
plt.savefig(scatter_png, dpi=170)
plt.close(fig)

# histogram by class
fig, ax = plt.subplots(figsize=(7, 5))
for cls in ["BACKBONE_SUCCESS", "AMPLITUDE_LIMITED", "MIXED_FAILURE", "STRUCTURAL_SURVIVOR"]:
    sub = df[df["class"].eq(cls)]
    if len(sub):
        ax.hist(sub["rmse_backbone"], bins=20, alpha=0.55, label=cls)
ax.set_xlabel("Backbone RMSE [km/s]")
ax.set_ylabel("Count")
ax.set_title("RMSE distribution by class")
ax.legend()
plt.tight_layout()
hist_png = os.path.join(OUTDIR, "mts_claude_backbone_rmse_by_class.png")
plt.savefig(hist_png, dpi=170)
plt.close(fig)

# shape rescue plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["rmse_best"], df["rmse_backbone"], s=20, alpha=0.7)
mx = max(df["rmse_backbone"].max(), df["rmse_best"].max())
ax.plot([0, mx], [0, mx], "k--", lw=1)
ax.axvline(STRUCTURAL_RMSE_BEST_MIN, ls="--", lw=1)
ax.set_xlabel("Best-amplitude RMSE [km/s]")
ax.set_ylabel("Backbone RMSE [km/s]")
ax.set_title("Shape rescue diagnostic")
plt.tight_layout()
rescue_png = os.path.join(OUTDIR, "mts_claude_backbone_shape_rescue.png")
plt.savefig(rescue_png, dpi=170)
plt.close(fig)

# optional focus galaxy mini-panels
focus_found = [n for n in sorted(FOCUS) if n in curves]
if focus_found:
    n = len(focus_found)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.8*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, name in zip(axes, focus_found):
        out = curves[name]
        cls = df.loc[df["name"].eq(name), "class"].iloc[0]
        rb = df.loc[df["name"].eq(name), "rmse_backbone"].iloc[0]
        rk = df.loc[df["name"].eq(name), "rmse_best"].iloc[0]
        ax.plot(out["r"], out["vobs"], "o", ms=3, label="obs")
        ax.plot(out["r"], out["vpred"], "-", lw=2, label="backbone")
        ax.plot(out["r"], out["vpred_best"], "--", lw=2, label="best amp")
        ax.set_title(f"{name}\n{cls} | rmse={rb:.1f} | best={rk:.1f}")
        ax.set_xlabel("r [kpc]")
        ax.set_ylabel("V [km/s]")
        ax.legend(fontsize=8)

    for ax in axes[len(focus_found):]:
        ax.axis("off")

    plt.tight_layout()
    focus_png = os.path.join(OUTDIR, "mts_claude_backbone_focus_panels.png")
    plt.savefig(focus_png, dpi=170)
    plt.close(fig)
else:
    focus_png = None

print("\nSaved:")
print(" -", full_csv)
print(" -", amp_csv)
print(" -", str_csv)
print(" -", mix_csv)
print(" -", split_csv)
print(" -", scatter_png)
print(" -", hist_png)
print(" -", rescue_png)
if focus_png:
    print(" -", focus_png)

print("\nCELL MTS_CLAUDE_BACKBONE_CLASSIFIER_v1 complete.")

CELL: MTS_CLAUDE_BACKBONE_CLASSIFIER_v1

GLOBAL BACKBONE SUMMARY
N galaxies                  : 191
Median backbone RMSE        : 21.491 km/s
P90 backbone RMSE           : 119.397 km/s
Median best-amp RMSE        : 13.608 km/s
P90 best-amp RMSE           : 92.607 km/s
Median amp gap              : 6.873 km/s

DYNAMIC-RANGE COMPRESSION
All-point log fit slope     : 0.725
All-point log fit intercept : 0.491
Through-origin log slope    : 0.957

Split by observed velocity:
   subset  npts    slope  slope_origin  intercept  median_pred_over_obs  mean_pred_minus_obs
 LOW_VOBS   992 0.743096      1.011683   0.453209              1.116091             3.568212
 MID_VOBS   837 0.955579      0.974777   0.039190              0.910166           -10.248711
HIGH_VOBS  1402 0.426425      0.930166   1.189896              0.760802           -65.834929

CLASS COUNTS
class
BACKBONE_SUCCESS       90
STRUCTURAL_SURVIVOR    38
AMPLITUDE_LIMITED      32
MIXED_FAILURE          31

TOP AMPLITUDE-LIMITED
       n

In [ ]:
# ============================================================
# CELL: MTS_AMPLITUDE_PROGRAM_v1
#
# Purpose:
#   Amplitude-only research program for the Claude backbone.
#
#   Restricts attention to galaxies classified as AMPLITUDE_LIMITED
#   under the backbone classifier and tests whether improved Vinf
#   closure can materially reduce error WITHOUT changing the shape law.
#
# Core idea:
#   Keep fixed:
#     gb -> m=gb^gamma -> u -> Rs -> shape
#
#   Only vary amplitude closure:
#     Vinf^4 = G * Mb_eff * a_dagger_eff
#
#   using a small family of global / semi-global Mb_eff laws.
#
# What it tests:
#   A) baseline_max
#      Mb_eff = max_r [ r Vbar^2 / G ]
#
#   B) tail_boost
#      cautious outer completion based on final vb^2 slope
#
#   C) conc_boost
#      Mb_eff = Mb_max * (r_last / Rs)^q
#      for a small global q-grid
#
#   D) hybrid_tail_conc
#      combine tail boost and concentration boost
#
#   E) a_dagger_rescale
#      test a small global a_dagger grid
#
# Outputs:
#   - summary table per amplitude law
#   - best global amplitude law on amplitude-limited subset
#   - per-galaxy comparison for focus targets
#
# Notes:
#   - no change to shape law
#   - no per-galaxy tuning
#   - only global or weakly structural amplitude closures
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_AMPLITUDE_PROGRAM_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_amplitude_program_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_amplitude_program_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

UPS_DISK = 0.5
UPS_BUL  = 0.7

GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER_REF = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# Pull amplitude-limited set from previous classifier if present
CLASSIFIER_CSV = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_amplitude_limited.csv"

# global grids
Q_GRID = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]
A_DAGGER_GRID = [2600.0, 2900.0, 3209.0, 3500.0, 3800.0, 4200.0]

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC02953", "NGC5985"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def build_shape(rot):
    r = rot["r"]
    vobs = rot["vobs"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None
    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])
    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None
    shape = 1.0 - np.exp(-u / u_Rs)
    return {
        "r": r,
        "vobs": vobs,
        "vb2": vb2,
        "gb": gb,
        "m": m,
        "u": u,
        "Rs": Rs,
        "u_Rs": u_Rs,
        "shape": shape,
    }

def mass_proxies(rot, Rs):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_outer = float(Menc[-1])
    Mb_max   = float(np.max(Menc))
    r_last   = float(r[-1])

    # final slope of vb^2 in log-log on last few points
    if len(r) >= 4:
        x = np.log(np.maximum(r[-4:], 1e-9))
        y = np.log(np.maximum(vb2[-4:], 1e-9))
        slope_last = float(np.polyfit(x, y, 1)[0])
    else:
        slope_last = -2.0

    # cautious tail-completion
    tail_boost = 1.0
    if slope_last > -1.0:
        tail_boost = 1.25
    elif slope_last > -1.5:
        tail_boost = 1.10

    Mb_tail = Mb_max * tail_boost

    return {
        "Mb_outer": Mb_outer,
        "Mb_max": Mb_max,
        "Mb_tail": Mb_tail,
        "r_last": r_last,
        "slope_last": slope_last,
        "Rs": Rs,
    }

def vinf_from_mb(Mb_eff, a_dagger):
    return float((G_KPC_KMS2_MSun * Mb_eff * a_dagger) ** 0.25)

def eval_amp_law(shape_rec, mp_rec, law_name, q=0.0, a_dagger=A_DAGGER_REF):
    shape = shape_rec["shape"]
    vobs = shape_rec["vobs"]

    Mb_outer = mp_rec["Mb_outer"]
    Mb_max   = mp_rec["Mb_max"]
    Mb_tail  = mp_rec["Mb_tail"]
    r_last   = mp_rec["r_last"]
    Rs       = mp_rec["Rs"]

    conc_factor = float((r_last / max(Rs, 1e-30)) ** q)

    if law_name == "baseline_max":
        Mb_eff = Mb_max
    elif law_name == "outer_only":
        Mb_eff = Mb_outer
    elif law_name == "tail_boost":
        Mb_eff = Mb_tail
    elif law_name == "conc_boost":
        Mb_eff = Mb_max * conc_factor
    elif law_name == "hybrid_tail_conc":
        Mb_eff = Mb_tail * conc_factor
    else:
        raise ValueError(f"Unknown law: {law_name}")

    Vinf = vinf_from_mb(Mb_eff, a_dagger)
    vpred = Vinf * shape
    rmse = safe_rmse(vobs, vpred)

    return {
        "Mb_eff": float(Mb_eff),
        "Vinf": Vinf,
        "rmse": rmse,
        "q": q,
        "a_dagger": a_dagger,
        "law": law_name,
        "r_last_over_Rs": float(r_last / max(Rs, 1e-30)),
    }

# ============================================================
# LOAD TARGET SET
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if os.path.exists(CLASSIFIER_CSV):
    df_amp_input = pd.read_csv(CLASSIFIER_CSV)
    amp_names = list(df_amp_input["name"].unique())
    print(f"Loaded amplitude-limited set from classifier: {len(amp_names)} galaxies")
else:
    raise FileNotFoundError(f"Missing classifier amplitude-limited CSV: {CLASSIFIER_CSV}")

# build shape records once
shape_cache = {}
mp_cache = {}

for name in amp_names:
    if name not in name_to_path:
        continue
    rot = read_rotmod(name_to_path[name])
    sr = build_shape(rot)
    if sr is None:
        continue
    mp = mass_proxies(rot, sr["Rs"])
    shape_cache[name] = sr
    mp_cache[name] = mp

amp_names = sorted(shape_cache.keys())
print(f"Usable amplitude-limited galaxies: {len(amp_names)}")

# ============================================================
# GLOBAL LAW SCAN
# ============================================================

summary_rows = []
detail_rows = []

# reference best-fit amplitude for comparison only
for name in amp_names:
    sr = shape_cache[name]
    shape = sr["shape"]
    vobs = sr["vobs"]
    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan
    vpred_best = Vinf_best * shape if np.isfinite(Vinf_best) else np.full_like(vobs, np.nan)
    detail_rows.append({
        "name": name,
        "law": "best_fit_amp_diag",
        "q": np.nan,
        "a_dagger": np.nan,
        "rmse": safe_rmse(vobs, vpred_best),
        "Vinf": Vinf_best,
        "Mb_eff": np.nan,
        "r_last_over_Rs": mp_cache[name]["r_last"] / max(mp_cache[name]["Rs"], 1e-30),
    })

law_specs = []

# plain laws at reference a_dagger
for law in ["outer_only", "baseline_max", "tail_boost"]:
    law_specs.append((law, 0.0, A_DAGGER_REF))

# concentration laws
for q in Q_GRID:
    law_specs.append(("conc_boost", q, A_DAGGER_REF))
    law_specs.append(("hybrid_tail_conc", q, A_DAGGER_REF))

# a_dagger rescale on the most plausible families
for a_dag in A_DAGGER_GRID:
    law_specs.append(("baseline_max", 0.0, a_dag))
    law_specs.append(("tail_boost", 0.0, a_dag))
    law_specs.append(("conc_boost", 0.10, a_dag))
    law_specs.append(("hybrid_tail_conc", 0.10, a_dag))

# de-duplicate
law_specs = list(dict.fromkeys(law_specs))

for law, q, a_dag in law_specs:
    rmses = []
    vins = []
    for name in amp_names:
        sr = shape_cache[name]
        mp = mp_cache[name]
        out = eval_amp_law(sr, mp, law_name=law, q=q, a_dagger=a_dag)
        rmses.append(out["rmse"])
        vins.append(out["Vinf"])
        detail_rows.append({
            "name": name,
            **out,
        })

    summary_rows.append({
        "law": law,
        "q": q,
        "a_dagger": a_dag,
        "med_rmse": float(np.nanmedian(rmses)),
        "p90_rmse": float(np.nanpercentile(rmses, 90)),
        "mean_rmse": float(np.nanmean(rmses)),
    })
    print(f"[done] {law:16s} q={q:>4.2f} a_dagger={a_dag:>7.1f}")

df_summary = pd.DataFrame(summary_rows).sort_values(["med_rmse", "p90_rmse", "mean_rmse"]).reset_index(drop=True)
df_detail = pd.DataFrame(detail_rows)

# ============================================================
# BEST LAW + PER-GALAXY COMPARISON
# ============================================================

best_row = df_summary.iloc[0].to_dict()
best_law = best_row["law"]
best_q = float(best_row["q"])
best_a_dagger = float(best_row["a_dagger"])

print("\n" + "="*90)
print("AMPLITUDE PROGRAM SUMMARY")
print("="*90)
print(f"N amplitude-limited galaxies : {len(amp_names)}")
print("\nTop candidate laws:")
print(df_summary.head(20).to_string(index=False))

print("\nBest global amplitude law:")
for k, v in best_row.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

# compare baseline_max vs best law vs best-fit amplitude
comp_rows = []
for name in amp_names:
    s = shape_cache[name]
    m = mp_cache[name]

    base = eval_amp_law(s, m, law_name="baseline_max", q=0.0, a_dagger=A_DAGGER_REF)
    best = eval_amp_law(s, m, law_name=best_law, q=best_q, a_dagger=best_a_dagger)

    sub = df_detail[(df_detail["name"].eq(name)) & (df_detail["law"].eq("best_fit_amp_diag"))]
    rmse_bestfit = float(sub["rmse"].iloc[0]) if len(sub) else np.nan
    Vinf_bestfit = float(sub["Vinf"].iloc[0]) if len(sub) else np.nan

    comp_rows.append({
        "name": name,
        "rmse_baseline": base["rmse"],
        "rmse_best_global_law": best["rmse"],
        "rmse_best_fit_amp": rmse_bestfit,
        "gain_vs_baseline": base["rmse"] - best["rmse"],
        "residual_to_bestfit": best["rmse"] - rmse_bestfit,
        "Vinf_baseline": base["Vinf"],
        "Vinf_best_global_law": best["Vinf"],
        "Vinf_best_fit_amp": Vinf_bestfit,
        "r_last_over_Rs": best["r_last_over_Rs"],
    })

df_comp = pd.DataFrame(comp_rows).sort_values(["gain_vs_baseline", "rmse_best_global_law"], ascending=[False, True])

print("\n" + "="*90)
print("PER-GALAXY COMPARISON (baseline vs best global law)")
print("="*90)
print(df_comp.to_string(index=False))

print("\n" + "="*90)
print("FOCUS OBJECTS")
print("="*90)
focus_df = df_comp[df_comp["name"].isin(FOCUS)].copy()
if len(focus_df):
    print(focus_df.to_string(index=False))
else:
    print("No focus objects in amplitude-limited set.")

# ============================================================
# SAVE
# ============================================================

summary_csv = os.path.join(OUTDIR, "mts_amplitude_program_summary.csv")
detail_csv  = os.path.join(OUTDIR, "mts_amplitude_program_detail.csv")
comp_csv    = os.path.join(OUTDIR, "mts_amplitude_program_comparison.csv")

df_summary.to_csv(summary_csv, index=False)
df_detail.to_csv(detail_csv, index=False)
df_comp.to_csv(comp_csv, index=False)

print("\nSaved:")
print(" -", summary_csv)
print(" -", detail_csv)
print(" -", comp_csv)

print("\nCELL MTS_AMPLITUDE_PROGRAM_v1 complete.")

CELL: MTS_AMPLITUDE_PROGRAM_v1
Loaded amplitude-limited set from classifier: 32 galaxies
Usable amplitude-limited galaxies: 32
[done] outer_only       q=0.00 a_dagger= 3209.0
[done] baseline_max     q=0.00 a_dagger= 3209.0
[done] tail_boost       q=0.00 a_dagger= 3209.0
[done] conc_boost       q=0.00 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.00 a_dagger= 3209.0
[done] conc_boost       q=0.05 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.05 a_dagger= 3209.0
[done] conc_boost       q=0.10 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.10 a_dagger= 3209.0
[done] conc_boost       q=0.15 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.15 a_dagger= 3209.0
[done] conc_boost       q=0.20 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.20 a_dagger= 3209.0
[done] conc_boost       q=0.25 a_dagger= 3209.0
[done] hybrid_tail_conc q=0.25 a_dagger= 3209.0
[done] baseline_max     q=0.00 a_dagger= 2600.0
[done] tail_boost       q=0.00 a_dagger= 2600.0
[done] conc_boost       q=0.10 a_dagger= 2600.0
[done] hy

In [ ]:
# ============================================================
# CELL: MTS_AMPLITUDE_RESIDUAL_LAW_DISCOVERY_v1
#
# Purpose:
#   Discover what controls the remaining amplitude gap in the
#   AMPLITUDE_LIMITED backbone class.
#
# Strategy:
#   For each amplitude-limited galaxy, compute the required
#   amplitude lift:
#
#     dlogV = log10(Vinf_best_fit / Vinf_baseline)
#
#   and test whether it correlates with simple structural /
#   observable quantities such as:
#
#     - log10(r_last / Rs)
#     - outer slope of vb^2
#     - log10(Mb_max)
#     - log10(Vmax_obs)
#     - log10(Rs)
#     - log10(r_last)
#     - log10(r_peak / r_bar)   [if computable from baryons]
#     - log10(rt proxy / r_peak) [optional simple transport proxy]
#
# Goal:
#   Find a compact, data-earned amplitude residual law candidate,
#   rather than more blind global grids.
#
# Outputs:
#   - ranked single-feature correlations
#   - ranked two-feature linear models in log space
#   - best candidate law table
#   - per-galaxy predicted vs required amplitude lift
#
# Notes:
#   - shape law is held fixed
#   - best-fit amplitude is used only diagnostically
#   - this is not a final theory law, just discovery scaffolding
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_AMPLITUDE_RESIDUAL_LAW_DISCOVERY_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_amplitude_residual_law_discovery_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_amplitude_residual_law_discovery_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

CLASSIFIER_CSV = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_amplitude_limited.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC03205"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def build_shape(rot):
    r = rot["r"]
    vobs = rot["vobs"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None
    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])
    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None
    shape = 1.0 - np.exp(-u / u_Rs)
    return {
        "r": r,
        "vobs": vobs,
        "vb2": vb2,
        "gb": gb,
        "m": m,
        "u": u,
        "u_tot": ut,
        "Rs": Rs,
        "u_Rs": u_Rs,
        "shape": shape,
    }

def mass_proxy_max(rot):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    Menc = r * vb2 / G_KPC_KMS2_MSun
    return float(np.max(Menc))

def vinf_from_mb(Mb_eff, a_dagger=A_DAGGER):
    return float((G_KPC_KMS2_MSun * Mb_eff * a_dagger) ** 0.25)

def corr_1d(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0, 1])

def linfit_1d(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return None
    b, a = np.polyfit(x[m], y[m], 1)
    yhat = a + b * x[m]
    rmse = float(np.sqrt(np.mean((y[m] - yhat)**2)))
    ss_res = float(np.sum((y[m] - yhat)**2))
    ss_tot = float(np.sum((y[m] - np.mean(y[m]))**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"intercept": float(a), "coef": float(b), "rmse": rmse, "r2": r2}

def linfit_2d(X1, X2, y):
    X1 = np.asarray(X1, float)
    X2 = np.asarray(X2, float)
    y = np.asarray(y, float)
    m = np.isfinite(X1) & np.isfinite(X2) & np.isfinite(y)
    if np.sum(m) < 4:
        return None
    A = np.column_stack([np.ones(np.sum(m)), X1[m], X2[m]])
    coef, *_ = np.linalg.lstsq(A, y[m], rcond=None)
    yhat = A @ coef
    rmse = float(np.sqrt(np.mean((y[m] - yhat)**2)))
    ss_res = float(np.sum((y[m] - yhat)**2))
    ss_tot = float(np.sum((y[m] - np.mean(y[m]))**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {
        "intercept": float(coef[0]),
        "coef1": float(coef[1]),
        "coef2": float(coef[2]),
        "rmse": rmse,
        "r2": r2,
    }

# ============================================================
# LOAD TARGET SET
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if not os.path.exists(CLASSIFIER_CSV):
    raise FileNotFoundError(f"Missing classifier amplitude-limited CSV: {CLASSIFIER_CSV}")

df_amp = pd.read_csv(CLASSIFIER_CSV)
amp_names = [n for n in df_amp["name"].unique() if n in name_to_path]
print(f"Loaded amplitude-limited galaxies: {len(amp_names)}")

# ============================================================
# BUILD FEATURE TABLE
# ============================================================

rows = []

for name in amp_names:
    rot = read_rotmod(name_to_path[name])
    sr = build_shape(rot)
    if sr is None:
        continue

    r = sr["r"]
    vobs = sr["vobs"]
    vb2 = sr["vb2"]
    shape = sr["shape"]
    Rs = float(sr["Rs"])
    r_last = float(r[-1])

    Mb_max = mass_proxy_max(rot)
    Vinf_baseline = vinf_from_mb(Mb_max, A_DAGGER)

    denom = float(np.dot(shape, shape))
    if denom <= 0:
        continue
    Vinf_best = float(np.dot(vobs, shape) / denom)

    dlogV = float(np.log10(max(Vinf_best, 1e-30) / max(Vinf_baseline, 1e-30)))
    rmse_baseline = safe_rmse(vobs, Vinf_baseline * shape)
    rmse_best = safe_rmse(vobs, Vinf_best * shape)

    # outer slope using last 4 points
    if len(r) >= 4:
        slope_last = float(np.polyfit(
            np.log(np.maximum(r[-4:], 1e-9)),
            np.log(np.maximum(vb2[-4:], 1e-9)),
            1
        )[0])
    else:
        slope_last = np.nan

    # r_bar and r_peak from baryonic curve
    vb = np.sqrt(np.maximum(vb2, 0.0))
    vmax_bar = float(np.max(vb))
    i_peak = int(np.argmax(vb))
    r_peak = float(r[i_peak]) if np.isfinite(vmax_bar) and vmax_bar > 0 else np.nan
    idx_bar = np.where(vb >= 0.5 * vmax_bar)[0] if vmax_bar > 0 else []
    r_bar = float(r[idx_bar[0]]) if len(idx_bar) else np.nan

    # simple transport proxy rt: where u reaches 60% of total
    u = sr["u"]
    idx_rt = np.where(u >= 0.60 * float(np.max(u)))[0]
    rt = float(r[idx_rt[0]]) if len(idx_rt) else np.nan

    rows.append({
        "name": name,
        "rmse_baseline": rmse_baseline,
        "rmse_best": rmse_best,
        "Vinf_baseline": Vinf_baseline,
        "Vinf_best": Vinf_best,
        "dlogV_required": dlogV,
        "log_r_last_over_Rs": float(safe_log10(r_last / max(Rs, 1e-30))),
        "outer_slope_vb2": slope_last,
        "log_Mb_max": float(safe_log10(Mb_max)),
        "log_Vmax_obs": float(safe_log10(np.max(vobs))),
        "log_Rs": float(safe_log10(Rs)),
        "log_r_last": float(safe_log10(r_last)),
        "log_rpeak_over_rbar": float(safe_log10(r_peak / max(r_bar, 1e-30))) if np.isfinite(r_peak) and np.isfinite(r_bar) and r_bar > 0 else np.nan,
        "log_rt_over_rpeak": float(safe_log10(rt / max(r_peak, 1e-30))) if np.isfinite(rt) and np.isfinite(r_peak) and r_peak > 0 else np.nan,
        "Rs": Rs,
        "r_last": r_last,
        "r_peak": r_peak,
        "r_bar": r_bar,
        "rt": rt,
    })

df = pd.DataFrame(rows).sort_values("dlogV_required", ascending=False).reset_index(drop=True)

print("\n" + "="*90)
print("AMPLITUDE RESIDUAL TARGET TABLE")
print("="*90)
print(df[[
    "name", "rmse_baseline", "rmse_best",
    "Vinf_baseline", "Vinf_best", "dlogV_required",
    "log_r_last_over_Rs", "outer_slope_vb2", "log_Mb_max",
    "log_Vmax_obs", "log_rpeak_over_rbar", "log_rt_over_rpeak"
]].to_string(index=False))

# ============================================================
# SINGLE-FEATURE DISCOVERY
# ============================================================

candidate_features = [
    "log_r_last_over_Rs",
    "outer_slope_vb2",
    "log_Mb_max",
    "log_Vmax_obs",
    "log_Rs",
    "log_r_last",
    "log_rpeak_over_rbar",
    "log_rt_over_rpeak",
]

single_rows = []
for feat in candidate_features:
    fit = linfit_1d(df[feat], df["dlogV_required"])
    if fit is None:
        continue
    single_rows.append({
        "feature": feat,
        "corr": corr_1d(df[feat], df["dlogV_required"]),
        **fit
    })

df_single = pd.DataFrame(single_rows).sort_values(["rmse", "r2"], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*90)
print("RANKED SINGLE-FEATURE LAWS")
print("="*90)
print(df_single.to_string(index=False))

# ============================================================
# TWO-FEATURE DISCOVERY
# ============================================================

pair_rows = []
for f1, f2 in itertools.combinations(candidate_features, 2):
    fit = linfit_2d(df[f1], df[f2], df["dlogV_required"])
    if fit is None:
        continue
    pair_rows.append({
        "feature_1": f1,
        "feature_2": f2,
        **fit
    })

df_pair = pd.DataFrame(pair_rows).sort_values(["rmse", "r2"], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*90)
print("RANKED TWO-FEATURE LAWS")
print("="*90)
print(df_pair.head(25).to_string(index=False))

# ============================================================
# BUILD BEST CANDIDATE LAW
# ============================================================

if len(df_pair):
    best_pair = df_pair.iloc[0].to_dict()
    f1 = best_pair["feature_1"]
    f2 = best_pair["feature_2"]
    c0 = best_pair["intercept"]
    c1 = best_pair["coef1"]
    c2 = best_pair["coef2"]

    df["dlogV_pred_pair"] = c0 + c1 * df[f1] + c2 * df[f2]
    df["Vinf_pred_pair"] = df["Vinf_baseline"] * (10 ** df["dlogV_pred_pair"])
    df["pred_ratio_pair"] = df["Vinf_pred_pair"] / df["Vinf_best"]
    df["resid_dlogV_pair"] = df["dlogV_required"] - df["dlogV_pred_pair"]

    print("\n" + "="*90)
    print("BEST CANDIDATE AMPLITUDE RESIDUAL LAW")
    print("="*90)
    print(f"dlogV_required = {c0:.6f} + ({c1:.6f})*{f1} + ({c2:.6f})*{f2}")
    print(f"rmse={best_pair['rmse']:.6f}, r2={best_pair['r2']:.6f}")

    print("\nPer-galaxy predicted vs required lift:")
    print(df[[
        "name", "dlogV_required", "dlogV_pred_pair",
        "Vinf_baseline", "Vinf_best", "Vinf_pred_pair",
        "pred_ratio_pair", "resid_dlogV_pair"
    ]].to_string(index=False))
else:
    best_pair = None

# ============================================================
# FOCUS OBJECTS
# ============================================================

print("\n" + "="*90)
print("FOCUS OBJECTS")
print("="*90)
focus_df = df[df["name"].isin(FOCUS)].copy()
if len(focus_df):
    cols = [
        "name", "rmse_baseline", "rmse_best",
        "Vinf_baseline", "Vinf_best", "dlogV_required",
        "log_r_last_over_Rs", "outer_slope_vb2",
        "log_Mb_max", "log_Vmax_obs",
        "log_rpeak_over_rbar", "log_rt_over_rpeak"
    ]
    if "dlogV_pred_pair" in focus_df.columns:
        cols += ["dlogV_pred_pair", "resid_dlogV_pair"]
    print(focus_df[cols].to_string(index=False))
else:
    print("No focus objects present.")

# ============================================================
# PLOTS
# ============================================================

if len(df_single):
    best_feat = df_single.iloc[0]["feature"]
    fit = linfit_1d(df[best_feat], df["dlogV_required"])
    if fit is not None:
        x = np.asarray(df[best_feat], float)
        y = np.asarray(df["dlogV_required"], float)
        m = np.isfinite(x) & np.isfinite(y)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(x[m], y[m], s=35)
        xx = np.linspace(np.nanmin(x[m]), np.nanmax(x[m]), 200)
        yy = fit["intercept"] + fit["coef"] * xx
        ax.plot(xx, yy, lw=2)
        ax.set_xlabel(best_feat)
        ax.set_ylabel("dlogV_required")
        ax.set_title(f"Best single-feature amplitude residual law\nr2={fit['r2']:.3f}, rmse={fit['rmse']:.3f}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, "best_single_feature_law.png"), dpi=170)
        plt.close(fig)

if best_pair is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df["dlogV_required"], df["dlogV_pred_pair"], s=40)
    lo = min(df["dlogV_required"].min(), df["dlogV_pred_pair"].min())
    hi = max(df["dlogV_required"].max(), df["dlogV_pred_pair"].max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("Required dlogV")
    ax.set_ylabel("Predicted dlogV")
    ax.set_title(f"Best two-feature law\nr2={best_pair['r2']:.3f}, rmse={best_pair['rmse']:.3f}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "best_pair_required_vs_predicted_dlogV.png"), dpi=170)
    plt.close(fig)

# ============================================================
# SAVE
# ============================================================

target_csv = os.path.join(OUTDIR, "amplitude_residual_target_table.csv")
single_csv = os.path.join(OUTDIR, "amplitude_residual_single_feature_laws.csv")
pair_csv = os.path.join(OUTDIR, "amplitude_residual_two_feature_laws.csv")
full_csv = os.path.join(OUTDIR, "amplitude_residual_full_with_predictions.csv")

df.to_csv(target_csv, index=False)
df_single.to_csv(single_csv, index=False)
df_pair.to_csv(pair_csv, index=False)
df.to_csv(full_csv, index=False)

print("\nSaved:")
print(" -", target_csv)
print(" -", single_csv)
print(" -", pair_csv)
print(" -", full_csv)

print("\nCELL MTS_AMPLITUDE_RESIDUAL_LAW_DISCOVERY_v1 complete.")

CELL: MTS_AMPLITUDE_RESIDUAL_LAW_DISCOVERY_v1
Loaded amplitude-limited galaxies: 32

AMPLITUDE RESIDUAL TARGET TABLE
       name  rmse_baseline  rmse_best  Vinf_baseline  Vinf_best  dlogV_required  log_r_last_over_Rs  outer_slope_vb2  log_Mb_max  log_Vmax_obs  log_rpeak_over_rbar  log_rt_over_rpeak
    NGC6789      20.558024   8.305461      30.285367  56.245868        0.268858            0.374137         1.218799    7.785001      1.781037             0.374137          -0.143688
   UGC11914     125.417216  52.383357     184.726885 314.869414        0.231600            0.943335        -0.724791   10.926190      2.484300             1.028513           0.215410
   UGC06786      99.431054  55.060203     162.328749 271.634089        0.223589            0.961574        -1.059423   10.701652      2.359835             0.608793           1.361146
    NGC0024      31.706337  12.419531      82.815447 124.284277        0.176305            0.912045        -1.114163    9.532515      2.041393         

In [ ]:
# ============================================================
# CELL: MTS_AMPLITUDE_BARYONIC_PROXY_REPLACEMENT_v1
#
# Purpose:
#   Replace the discovery-law dependence on observed velocity
#   with purely baryonic / structural proxies.
#
# Goal:
#   We already found that the required amplitude lift
#
#     dlogV_required = log10(Vinf_best / Vinf_baseline)
#
#   is very well predicted by:
#
#     dlogV ~ -0.247*log_Mb_max + 1.022*log_Vmax_obs
#
#   But log_Vmax_obs is downstream of the thing we are trying to
#   predict, so it cannot stay in the final law.
#
#   This cell searches for baryonic substitutes for log_Vmax_obs.
#
# Strategy:
#   Build a broad table of baryonic-only structural proxies, then
#   rank:
#     1) single-feature laws for dlogV_required
#     2) two-feature laws
#     3) "Mb + proxy" laws, since the discovery fit suggests the
#        amplitude closure is fundamentally Mb plus one missing
#        structural/depth variable
#
# Outputs:
#   - ranked baryonic-only single-feature laws
#   - ranked baryonic-only two-feature laws
#   - ranked "log_Mb_max + proxy" laws
#   - best candidate baryonic replacement law
#   - per-galaxy predicted vs required amplitude lift
#
# Notes:
#   - shape law remains fixed
#   - uses best-fit amplitude only diagnostically
#   - excludes any feature derived from observed velocity
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_AMPLITUDE_BARYONIC_PROXY_REPLACEMENT_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_amplitude_baryonic_proxy_replacement_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_amplitude_baryonic_proxy_replacement_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

CLASSIFIER_CSV = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_amplitude_limited.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC03205"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def build_shape(rot):
    r = rot["r"]
    vobs = rot["vobs"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None
    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])
    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None
    shape = 1.0 - np.exp(-u / u_Rs)
    return {
        "r": r,
        "vobs": vobs,
        "vb2": vb2,
        "gb": gb,
        "m": m,
        "u": u,
        "u_tot": ut,
        "Rs": Rs,
        "u_Rs": u_Rs,
        "shape": shape,
    }

def mass_proxy_max(rot):
    r = rot["r"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    Menc = r * vb2 / G_KPC_KMS2_MSun
    return float(np.max(Menc))

def vinf_from_mb(Mb_eff, a_dagger=A_DAGGER):
    return float((G_KPC_KMS2_MSun * Mb_eff * a_dagger) ** 0.25)

def corr_1d(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0, 1])

def linfit_1d(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return None
    b, a = np.polyfit(x[m], y[m], 1)
    yhat = a + b * x[m]
    rmse = float(np.sqrt(np.mean((y[m] - yhat)**2)))
    ss_res = float(np.sum((y[m] - yhat)**2))
    ss_tot = float(np.sum((y[m] - np.mean(y[m]))**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"intercept": float(a), "coef": float(b), "rmse": rmse, "r2": r2}

def linfit_2d(X1, X2, y):
    X1 = np.asarray(X1, float)
    X2 = np.asarray(X2, float)
    y = np.asarray(y, float)
    m = np.isfinite(X1) & np.isfinite(X2) & np.isfinite(y)
    if np.sum(m) < 4:
        return None
    A = np.column_stack([np.ones(np.sum(m)), X1[m], X2[m]])
    coef, *_ = np.linalg.lstsq(A, y[m], rcond=None)
    yhat = A @ coef
    rmse = float(np.sqrt(np.mean((y[m] - yhat)**2)))
    ss_res = float(np.sum((y[m] - yhat)**2))
    ss_tot = float(np.sum((y[m] - np.mean(y[m]))**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {
        "intercept": float(coef[0]),
        "coef1": float(coef[1]),
        "coef2": float(coef[2]),
        "rmse": rmse,
        "r2": r2,
    }

# ============================================================
# LOAD TARGET SET
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if not os.path.exists(CLASSIFIER_CSV):
    raise FileNotFoundError(f"Missing classifier amplitude-limited CSV: {CLASSIFIER_CSV}")

df_amp = pd.read_csv(CLASSIFIER_CSV)
amp_names = [n for n in df_amp["name"].unique() if n in name_to_path]
print(f"Loaded amplitude-limited galaxies: {len(amp_names)}")

# ============================================================
# BUILD BARYONIC FEATURE TABLE
# ============================================================

rows = []

for name in amp_names:
    rot = read_rotmod(name_to_path[name])
    sr = build_shape(rot)
    if sr is None:
        continue

    r = sr["r"]
    vobs = sr["vobs"]
    vb2 = sr["vb2"]
    gb = sr["gb"]
    m = sr["m"]
    u = sr["u"]
    shape = sr["shape"]
    Rs = float(sr["Rs"])

    vgas = rot["vgas"]
    vdisk = rot["vdisk"]
    vbul = rot["vbul"]

    Mb_max = mass_proxy_max(rot)
    Vinf_baseline = vinf_from_mb(Mb_max, A_DAGGER)

    denom = float(np.dot(shape, shape))
    if denom <= 0:
        continue
    Vinf_best = float(np.dot(vobs, shape) / denom)
    dlogV = float(np.log10(max(Vinf_best, 1e-30) / max(Vinf_baseline, 1e-30)))

    rmse_baseline = safe_rmse(vobs, Vinf_baseline * shape)
    rmse_best = safe_rmse(vobs, Vinf_best * shape)

    # baryonic components
    vgas_abs = np.abs(vgas)
    vdisk_abs = np.abs(vdisk)
    vbul_abs = np.abs(vbul)
    vb = np.sqrt(np.maximum(vb2, 0.0))

    vmax_bar = float(np.max(vb))
    vmax_gas = float(np.max(vgas_abs))
    vmax_disk = float(np.max(vdisk_abs))
    vmax_bul = float(np.max(vbul_abs))

    # radial markers from baryonic curve
    i_peak = int(np.argmax(vb))
    r_peak = float(r[i_peak]) if np.isfinite(vmax_bar) and vmax_bar > 0 else np.nan
    idx_bar = np.where(vb >= 0.5 * vmax_bar)[0] if vmax_bar > 0 else []
    r_bar = float(r[idx_bar[0]]) if len(idx_bar) else np.nan

    # inner baryonic acceleration markers
    gb_peak = float(np.max(gb))
    gb_first = float(gb[0])
    gb_last = float(gb[-1])

    # transport markers
    idx_rt60 = np.where(u >= 0.60 * float(np.max(u)))[0]
    rt60 = float(r[idx_rt60[0]]) if len(idx_rt60) else np.nan

    idx_rt80 = np.where(u >= 0.80 * float(np.max(u)))[0]
    rt80 = float(r[idx_rt80[0]]) if len(idx_rt80) else np.nan

    # outer slopes on last 4 points
    if len(r) >= 4:
        outer_slope_vb2 = float(np.polyfit(
            np.log(np.maximum(r[-4:], 1e-9)),
            np.log(np.maximum(vb2[-4:], 1e-9)),
            1
        )[0])
        outer_slope_gb = float(np.polyfit(
            np.log(np.maximum(r[-4:], 1e-9)),
            np.log(np.maximum(gb[-4:], 1e-9)),
            1
        )[0])
    else:
        outer_slope_vb2 = np.nan
        outer_slope_gb = np.nan

    rows.append({
        "name": name,
        "rmse_baseline": rmse_baseline,
        "rmse_best": rmse_best,
        "Vinf_baseline": Vinf_baseline,
        "Vinf_best": Vinf_best,
        "dlogV_required": dlogV,

        # keep Mb
        "log_Mb_max": float(safe_log10(Mb_max)),

        # pure baryonic proxy candidates
        "log_vbar_max": float(safe_log10(vmax_bar)),
        "log_vgas_max": float(safe_log10(vmax_gas)),
        "log_vdisk_max": float(safe_log10(vmax_disk)),
        "log_vbul_max": float(safe_log10(vmax_bul)),
        "log_vbul_over_vdisk": float(safe_log10(vmax_bul / max(vmax_disk, 1e-30))),
        "log_vbul_over_vbar": float(safe_log10(vmax_bul / max(vmax_bar, 1e-30))),
        "log_vdisk_over_vbar": float(safe_log10(vmax_disk / max(vmax_bar, 1e-30))),
        "log_vgas_over_vbar": float(safe_log10(vmax_gas / max(vmax_bar, 1e-30))),

        "log_r_last_over_Rs": float(safe_log10(r[-1] / max(Rs, 1e-30))),
        "log_Rs": float(safe_log10(Rs)),
        "log_r_last": float(safe_log10(r[-1])),
        "log_rpeak_over_rbar": float(safe_log10(r_peak / max(r_bar, 1e-30))) if np.isfinite(r_peak) and np.isfinite(r_bar) and r_bar > 0 else np.nan,
        "log_rt60_over_rpeak": float(safe_log10(rt60 / max(r_peak, 1e-30))) if np.isfinite(rt60) and np.isfinite(r_peak) and r_peak > 0 else np.nan,
        "log_rt80_over_rpeak": float(safe_log10(rt80 / max(r_peak, 1e-30))) if np.isfinite(rt80) and np.isfinite(r_peak) and r_peak > 0 else np.nan,

        "outer_slope_vb2": outer_slope_vb2,
        "outer_slope_gb": outer_slope_gb,
        "log_gb_peak": float(safe_log10(gb_peak)),
        "log_gb_first": float(safe_log10(gb_first)),
        "log_gb_last": float(safe_log10(gb_last)),
        "log_gb_peak_over_gb_last": float(safe_log10(gb_peak / max(gb_last, 1e-30))),
        "log_gb_first_over_gb_last": float(safe_log10(gb_first / max(gb_last, 1e-30))),
        "log_gb_peak_over_gb_first": float(safe_log10(gb_peak / max(gb_first, 1e-30))),
    })

df = pd.DataFrame(rows).sort_values("dlogV_required", ascending=False).reset_index(drop=True)

print("\n" + "="*100)
print("BARYONIC PROXY TARGET TABLE")
print("="*100)
print(df[[
    "name", "dlogV_required", "log_Mb_max", "log_vbar_max",
    "log_vbul_max", "log_vbul_over_vdisk", "log_r_last_over_Rs",
    "outer_slope_vb2", "log_gb_peak", "log_gb_peak_over_gb_last"
]].to_string(index=False))

# ============================================================
# FEATURE SETS
# ============================================================

baryonic_features = [
    "log_vbar_max",
    "log_vgas_max",
    "log_vdisk_max",
    "log_vbul_max",
    "log_vbul_over_vdisk",
    "log_vbul_over_vbar",
    "log_vdisk_over_vbar",
    "log_vgas_over_vbar",
    "log_r_last_over_Rs",
    "log_Rs",
    "log_r_last",
    "log_rpeak_over_rbar",
    "log_rt60_over_rpeak",
    "log_rt80_over_rpeak",
    "outer_slope_vb2",
    "outer_slope_gb",
    "log_gb_peak",
    "log_gb_first",
    "log_gb_last",
    "log_gb_peak_over_gb_last",
    "log_gb_first_over_gb_last",
    "log_gb_peak_over_gb_first",
]

# ============================================================
# SINGLE-FEATURE LAWS
# ============================================================

single_rows = []
for feat in baryonic_features:
    fit = linfit_1d(df[feat], df["dlogV_required"])
    if fit is None:
        continue
    single_rows.append({
        "feature": feat,
        "corr": corr_1d(df[feat], df["dlogV_required"]),
        **fit
    })

df_single = pd.DataFrame(single_rows).sort_values(["rmse", "r2"], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*100)
print("RANKED BARYONIC-ONLY SINGLE-FEATURE LAWS")
print("="*100)
print(df_single.head(25).to_string(index=False))

# ============================================================
# TWO-FEATURE BARYONIC-ONLY LAWS
# ============================================================

pair_rows = []
for f1, f2 in itertools.combinations(baryonic_features, 2):
    fit = linfit_2d(df[f1], df[f2], df["dlogV_required"])
    if fit is None:
        continue
    pair_rows.append({
        "feature_1": f1,
        "feature_2": f2,
        **fit
    })

df_pair = pd.DataFrame(pair_rows).sort_values(["rmse", "r2"], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*100)
print("RANKED BARYONIC-ONLY TWO-FEATURE LAWS")
print("="*100)
print(df_pair.head(25).to_string(index=False))

# ============================================================
# Mb + proxy laws
# ============================================================

mb_proxy_rows = []
for feat in baryonic_features:
    fit = linfit_2d(df["log_Mb_max"], df[feat], df["dlogV_required"])
    if fit is None:
        continue
    mb_proxy_rows.append({
        "proxy_feature": feat,
        **fit
    })

df_mb_proxy = pd.DataFrame(mb_proxy_rows).sort_values(["rmse", "r2"], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*100)
print("RANKED [log_Mb_max + baryonic proxy] LAWS")
print("="*100)
print(df_mb_proxy.head(25).to_string(index=False))

# ============================================================
# BEST CANDIDATE LAW
# Prefer Mb + baryonic proxy, since that is the physically
# closest replacement of the discovery law.
# ============================================================

if len(df_mb_proxy):
    best = df_mb_proxy.iloc[0].to_dict()
    feat = best["proxy_feature"]
    c0 = best["intercept"]
    c1 = best["coef1"]   # on log_Mb_max
    c2 = best["coef2"]   # on proxy_feature

    df["dlogV_pred_best"] = c0 + c1 * df["log_Mb_max"] + c2 * df[feat]
    df["Vinf_pred_best"] = df["Vinf_baseline"] * (10 ** df["dlogV_pred_best"])
    df["pred_ratio_best"] = df["Vinf_pred_best"] / df["Vinf_best"]
    df["resid_dlogV_best"] = df["dlogV_required"] - df["dlogV_pred_best"]

    print("\n" + "="*100)
    print("BEST CANDIDATE BARYONIC REPLACEMENT LAW")
    print("="*100)
    print(f"dlogV_required = {c0:.6f} + ({c1:.6f})*log_Mb_max + ({c2:.6f})*{feat}")
    print(f"rmse={best['rmse']:.6f}, r2={best['r2']:.6f}")

    print("\nPer-galaxy predicted vs required lift:")
    print(df[[
        "name", "dlogV_required", "dlogV_pred_best",
        "Vinf_baseline", "Vinf_best", "Vinf_pred_best",
        "pred_ratio_best", "resid_dlogV_best"
    ]].to_string(index=False))
else:
    best = None

# ============================================================
# FOCUS OBJECTS
# ============================================================

print("\n" + "="*100)
print("FOCUS OBJECTS")
print("="*100)
focus_df = df[df["name"].isin(FOCUS)].copy()
if len(focus_df):
    cols = [
        "name", "dlogV_required", "log_Mb_max", "log_vbar_max",
        "log_vbul_max", "log_vbul_over_vdisk", "log_r_last_over_Rs",
        "outer_slope_vb2", "log_gb_peak", "log_gb_peak_over_gb_last"
    ]
    if "dlogV_pred_best" in focus_df.columns:
        cols += ["dlogV_pred_best", "resid_dlogV_best"]
    print(focus_df[cols].to_string(index=False))
else:
    print("No focus objects present.")

# ============================================================
# PLOTS
# ============================================================

if len(df_single):
    feat = df_single.iloc[0]["feature"]
    fit = linfit_1d(df[feat], df["dlogV_required"])
    if fit is not None:
        x = np.asarray(df[feat], float)
        y = np.asarray(df["dlogV_required"], float)
        m = np.isfinite(x) & np.isfinite(y)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(x[m], y[m], s=35)
        xx = np.linspace(np.nanmin(x[m]), np.nanmax(x[m]), 200)
        yy = fit["intercept"] + fit["coef"] * xx
        ax.plot(xx, yy, lw=2)
        ax.set_xlabel(feat)
        ax.set_ylabel("dlogV_required")
        ax.set_title(f"Best baryonic single-feature law\nr2={fit['r2']:.3f}, rmse={fit['rmse']:.3f}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, "best_baryonic_single_feature_law.png"), dpi=170)
        plt.close(fig)

if best is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df["dlogV_required"], df["dlogV_pred_best"], s=40)
    lo = min(df["dlogV_required"].min(), df["dlogV_pred_best"].min())
    hi = max(df["dlogV_required"].max(), df["dlogV_pred_best"].max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("Required dlogV")
    ax.set_ylabel("Predicted dlogV")
    ax.set_title(f"Best baryonic replacement law\nr2={best['r2']:.3f}, rmse={best['rmse']:.3f}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "best_baryonic_replacement_required_vs_pred.png"), dpi=170)
    plt.close(fig)

# ============================================================
# SAVE
# ============================================================

target_csv = os.path.join(OUTDIR, "baryonic_proxy_target_table.csv")
single_csv = os.path.join(OUTDIR, "baryonic_proxy_single_feature_laws.csv")
pair_csv = os.path.join(OUTDIR, "baryonic_proxy_two_feature_laws.csv")
mb_proxy_csv = os.path.join(OUTDIR, "baryonic_proxy_mb_plus_proxy_laws.csv")
full_csv = os.path.join(OUTDIR, "baryonic_proxy_full_with_predictions.csv")

df.to_csv(target_csv, index=False)
df_single.to_csv(single_csv, index=False)
df_pair.to_csv(pair_csv, index=False)
df_mb_proxy.to_csv(mb_proxy_csv, index=False)
df.to_csv(full_csv, index=False)

print("\nSaved:")
print(" -", target_csv)
print(" -", single_csv)
print(" -", pair_csv)
print(" -", mb_proxy_csv)
print(" -", full_csv)

print("\nCELL MTS_AMPLITUDE_BARYONIC_PROXY_REPLACEMENT_v1 complete.")

CELL: MTS_AMPLITUDE_BARYONIC_PROXY_REPLACEMENT_v1
Loaded amplitude-limited galaxies: 32

BARYONIC PROXY TARGET TABLE
       name  dlogV_required  log_Mb_max  log_vbar_max  log_vbul_max  log_vbul_over_vdisk  log_r_last_over_Rs  outer_slope_vb2  log_gb_peak  log_gb_peak_over_gb_last
    NGC6789        0.268858    7.785001      1.283652    -30.000000           -30.000000            0.374137         1.218799     2.842094                  0.126049
   UGC11914        0.231600   10.926190      2.388142      2.404714             0.006184            0.943335        -0.724791     5.060946                  1.486302
   UGC06786        0.223589   10.701652      2.243131      2.317311             0.075317            0.961574        -1.059423     4.872526                  2.614870
    NGC0024        0.176305    9.532515      1.720503    -30.000000           -30.000000            0.912045        -1.114163     3.129976                  1.094448
    NGC3521        0.174776   10.725286      2.301751    -

In [ ]:
# ============================================================
# CELL: MTS_DEPTH_CORRECTED_AMPLITUDE_LAW_v1
#
# Purpose:
#   Test theory-shaped amplitude closures suggested by the
#   baryonic proxy search:
#
#     Vinf^4 = G * Mb * a_dagger * Psi(depth)
#
#   where the missing factor Psi is built from baryonic depth /
#   concentration variables, not observed velocity.
#
# Main families tested:
#   A) gb_peak power law
#      Psi = (gb_peak / g_ref)^delta
#
#   B) contrast power law
#      Psi = (gb_peak / gb_last)^delta
#
#   C) mixed depth-scale law
#      Psi = (gb_peak / g_ref)^delta * (Rs / R_ref)^eta
#
#   D) depth+contrast law
#      Psi = (gb_peak / g_ref)^delta * (gb_peak/gb_last)^kappa
#
# Strategy:
#   - keep Claude backbone shape fixed
#   - only alter amplitude closure
#   - evaluate first on the AMPLITUDE_LIMITED subset
#   - then also report effect on the full catalogue
#
# Goal:
#   Find a compact, physically-shaped amplitude correction that
#   materially improves the amplitude-limited class without
#   wrecking the backbone elsewhere.
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_DEPTH_CORRECTED_AMPLITUDE_LAW_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_depth_corrected_amplitude_law_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_depth_corrected_amplitude_law_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

CLASSIFIER_FULL_CSV = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_full.csv"
CLASSIFIER_AMP_CSV  = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_amplitude_limited.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# reference scales for dimensionless depth laws
G_REF = 1000.0   # (km/s)^2 / kpc
R_REF = 1.0      # kpc

# theory-shaped scan grids
DELTA_GRID = [-0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
ETA_GRID   = [-0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15, 0.20]
KAPPA_GRID = [-0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15, 0.20]

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC03205",
    "UGC02487", "NGC2841", "UGC02953", "NGC5985"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def build_backbone_shape(rot):
    r = rot["r"]
    vobs = rot["vobs"]
    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None

    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])

    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None

    shape = 1.0 - np.exp(-u / u_Rs)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_max = float(np.max(Menc))
    Vinf_baseline = float((G_KPC_KMS2_MSun * Mb_max * A_DAGGER) ** 0.25)

    gb_peak = float(np.max(gb))
    gb_last = float(gb[-1])

    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan

    return {
        "r": r,
        "vobs": vobs,
        "shape": shape,
        "Rs": Rs,
        "Mb_max": Mb_max,
        "gb_peak": gb_peak,
        "gb_last": gb_last,
        "Vinf_baseline": Vinf_baseline,
        "Vinf_best": Vinf_best,
        "rmse_baseline": safe_rmse(vobs, Vinf_baseline * shape),
        "rmse_best_amp": safe_rmse(vobs, Vinf_best * shape),
    }

def psi_family(rec, family, delta=0.0, eta=0.0, kappa=0.0):
    gb_peak = max(rec["gb_peak"], 1e-30)
    gb_last = max(rec["gb_last"], 1e-30)
    Rs = max(rec["Rs"], 1e-30)

    if family == "baseline":
        return 1.0

    if family == "gb_peak_power":
        return (gb_peak / G_REF) ** delta

    if family == "contrast_power":
        return (gb_peak / gb_last) ** delta

    if family == "depth_scale":
        return (gb_peak / G_REF) ** delta * (Rs / R_REF) ** eta

    if family == "depth_contrast":
        return (gb_peak / G_REF) ** delta * (gb_peak / gb_last) ** kappa

    raise ValueError(f"Unknown family: {family}")

def eval_family_on_record(rec, family, delta=0.0, eta=0.0, kappa=0.0):
    psi = psi_family(rec, family, delta=delta, eta=eta, kappa=kappa)
    Mb_eff = rec["Mb_max"] * psi
    Vinf = float((G_KPC_KMS2_MSun * Mb_eff * A_DAGGER) ** 0.25)
    vpred = Vinf * rec["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)
    return {
        "psi": float(psi),
        "Mb_eff": float(Mb_eff),
        "Vinf": float(Vinf),
        "rmse": rmse,
    }

# ============================================================
# LOAD CATALOGUE + TARGET SUBSETS
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if not os.path.exists(CLASSIFIER_FULL_CSV):
    raise FileNotFoundError(CLASSIFIER_FULL_CSV)
if not os.path.exists(CLASSIFIER_AMP_CSV):
    raise FileNotFoundError(CLASSIFIER_AMP_CSV)

df_full_cls = pd.read_csv(CLASSIFIER_FULL_CSV)
df_amp_cls  = pd.read_csv(CLASSIFIER_AMP_CSV)

amp_names = set(df_amp_cls["name"].tolist())
all_names = [n for n in df_full_cls["name"].tolist() if n in name_to_path]

cache = {}
for name in all_names:
    rec = build_backbone_shape(read_rotmod(name_to_path[name]))
    if rec is not None:
        cache[name] = rec

all_names = [n for n in all_names if n in cache]
amp_names = [n for n in all_names if n in amp_names]

print(f"Usable full catalogue galaxies : {len(all_names)}")
print(f"Usable amplitude-limited set   : {len(amp_names)}")

# ============================================================
# SCAN THEORY-SHAPED FAMILIES
# ============================================================

summary_rows = []
detail_rows = []

family_specs = [("baseline", 0.0, 0.0, 0.0)]

for delta in DELTA_GRID:
    family_specs.append(("gb_peak_power", delta, 0.0, 0.0))
    family_specs.append(("contrast_power", delta, 0.0, 0.0))

for delta in DELTA_GRID:
    for eta in ETA_GRID:
        family_specs.append(("depth_scale", delta, eta, 0.0))

for delta in DELTA_GRID:
    for kappa in KAPPA_GRID:
        family_specs.append(("depth_contrast", delta, 0.0, kappa))

# dedupe
family_specs = list(dict.fromkeys(family_specs))

for family, delta, eta, kappa in family_specs:
    rmses_amp = []
    rmses_all = []

    for name in all_names:
        rec = cache[name]
        out = eval_family_on_record(rec, family, delta=delta, eta=eta, kappa=kappa)
        rmses_all.append(out["rmse"])
        if name in amp_names:
            rmses_amp.append(out["rmse"])

        detail_rows.append({
            "name": name,
            "subset": "AMPLITUDE_LIMITED" if name in amp_names else "OTHER",
            "family": family,
            "delta": delta,
            "eta": eta,
            "kappa": kappa,
            **out
        })

    summary_rows.append({
        "family": family,
        "delta": delta,
        "eta": eta,
        "kappa": kappa,
        "med_amp": float(np.nanmedian(rmses_amp)),
        "p90_amp": float(np.nanpercentile(rmses_amp, 90)),
        "mean_amp": float(np.nanmean(rmses_amp)),
        "med_all": float(np.nanmedian(rmses_all)),
        "p90_all": float(np.nanpercentile(rmses_all, 90)),
        "mean_all": float(np.nanmean(rmses_all)),
    })
    print(f"[done] {family:15s} delta={delta:>5.2f} eta={eta:>5.2f} kappa={kappa:>5.2f}")

df_summary = pd.DataFrame(summary_rows)
df_detail = pd.DataFrame(detail_rows)

# rank primarily by amplitude-limited subset, then by global catalogue
df_rank = df_summary.sort_values(
    ["med_amp", "p90_amp", "med_all", "p90_all", "mean_amp"],
    ascending=[True, True, True, True, True]
).reset_index(drop=True)

print("\n" + "="*100)
print("DEPTH-CORRECTED AMPLITUDE LAW SUMMARY")
print("="*100)
print(df_rank.head(30).to_string(index=False))

best = df_rank.iloc[0].to_dict()
best_family = best["family"]
best_delta = float(best["delta"])
best_eta = float(best["eta"])
best_kappa = float(best["kappa"])

print("\nBest theory-shaped law:")
for k, v in best.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

# ============================================================
# PER-GALAXY COMPARISON FOR BEST LAW
# ============================================================

comp_rows = []
for name in all_names:
    rec = cache[name]
    base = eval_family_on_record(rec, "baseline", 0.0, 0.0, 0.0)
    best_out = eval_family_on_record(rec, best_family, delta=best_delta, eta=best_eta, kappa=best_kappa)

    comp_rows.append({
        "name": name,
        "subset": "AMPLITUDE_LIMITED" if name in amp_names else "OTHER",
        "rmse_baseline": rec["rmse_baseline"],
        "rmse_best_amp_diag": rec["rmse_best_amp"],
        "rmse_best_theory_law": best_out["rmse"],
        "gain_vs_baseline": rec["rmse_baseline"] - best_out["rmse"],
        "residual_to_best_amp": best_out["rmse"] - rec["rmse_best_amp"],
        "Vinf_baseline": rec["Vinf_baseline"],
        "Vinf_best_amp_diag": rec["Vinf_best"],
        "Vinf_best_theory_law": best_out["Vinf"],
        "gb_peak": rec["gb_peak"],
        "gb_last": rec["gb_last"],
        "Rs": rec["Rs"],
        "psi_best": best_out["psi"],
    })

df_comp = pd.DataFrame(comp_rows)

print("\n" + "="*100)
print("AMPLITUDE-LIMITED PER-GALAXY COMPARISON")
print("="*100)
print(
    df_comp[df_comp["subset"].eq("AMPLITUDE_LIMITED")]
    .sort_values(["gain_vs_baseline", "rmse_best_theory_law"], ascending=[False, True])
    .to_string(index=False)
)

print("\n" + "="*100)
print("FOCUS OBJECTS")
print("="*100)
focus_df = df_comp[df_comp["name"].isin(FOCUS)].copy()
if len(focus_df):
    print(focus_df.sort_values(["subset", "rmse_baseline"], ascending=[True, False]).to_string(index=False))
else:
    print("No focus objects found.")

# ============================================================
# SAVE
# ============================================================

summary_csv = os.path.join(OUTDIR, "depth_corrected_amplitude_law_summary.csv")
detail_csv  = os.path.join(OUTDIR, "depth_corrected_amplitude_law_detail.csv")
comp_csv    = os.path.join(OUTDIR, "depth_corrected_amplitude_law_comparison.csv")

df_rank.to_csv(summary_csv, index=False)
df_detail.to_csv(detail_csv, index=False)
df_comp.to_csv(comp_csv, index=False)

print("\nSaved:")
print(" -", summary_csv)
print(" -", detail_csv)
print(" -", comp_csv)

print("\nCELL MTS_DEPTH_CORRECTED_AMPLITUDE_LAW_v1 complete.")

CELL: MTS_DEPTH_CORRECTED_AMPLITUDE_LAW_v1
Usable full catalogue galaxies : 191
Usable amplitude-limited set   : 32
[done] baseline        delta= 0.00 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta=-0.20 eta= 0.00 kappa= 0.00
[done] contrast_power  delta=-0.20 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta=-0.10 eta= 0.00 kappa= 0.00
[done] contrast_power  delta=-0.10 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta=-0.05 eta= 0.00 kappa= 0.00
[done] contrast_power  delta=-0.05 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta= 0.00 eta= 0.00 kappa= 0.00
[done] contrast_power  delta= 0.00 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta= 0.05 eta= 0.00 kappa= 0.00
[done] contrast_power  delta= 0.05 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta= 0.10 eta= 0.00 kappa= 0.00
[done] contrast_power  delta= 0.10 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta= 0.15 eta= 0.00 kappa= 0.00
[done] contrast_power  delta= 0.15 eta= 0.00 kappa= 0.00
[done] gb_peak_power   delta=

In [ ]:
# ============================================================
# CELL: MTS_DEPTH_CONTRAST_BACKBONE_VALIDATION_v1
#
# Purpose:
#   Validate the full Claude backbone with the new depth-contrast
#   amplitude closure promoted into the main theory candidate.
#
# Baseline backbone:
#   gb = Vbar^2 / r
#   m  = gb^(1/3)
#   u  = ∫ m dr
#   Rs from u(Rs) = frac * u_total
#   shape = 1 - exp(-u/u(Rs))
#
# New promoted amplitude closure:
#   Vinf^4 = G * Mb_max * a_dagger * Psi
#
#   Psi = (gb_peak / G_REF)^delta * (gb_peak / gb_last)^kappa
#
# with promoted candidate values:
#   delta = 0.15
#   kappa = 0.15
#
# Goals:
#   1) rerun the full catalogue under the promoted law
#   2) compare old baseline backbone vs new depth-contrast backbone
#   3) measure whether dynamic-range compression improves
#   4) reclassify galaxies:
#        - BACKBONE_SUCCESS
#        - AMPLITUDE_LIMITED
#        - STRUCTURAL_SURVIVOR
#        - MIXED_FAILURE
#   5) identify who moved class
#
# Notes:
#   - no per-galaxy fitting in the promoted model
#   - best-fit amplitude remains diagnostic only
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_DEPTH_CONTRAST_BACKBONE_VALIDATION_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_depth_contrast_backbone_validation_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_depth_contrast_backbone_validation_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

PREV_CLASSIFIER_FULL = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_full.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# promoted amplitude law
G_REF = 1000.0
DELTA = 0.15
KAPPA = 0.15

# class thresholds
SUCCESS_RMSE_MAX = 20.0
STRUCTURAL_RMSE_BEST_MIN = 60.0
AMP_RATIO_MAX = 0.65
MIXED_RMSE_MIN = 20.0

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC03205",
    "UGC02487", "NGC2841", "UGC02953", "NGC5985",
    "UGC02885", "UGC05253", "UGC06787", "NGC7814"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def classify_row(rmse_backbone, rmse_best):
    if rmse_backbone <= SUCCESS_RMSE_MAX:
        return "BACKBONE_SUCCESS"
    if np.isfinite(rmse_best) and rmse_best >= STRUCTURAL_RMSE_BEST_MIN:
        return "STRUCTURAL_SURVIVOR"
    if np.isfinite(rmse_best) and np.isfinite(rmse_backbone) and rmse_backbone > 0:
        if rmse_best <= AMP_RATIO_MAX * rmse_backbone:
            return "AMPLITUDE_LIMITED"
    if rmse_backbone >= MIXED_RMSE_MIN:
        return "MIXED_FAILURE"
    return "UNCLASSIFIED"

def build_record(rot, use_depth_contrast=False):
    r = rot["r"]
    vobs = rot["vobs"]

    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)

    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None

    target = FRAC * ut
    idx = np.where(u >= target)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])

    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if not np.isfinite(u_Rs) or u_Rs <= 0:
        return None

    shape = 1.0 - np.exp(-u / u_Rs)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_max = float(np.max(Menc))
    gb_peak = float(np.max(gb))
    gb_last = float(gb[-1])

    psi = 1.0
    if use_depth_contrast:
        psi = (max(gb_peak, 1e-30) / G_REF) ** DELTA * (max(gb_peak, 1e-30) / max(gb_last, 1e-30)) ** KAPPA

    Mb_eff = Mb_max * psi
    Vinf = float((G_KPC_KMS2_MSun * Mb_eff * A_DAGGER) ** 0.25)
    vpred = Vinf * shape

    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan
    vpred_best = Vinf_best * shape if np.isfinite(Vinf_best) else np.full_like(vobs, np.nan)

    mp = np.isfinite(vobs) & np.isfinite(vpred) & (vobs > 0) & (vpred > 0)
    if np.sum(mp) >= 3:
        lx = np.log10(vobs[mp])
        ly = np.log10(vpred[mp])
        slope_point = float(np.sum(lx * ly) / np.sum(lx * lx))
    else:
        slope_point = np.nan

    return {
        "r": r,
        "vobs": vobs,
        "shape": shape,
        "Rs": Rs,
        "u_tot": ut,
        "u_Rs": u_Rs,
        "Mb_max": Mb_max,
        "Mb_eff": Mb_eff,
        "gb_peak": gb_peak,
        "gb_last": gb_last,
        "psi": psi,
        "Vinf": Vinf,
        "vpred": vpred,
        "Vinf_best": Vinf_best,
        "vpred_best": vpred_best,
        "rmse_backbone": safe_rmse(vobs, vpred),
        "rmse_best": safe_rmse(vobs, vpred_best),
        "amp_gap": safe_rmse(vobs, vpred) - safe_rmse(vobs, vpred_best),
        "slope_point": slope_point,
        "vmax_obs": float(np.max(vobs)),
        "vmax_pred": float(np.max(vpred)),
    }

def global_scatter_metrics(df_rows, curves_dict):
    all_vobs = []
    all_vpred = []
    for name, rec in curves_dict.items():
        vobs = rec["vobs"]
        vpred = rec["vpred"]
        m = np.isfinite(vobs) & np.isfinite(vpred) & (vobs > 0) & (vpred > 0)
        all_vobs.extend(vobs[m])
        all_vpred.extend(vpred[m])

    vobs_all = np.asarray(all_vobs, float)
    vpred_all = np.asarray(all_vpred, float)

    m = np.isfinite(vobs_all) & np.isfinite(vpred_all) & (vobs_all > 0) & (vpred_all > 0)
    vobs_all = vobs_all[m]
    vpred_all = vpred_all[m]

    lx = np.log10(vobs_all)
    ly = np.log10(vpred_all)

    slope, intercept = np.polyfit(lx, ly, 1)
    slope0 = float(np.sum(lx * ly) / np.sum(lx * lx))

    split_rows = []
    bins = [
        ("LOW_VOBS", 0.0, 80.0),
        ("MID_VOBS", 80.0, 160.0),
        ("HIGH_VOBS", 160.0, np.inf),
    ]
    for label, lo, hi in bins:
        ms = (vobs_all >= lo) & (vobs_all < hi)
        if np.sum(ms) >= 5:
            lxs = np.log10(vobs_all[ms])
            lys = np.log10(vpred_all[ms])
            bs, a_s = np.polyfit(lxs, lys, 1)
            b0s = float(np.sum(lxs * lys) / np.sum(lxs * lxs))
            ratio_med = float(np.median(vpred_all[ms] / vobs_all[ms]))
            bias_lin = float(np.mean(vpred_all[ms] - vobs_all[ms]))
            split_rows.append({
                "subset": label,
                "npts": int(np.sum(ms)),
                "slope": float(bs),
                "slope_origin": b0s,
                "intercept": float(a_s),
                "median_pred_over_obs": ratio_med,
                "mean_pred_minus_obs": bias_lin,
            })

    return {
        "slope": float(slope),
        "intercept": float(intercept),
        "slope_origin": float(slope0),
        "splits": pd.DataFrame(split_rows),
        "vobs_all": vobs_all,
        "vpred_all": vpred_all,
    }

# ============================================================
# LOAD + RUN BOTH MODELS
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if os.path.exists(PREV_CLASSIFIER_FULL):
    prev_df = pd.read_csv(PREV_CLASSIFIER_FULL)
    ordered_names = [n for n in prev_df["name"].tolist() if n in name_to_path]
else:
    ordered_names = sorted(name_to_path.keys())

rows_old = []
rows_new = []
curves_old = {}
curves_new = {}

for name in ordered_names:
    rot = read_rotmod(name_to_path[name])

    rec_old = build_record(rot, use_depth_contrast=False)
    rec_new = build_record(rot, use_depth_contrast=True)

    if rec_old is None or rec_new is None:
        continue

    class_old = classify_row(rec_old["rmse_backbone"], rec_old["rmse_best"])
    class_new = classify_row(rec_new["rmse_backbone"], rec_new["rmse_best"])

    rows_old.append({
        "name": name,
        "class": class_old,
        "rmse_backbone": rec_old["rmse_backbone"],
        "rmse_best": rec_old["rmse_best"],
        "amp_gap": rec_old["amp_gap"],
        "Vinf": rec_old["Vinf"],
        "Mb_eff": rec_old["Mb_eff"],
        "psi": rec_old["psi"],
        "vmax_obs": rec_old["vmax_obs"],
        "vmax_pred": rec_old["vmax_pred"],
        "Rs": rec_old["Rs"],
        "gb_peak": rec_old["gb_peak"],
        "gb_last": rec_old["gb_last"],
    })

    rows_new.append({
        "name": name,
        "class": class_new,
        "rmse_backbone": rec_new["rmse_backbone"],
        "rmse_best": rec_new["rmse_best"],
        "amp_gap": rec_new["amp_gap"],
        "Vinf": rec_new["Vinf"],
        "Mb_eff": rec_new["Mb_eff"],
        "psi": rec_new["psi"],
        "vmax_obs": rec_new["vmax_obs"],
        "vmax_pred": rec_new["vmax_pred"],
        "Rs": rec_new["Rs"],
        "gb_peak": rec_new["gb_peak"],
        "gb_last": rec_new["gb_last"],
    })

    curves_old[name] = rec_old
    curves_new[name] = rec_new

df_old = pd.DataFrame(rows_old)
df_new = pd.DataFrame(rows_new)

# ============================================================
# GLOBAL SUMMARIES
# ============================================================

def summarize_df(df):
    return {
        "N": len(df),
        "med": float(np.nanmedian(df["rmse_backbone"])),
        "p90": float(np.nanpercentile(df["rmse_backbone"], 90)),
        "mean": float(np.nanmean(df["rmse_backbone"])),
        "frac20": float(np.mean(df["rmse_backbone"] <= 20.0)),
        "frac30": float(np.mean(df["rmse_backbone"] <= 30.0)),
        "med_best": float(np.nanmedian(df["rmse_best"])),
        "p90_best": float(np.nanpercentile(df["rmse_best"], 90)),
    }

sum_old = summarize_df(df_old)
sum_new = summarize_df(df_new)

met_old = global_scatter_metrics(df_old, curves_old)
met_new = global_scatter_metrics(df_new, curves_new)

print("\n" + "="*100)
print("FULL CATALOGUE VALIDATION SUMMARY")
print("="*100)
print("OLD BASELINE BACKBONE")
for k, v in sum_old.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nNEW DEPTH-CONTRAST BACKBONE")
for k, v in sum_new.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nDelta (new - old):")
for k in ["med", "p90", "mean", "frac20", "frac30", "med_best", "p90_best"]:
    dv = sum_new[k] - sum_old[k]
    print(f"  {k}: {dv:+.6f}")

print("\n" + "="*100)
print("DYNAMIC-RANGE COMPRESSION COMPARISON")
print("="*100)
print("OLD")
print(f"  slope         : {met_old['slope']:.6f}")
print(f"  intercept     : {met_old['intercept']:.6f}")
print(f"  slope_origin  : {met_old['slope_origin']:.6f}")
if len(met_old["splits"]):
    print(met_old["splits"].to_string(index=False))

print("\nNEW")
print(f"  slope         : {met_new['slope']:.6f}")
print(f"  intercept     : {met_new['intercept']:.6f}")
print(f"  slope_origin  : {met_new['slope_origin']:.6f}")
if len(met_new["splits"]):
    print(met_new["splits"].to_string(index=False))

# ============================================================
# CLASS COMPARISON
# ============================================================

merged = df_old[["name", "class", "rmse_backbone"]].rename(columns={
    "class": "class_old",
    "rmse_backbone": "rmse_old"
}).merge(
    df_new[["name", "class", "rmse_backbone", "psi", "Vinf", "vmax_obs", "vmax_pred"]].rename(columns={
        "class": "class_new",
        "rmse_backbone": "rmse_new",
        "Vinf": "Vinf_new"
    }),
    on="name",
    how="inner"
)

merged["gain"] = merged["rmse_old"] - merged["rmse_new"]
merged["moved"] = merged["class_old"] != merged["class_new"]

print("\n" + "="*100)
print("CLASS COUNTS OLD")
print("="*100)
print(df_old["class"].value_counts(dropna=False).to_string())

print("\n" + "="*100)
print("CLASS COUNTS NEW")
print("="*100)
print(df_new["class"].value_counts(dropna=False).to_string())

print("\n" + "="*100)
print("MOVED CLASS")
print("="*100)
moved_df = merged[merged["moved"]].sort_values(["gain", "rmse_new"], ascending=[False, True])
if len(moved_df):
    print(moved_df[[
        "name", "class_old", "class_new", "rmse_old", "rmse_new", "gain", "psi", "Vinf_new", "vmax_obs", "vmax_pred"
    ]].to_string(index=False))
else:
    print("None")

print("\n" + "="*100)
print("TOP GAINS")
print("="*100)
print(
    merged.sort_values(["gain", "rmse_new"], ascending=[False, True])
    .head(30)[["name", "class_old", "class_new", "rmse_old", "rmse_new", "gain", "psi", "Vinf_new", "vmax_obs", "vmax_pred"]]
    .to_string(index=False)
)

print("\n" + "="*100)
print("FOCUS OBJECTS")
print("="*100)
focus_df = merged[merged["name"].isin(FOCUS)].sort_values(["gain", "rmse_new"], ascending=[False, True])
if len(focus_df):
    print(focus_df[[
        "name", "class_old", "class_new", "rmse_old", "rmse_new", "gain", "psi", "Vinf_new", "vmax_obs", "vmax_pred"
    ]].to_string(index=False))
else:
    print("No focus objects found.")

# ============================================================
# SAVE
# ============================================================

old_csv = os.path.join(OUTDIR, "depth_contrast_validation_old_baseline.csv")
new_csv = os.path.join(OUTDIR, "depth_contrast_validation_new_backbone.csv")
merged_csv = os.path.join(OUTDIR, "depth_contrast_validation_comparison.csv")
old_split_csv = os.path.join(OUTDIR, "depth_contrast_validation_old_velocity_splits.csv")
new_split_csv = os.path.join(OUTDIR, "depth_contrast_validation_new_velocity_splits.csv")

df_old.to_csv(old_csv, index=False)
df_new.to_csv(new_csv, index=False)
merged.to_csv(merged_csv, index=False)
met_old["splits"].to_csv(old_split_csv, index=False)
met_new["splits"].to_csv(new_split_csv, index=False)

# plots
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(met_old["vobs_all"], met_old["vpred_all"], s=4, alpha=0.25, label="old")
ax.scatter(met_new["vobs_all"], met_new["vpred_all"], s=4, alpha=0.25, label="new")
mx = max(np.max(met_old["vobs_all"]), np.max(met_old["vpred_all"]), np.max(met_new["vpred_all"]))
ax.plot([1, mx], [1, mx], "k--", lw=1.5, label="1:1")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Observed velocity [km/s]")
ax.set_ylabel("Predicted velocity [km/s]")
ax.set_title("Old vs new backbone predicted-observed velocity")
ax.legend()
plt.tight_layout()
scatter_png = os.path.join(OUTDIR, "depth_contrast_validation_pred_vs_obs_overlay.png")
plt.savefig(scatter_png, dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df_old["rmse_backbone"], bins=30, alpha=0.5, label="old")
ax.hist(df_new["rmse_backbone"], bins=30, alpha=0.5, label="new")
ax.set_xlabel("RMSE [km/s]")
ax.set_ylabel("Count")
ax.set_title("RMSE distribution: old vs new backbone")
ax.legend()
plt.tight_layout()
hist_png = os.path.join(OUTDIR, "depth_contrast_validation_rmse_hist_overlay.png")
plt.savefig(hist_png, dpi=170)
plt.close(fig)

print("\nSaved:")
print(" -", old_csv)
print(" -", new_csv)
print(" -", merged_csv)
print(" -", old_split_csv)
print(" -", new_split_csv)
print(" -", scatter_png)
print(" -", hist_png)

print("\nCELL MTS_DEPTH_CONTRAST_BACKBONE_VALIDATION_v1 complete.")

CELL: MTS_DEPTH_CONTRAST_BACKBONE_VALIDATION_v1

FULL CATALOGUE VALIDATION SUMMARY
OLD BASELINE BACKBONE
  N: 191
  med: 21.491024
  p90: 119.396845
  mean: 48.283869
  frac20: 0.471204
  frac30: 0.602094
  med_best: 13.608469
  p90_best: 92.606799

NEW DEPTH-CONTRAST BACKBONE
  N: 191
  med: 20.713453
  p90: 105.999487
  mean: 44.924953
  frac20: 0.481675
  frac30: 0.617801
  med_best: 13.608469
  p90_best: 92.606799

Delta (new - old):
  med: -0.777572
  p90: -13.397358
  mean: -3.358916
  frac20: +0.010471
  frac30: +0.015707
  med_best: +0.000000
  p90_best: +0.000000

DYNAMIC-RANGE COMPRESSION COMPARISON
OLD
  slope         : 0.724870
  intercept     : 0.491152
  slope_origin  : 0.957295
   subset  npts    slope  slope_origin  intercept  median_pred_over_obs  mean_pred_minus_obs
 LOW_VOBS   992 0.743096      1.011683   0.453209              1.116091             3.568212
 MID_VOBS   837 0.955579      0.974777   0.039190              0.910166           -10.248711
HIGH_VOBS  1402 0.4

In [ ]:
# ============================================================
# CELL: MTS_GUARDED_DEPTH_CONTRAST_AMPLITUDE_v1
#
# Purpose:
#   Refine the promoted depth-contrast amplitude law so it keeps
#   the high-end gains but reduces collateral overboost on
#   already-good systems.
#
# Current promoted law:
#   Psi_raw = (gb_peak / G_REF)^delta * (gb_peak / gb_last)^kappa
#
# Problem:
#   It improves the catalogue and greatly reduces dynamic-range
#   compression, but it can overboost some already-good systems.
#
# This cell tests guarded versions of Psi:
#
#   A) capped:
#      Psi = min(Psi_raw, psi_cap)
#
#   B) thresholded on contrast:
#      Psi = 1                        if contrast <= c0
#          = Psi_raw                  if contrast > c0
#
#   C) thresholded on gb_peak:
#      Psi = 1                        if gb_peak <= g0
#          = Psi_raw                  if gb_peak > g0
#
#   D) soft-ramped:
#      Psi = 1 + w * (Psi_raw - 1)
#      where w turns on smoothly with depth / contrast
#
# Goal:
#   Find a guarded amplitude law that:
#     - keeps or improves global med / p90
#     - keeps dynamic-range compression healthy
#     - reduces harm to already-good galaxies
#
# Notes:
#   - shape law remains fixed
#   - no per-galaxy fitting
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_GUARDED_DEPTH_CONTRAST_AMPLITUDE_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_guarded_depth_contrast_amplitude_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_guarded_depth_contrast_amplitude_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

PREV_CLASSIFIER_FULL = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_full.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# promoted raw law
G_REF = 1000.0
DELTA = 0.15
KAPPA = 0.15

# scan guards
PSI_CAP_GRID = [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
LOGC0_GRID = [0.2, 0.4, 0.6, 0.8, 1.0]     # contrast threshold in log10(gb_peak/gb_last)
LOGG0_GRID = [3.0, 3.3, 3.6, 4.0, 4.3]     # gb_peak threshold in log10((km/s)^2/kpc)
SOFT_ALPHA_GRID = [0.5, 1.0, 1.5, 2.0]
SOFT_LOGC0_GRID = [0.3, 0.5, 0.7, 0.9]

# class thresholds
SUCCESS_RMSE_MAX = 20.0
STRUCTURAL_RMSE_BEST_MIN = 60.0
AMP_RATIO_MAX = 0.65
MIXED_RMSE_MIN = 20.0

FOCUS = {
    "UGC11914", "ESO563-G021", "NGC5005", "NGC3521",
    "UGC06786", "NGC2903", "UGC11455", "UGC03205",
    "UGC02487", "NGC2841", "UGC02953", "NGC5985",
    "UGC02885", "NGC05253", "UGC06787", "NGC7814",
    "NGC6946", "NGC3198", "NGC4559"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def classify_row(rmse_backbone, rmse_best):
    if rmse_backbone <= SUCCESS_RMSE_MAX:
        return "BACKBONE_SUCCESS"
    if np.isfinite(rmse_best) and rmse_best >= STRUCTURAL_RMSE_BEST_MIN:
        return "STRUCTURAL_SURVIVOR"
    if np.isfinite(rmse_best) and np.isfinite(rmse_backbone) and rmse_backbone > 0:
        if rmse_best <= AMP_RATIO_MAX * rmse_backbone:
            return "AMPLITUDE_LIMITED"
    if rmse_backbone >= MIXED_RMSE_MIN:
        return "MIXED_FAILURE"
    return "UNCLASSIFIED"

def build_record(rot):
    r = rot["r"]
    vobs = rot["vobs"]

    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)
    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None

    idx = np.where(u >= FRAC * ut)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])
    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if u_Rs <= 0:
        return None

    shape = 1.0 - np.exp(-u / u_Rs)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_max = float(np.max(Menc))
    gb_peak = float(np.max(gb))
    gb_last = float(gb[-1])

    Vinf_base = float((G_KPC_KMS2_MSun * Mb_max * A_DAGGER) ** 0.25)

    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan
    vpred_best = Vinf_best * shape if np.isfinite(Vinf_best) else np.full_like(vobs, np.nan)

    return {
        "r": r,
        "vobs": vobs,
        "shape": shape,
        "Rs": Rs,
        "Mb_max": Mb_max,
        "gb_peak": gb_peak,
        "gb_last": gb_last,
        "Vinf_base": Vinf_base,
        "rmse_best": safe_rmse(vobs, vpred_best),
        "Vinf_best": Vinf_best,
        "vmax_obs": float(np.max(vobs)),
    }

def psi_raw(rec):
    return (max(rec["gb_peak"], 1e-30) / G_REF) ** DELTA * (max(rec["gb_peak"], 1e-30) / max(rec["gb_last"], 1e-30)) ** KAPPA

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def guarded_psi(rec, family, p1=0.0, p2=0.0):
    raw = psi_raw(rec)
    if family == "raw":
        return raw
    if family == "capped":
        psi_cap = p1
        return min(raw, psi_cap)
    if family == "threshold_contrast":
        logc = np.log10(max(rec["gb_peak"], 1e-30) / max(rec["gb_last"], 1e-30))
        c0 = p1
        return raw if logc > c0 else 1.0
    if family == "threshold_gbpeak":
        logg = np.log10(max(rec["gb_peak"], 1e-30))
        g0 = p1
        return raw if logg > g0 else 1.0
    if family == "soft_contrast":
        alpha = p1
        c0 = p2
        logc = np.log10(max(rec["gb_peak"], 1e-30) / max(rec["gb_last"], 1e-30))
        w = sigmoid(alpha * (logc - c0))
        return 1.0 + w * (raw - 1.0)
    raise ValueError(f"Unknown family: {family}")

def eval_model(rec, family, p1=0.0, p2=0.0):
    psi = guarded_psi(rec, family, p1, p2)
    Mb_eff = rec["Mb_max"] * psi
    Vinf = float((G_KPC_KMS2_MSun * Mb_eff * A_DAGGER) ** 0.25)
    vpred = Vinf * rec["shape"]
    rmse = safe_rmse(rec["vobs"], vpred)

    m = np.isfinite(rec["vobs"]) & np.isfinite(vpred) & (rec["vobs"] > 0) & (vpred > 0)
    if np.sum(m) >= 3:
        lx = np.log10(rec["vobs"][m])
        ly = np.log10(vpred[m])
        slope0 = float(np.sum(lx * ly) / np.sum(lx * lx))
    else:
        slope0 = np.nan

    return {
        "psi": float(psi),
        "Vinf": float(Vinf),
        "rmse": float(rmse),
        "slope_point": float(slope0),
        "vmax_pred": float(np.max(vpred)),
        "vpred": vpred,
    }

def summarize_rows(rows):
    df = pd.DataFrame(rows)
    return {
        "med": float(np.nanmedian(df["rmse"])),
        "p90": float(np.nanpercentile(df["rmse"], 90)),
        "mean": float(np.nanmean(df["rmse"])),
        "frac20": float(np.mean(df["rmse"] <= 20.0)),
        "frac30": float(np.mean(df["rmse"] <= 30.0)),
    }

# ============================================================
# LOAD CATALOG
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if os.path.exists(PREV_CLASSIFIER_FULL):
    prev_df = pd.read_csv(PREV_CLASSIFIER_FULL)
    ordered_names = [n for n in prev_df["name"].tolist() if n in name_to_path]
else:
    ordered_names = sorted(name_to_path.keys())

cache = {}
for name in ordered_names:
    rec = build_record(read_rotmod(name_to_path[name]))
    if rec is not None:
        cache[name] = rec

names = list(cache.keys())
print(f"Usable catalogue galaxies: {len(names)}")

# baseline old and raw promoted for comparison
scan_specs = [("raw", 0.0, 0.0)]

for cap in PSI_CAP_GRID:
    scan_specs.append(("capped", cap, 0.0))
for c0 in LOGC0_GRID:
    scan_specs.append(("threshold_contrast", c0, 0.0))
for g0 in LOGG0_GRID:
    scan_specs.append(("threshold_gbpeak", g0, 0.0))
for alpha in SOFT_ALPHA_GRID:
    for c0 in SOFT_LOGC0_GRID:
        scan_specs.append(("soft_contrast", alpha, c0))

# ============================================================
# SCAN GUARDED FAMILIES
# ============================================================

summary_rows = []
detail_rows = []

for family, p1, p2 in scan_specs:
    rows = []
    for name in names:
        rec = cache[name]
        out = eval_model(rec, family, p1, p2)
        rows.append({
            "name": name,
            "family": family,
            "p1": p1,
            "p2": p2,
            **out
        })
        detail_rows.append({
            "name": name,
            "family": family,
            "p1": p1,
            "p2": p2,
            **out
        })

    s = summarize_rows(rows)
    # collateral damage metric on already-good baseline galaxies
    collateral = []
    for row in rows:
        name = row["name"]
        base_rmse = cache[name]["rmse_best"]  # not used for ranking, just available
        # use old baseline backbone from cache
        old_rmse = safe_rmse(cache[name]["vobs"], cache[name]["Vinf_base"] * cache[name]["shape"])
        if old_rmse <= 20.0:
            collateral.append(row["rmse"] - old_rmse)
    collateral_med = float(np.nanmedian(collateral)) if len(collateral) else np.nan
    collateral_p90 = float(np.nanpercentile(collateral, 90)) if len(collateral) else np.nan

    summary_rows.append({
        "family": family,
        "p1": p1,
        "p2": p2,
        **s,
        "collateral_med": collateral_med,
        "collateral_p90": collateral_p90,
    })
    print(f"[done] {family:18s} p1={p1:>5.2f} p2={p2:>5.2f}")

df_summary = pd.DataFrame(summary_rows)

# rank by low med/p90 and low collateral
df_rank = df_summary.sort_values(
    ["med", "p90", "collateral_p90", "collateral_med", "mean"],
    ascending=[True, True, True, True, True]
).reset_index(drop=True)

print("\n" + "="*100)
print("GUARDED DEPTH-CONTRAST SUMMARY")
print("="*100)
print(df_rank.head(30).to_string(index=False))

best = df_rank.iloc[0].to_dict()
best_family = best["family"]
best_p1 = float(best["p1"])
best_p2 = float(best["p2"])

print("\nBest guarded law:")
for k, v in best.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

# ============================================================
# PER-GALAXY VALIDATION FOR BEST GUARDED LAW
# ============================================================

comp_rows = []
for name in names:
    rec = cache[name]
    raw = eval_model(rec, "raw", 0.0, 0.0)
    best_out = eval_model(rec, best_family, best_p1, best_p2)

    old_rmse = safe_rmse(rec["vobs"], rec["Vinf_base"] * rec["shape"])

    class_old = classify_row(old_rmse, rec["rmse_best"])
    class_new = classify_row(best_out["rmse"], rec["rmse_best"])

    comp_rows.append({
        "name": name,
        "class_old": class_old,
        "class_new": class_new,
        "rmse_old": old_rmse,
        "rmse_raw": raw["rmse"],
        "rmse_new": best_out["rmse"],
        "gain_vs_old": old_rmse - best_out["rmse"],
        "gain_vs_raw": raw["rmse"] - best_out["rmse"],
        "residual_to_best_amp": best_out["rmse"] - rec["rmse_best"],
        "psi_raw": raw["psi"],
        "psi_new": best_out["psi"],
        "Vinf_old": rec["Vinf_base"],
        "Vinf_raw": raw["Vinf"],
        "Vinf_new": best_out["Vinf"],
        "vmax_obs": rec["vmax_obs"],
        "vmax_pred_new": best_out["vmax_pred"],
        "gb_peak": rec["gb_peak"],
        "gb_last": rec["gb_last"],
    })

df_comp = pd.DataFrame(comp_rows)
moved = df_comp[df_comp["class_old"] != df_comp["class_new"]].sort_values(["gain_vs_old", "rmse_new"], ascending=[False, True])

print("\n" + "="*100)
print("TOP GAINS UNDER BEST GUARDED LAW")
print("="*100)
print(df_comp.sort_values(["gain_vs_old", "rmse_new"], ascending=[False, True]).head(30).to_string(index=False))

print("\n" + "="*100)
print("MOVED CLASS")
print("="*100)
if len(moved):
    print(moved.to_string(index=False))
else:
    print("None")

print("\n" + "="*100)
print("FOCUS OBJECTS")
print("="*100)
focus_df = df_comp[df_comp["name"].isin(FOCUS)].sort_values(["gain_vs_old", "rmse_new"], ascending=[False, True])
if len(focus_df):
    print(focus_df.to_string(index=False))
else:
    print("No focus objects found.")

# ============================================================
# SAVE
# ============================================================

summary_csv = os.path.join(OUTDIR, "guarded_depth_contrast_summary.csv")
detail_csv = os.path.join(OUTDIR, "guarded_depth_contrast_detail.csv")
comp_csv = os.path.join(OUTDIR, "guarded_depth_contrast_comparison.csv")

df_rank.to_csv(summary_csv, index=False)
pd.DataFrame(detail_rows).to_csv(detail_csv, index=False)
df_comp.to_csv(comp_csv, index=False)

print("\nSaved:")
print(" -", summary_csv)
print(" -", detail_csv)
print(" -", comp_csv)

print("\nCELL MTS_GUARDED_DEPTH_CONTRAST_AMPLITUDE_v1 complete.")

CELL: MTS_GUARDED_DEPTH_CONTRAST_AMPLITUDE_v1
Usable catalogue galaxies: 191
[done] raw                p1= 0.00 p2= 0.00
[done] capped             p1= 1.50 p2= 0.00
[done] capped             p1= 2.00 p2= 0.00
[done] capped             p1= 2.50 p2= 0.00
[done] capped             p1= 3.00 p2= 0.00
[done] capped             p1= 4.00 p2= 0.00
[done] capped             p1= 5.00 p2= 0.00
[done] threshold_contrast p1= 0.20 p2= 0.00
[done] threshold_contrast p1= 0.40 p2= 0.00
[done] threshold_contrast p1= 0.60 p2= 0.00
[done] threshold_contrast p1= 0.80 p2= 0.00
[done] threshold_contrast p1= 1.00 p2= 0.00
[done] threshold_gbpeak   p1= 3.00 p2= 0.00
[done] threshold_gbpeak   p1= 3.30 p2= 0.00
[done] threshold_gbpeak   p1= 3.60 p2= 0.00
[done] threshold_gbpeak   p1= 4.00 p2= 0.00
[done] threshold_gbpeak   p1= 4.30 p2= 0.00
[done] soft_contrast      p1= 0.50 p2= 0.30
[done] soft_contrast      p1= 0.50 p2= 0.50
[done] soft_contrast      p1= 0.50 p2= 0.70
[done] soft_contrast      p1= 0.50 p2= 0.90

In [ ]:
# ============================================================
# CELL: MTS_DEPTH_CONTRAST_RESIDUAL_CENSUS_v1
#
# Purpose:
#   Rebuild the backbone classifier / residual census using the
#   RAW depth-contrast amplitude law as the new active baseline.
#
# Active baseline promoted into this census:
#
#   gb = Vbar^2 / r
#   m  = gb^(1/3)
#   u  = ∫ m dr
#   Rs from u(Rs) = frac * u_total
#   shape = 1 - exp(-u/u(Rs))
#
#   Vinf^4 = G * Mb_max * a_dagger * Psi
#
#   Psi = (gb_peak / G_REF)^delta * (gb_peak / gb_last)^kappa
#
# with:
#   delta = 0.15
#   kappa = 0.15
#
# Goals:
#   1) make this the new active no-fit baseline
#   2) reclassify the whole catalogue
#   3) identify the new hard pool
#   4) compare old vs new baseline classes
#   5) produce clean tables for next-session work
#
# Class definitions:
#   BACKBONE_SUCCESS
#   AMPLITUDE_LIMITED
#   STRUCTURAL_SURVIVOR
#   MIXED_FAILURE
#
# Notes:
#   - best-fit amplitude remains diagnostic only
#   - no per-galaxy fitting in the promoted baseline
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore")

print("CELL: MTS_DEPTH_CONTRAST_RESIDUAL_CENSUS_v1")

# ============================================================
# CONFIG
# ============================================================

WORKDIR = "/content/mts_depth_contrast_residual_census_v1"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD")
OUTDIR = "/content/mts_depth_contrast_residual_census_v1_outputs"
os.makedirs(ROTMOD_DIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    "/content/Rotmod_LTG (3).zip",
    "/content/Rotmod_ETG (1).zip",
    "/content/Rotmod_ETG.zip",
]

OLD_CLASSIFIER_FULL = "/content/mts_claude_backbone_classifier_v1_outputs/mts_claude_backbone_classifier_full.csv"

UPS_DISK = 0.5
UPS_BUL  = 0.7
GAMMA    = 1.0 / 3.0
FRAC     = 0.135
A_DAGGER = 3209.0
G_KPC_KMS2_MSun = 4.30091e-6

# promoted raw depth-contrast amplitude law
G_REF = 1000.0
DELTA = 0.15
KAPPA = 0.15

# class thresholds
SUCCESS_RMSE_MAX = 20.0
STRUCTURAL_RMSE_BEST_MIN = 60.0
AMP_RATIO_MAX = 0.65
MIXED_RMSE_MIN = 20.0

# hard-pool reporting threshold
HARD_RMSE_MIN = 40.0

FOCUS = {
    "UGC02487", "NGC2841", "UGC02953", "NGC5985", "UGC05253",
    "UGC06787", "NGC7814", "UGC02885", "UGC11914", "ESO563-G021",
    "NGC5005", "UGC06786", "NGC3521", "UGC11455", "NGC2903",
    "UGC03205", "NGC6946", "NGC3198", "NGC4559"
}

# ============================================================
# HELPERS
# ============================================================

def bootstrap_rotmods():
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if files:
        return files
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(ROTMOD_DIR)
    files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not files:
        raise FileNotFoundError("No ROTMOD .dat files found.")
    return files

def galaxy_name_from_path(fp):
    base = os.path.basename(fp)
    key = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    key = re.sub(r"\.dat$", "", key, flags=re.IGNORECASE)
    return key

def read_rotmod(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", "%", "//")):
                continue
            vals = []
            for p in re.split(r"\s+", s):
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No rows found: {path}")

    mc = max(len(r) for r in rows)
    arr = np.full((len(rows), mc), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r     = arr[:, 0]
    vobs  = arr[:, 1]
    ev    = arr[:, 2] if mc >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if mc >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if mc >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if mc >= 6 else np.zeros_like(vobs)

    m = (
        np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev) &
        np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul) &
        (r > 0) & (ev > 0)
    )
    if not m.any():
        raise ValueError(f"No valid rows in {path}")

    o = np.argsort(r[m])
    return {
        "r": r[m][o],
        "vobs": vobs[m][o],
        "ev": ev[m][o],
        "vgas": vgas[m][o],
        "vdisk": vdisk[m][o],
        "vbul": vbul[m][o],
    }

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m])**2))) if m.sum() else np.nan

def safe_log10(x, floor=1e-30):
    return np.log10(np.maximum(np.asarray(x, dtype=float), floor))

def baryonic_v2(rot):
    return (
        rot["vgas"]  * np.abs(rot["vgas"]) +
        UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]) +
        UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def segment_metrics(r, y_obs, y_pred):
    r = np.asarray(r, float)
    y_obs = np.asarray(y_obs, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(r) & np.isfinite(y_obs) & np.isfinite(y_pred)
    r = r[m]
    y_obs = y_obs[m]
    y_pred = y_pred[m]

    if len(r) < 6:
        return {
            "inner_rmse": np.nan,
            "mid_rmse": np.nan,
            "outer_rmse": np.nan,
            "inner_bias": np.nan,
            "outer_bias": np.nan,
            "peak_r_shift_frac": np.nan,
            "peak_v_err_frac": np.nan,
        }

    q1, q2 = np.quantile(r, [1/3, 2/3])
    m1 = r <= q1
    m2 = (r > q1) & (r <= q2)
    m3 = r > q2

    def rm(mask):
        return float(np.sqrt(np.mean((y_obs[mask] - y_pred[mask])**2))) if mask.sum() else np.nan

    def bias(mask):
        return float(np.mean(y_pred[mask] - y_obs[mask])) if mask.sum() else np.nan

    i_obs = int(np.argmax(y_obs))
    i_pred = int(np.argmax(y_pred))
    peak_r_shift_frac = float((r[i_pred] - r[i_obs]) / max(r[i_obs], 1e-30))
    peak_v_err_frac   = float((y_pred[i_pred] - y_obs[i_obs]) / max(abs(y_obs[i_obs]), 1e-30))

    return {
        "inner_rmse": rm(m1),
        "mid_rmse": rm(m2),
        "outer_rmse": rm(m3),
        "inner_bias": bias(m1),
        "outer_bias": bias(m3),
        "peak_r_shift_frac": peak_r_shift_frac,
        "peak_v_err_frac": peak_v_err_frac,
    }

def classify_row(rmse_backbone, rmse_best):
    if rmse_backbone <= SUCCESS_RMSE_MAX:
        return "BACKBONE_SUCCESS"
    if np.isfinite(rmse_best) and rmse_best >= STRUCTURAL_RMSE_BEST_MIN:
        return "STRUCTURAL_SURVIVOR"
    if np.isfinite(rmse_best) and np.isfinite(rmse_backbone) and rmse_backbone > 0:
        if rmse_best <= AMP_RATIO_MAX * rmse_backbone:
            return "AMPLITUDE_LIMITED"
    if rmse_backbone >= MIXED_RMSE_MIN:
        return "MIXED_FAILURE"
    return "UNCLASSIFIED"

def build_record(rot):
    r = rot["r"]
    vobs = rot["vobs"]

    vb2 = np.maximum(baryonic_v2(rot), 0.0)
    gb = vb2 / np.maximum(r, 1e-30)

    m = np.maximum(gb, 0.0) ** GAMMA
    u = cumulative_trapezoid(m, r, initial=0.0)
    ut = float(np.max(u))
    if not np.isfinite(ut) or ut <= 0:
        return None

    idx = np.where(u >= FRAC * ut)[0]
    if len(idx) == 0:
        return None
    i_rs = int(idx[0])

    Rs = float(r[i_rs])
    u_Rs = float(u[i_rs])
    if u_Rs <= 0:
        return None

    shape = 1.0 - np.exp(-u / u_Rs)

    Menc = r * vb2 / G_KPC_KMS2_MSun
    Mb_max = float(np.max(Menc))
    gb_peak = float(np.max(gb))
    gb_last = float(gb[-1])

    psi = (max(gb_peak, 1e-30) / G_REF) ** DELTA * (max(gb_peak, 1e-30) / max(gb_last, 1e-30)) ** KAPPA
    Mb_eff = Mb_max * psi

    Vinf = float((G_KPC_KMS2_MSun * Mb_eff * A_DAGGER) ** 0.25)
    vpred = Vinf * shape

    denom = float(np.dot(shape, shape))
    Vinf_best = float(np.dot(vobs, shape) / denom) if denom > 0 else np.nan
    vpred_best = Vinf_best * shape if np.isfinite(Vinf_best) else np.full_like(vobs, np.nan)

    vb = np.sqrt(np.maximum(vb2, 0.0))
    vmax_bar = float(np.max(vb))
    i_peak_bar = int(np.argmax(vb))
    r_peak = float(r[i_peak_bar]) if vmax_bar > 0 else np.nan
    idx_bar = np.where(vb >= 0.5 * vmax_bar)[0] if vmax_bar > 0 else []
    r_bar = float(r[idx_bar[0]]) if len(idx_bar) else np.nan

    return {
        "r": r,
        "vobs": vobs,
        "shape": shape,
        "Rs": Rs,
        "u_tot": ut,
        "u_Rs": u_Rs,
        "Mb_max": Mb_max,
        "Mb_eff": Mb_eff,
        "gb_peak": gb_peak,
        "gb_last": gb_last,
        "psi": psi,
        "Vinf": Vinf,
        "vpred": vpred,
        "Vinf_best": Vinf_best,
        "vpred_best": vpred_best,
        "rmse_backbone": safe_rmse(vobs, vpred),
        "rmse_best": safe_rmse(vobs, vpred_best),
        "amp_gap": safe_rmse(vobs, vpred) - safe_rmse(vobs, vpred_best),
        "vmax_obs": float(np.max(vobs)),
        "vmax_pred": float(np.max(vpred)),
        "r_bar": r_bar,
        "r_peak": r_peak,
    }

def residual_bucket(row):
    # light heuristic bucket for quick navigation of remaining hard pool
    if pd.isna(row["inner_rmse"]) or pd.isna(row["outer_rmse"]):
        return "UNCLASSIFIED"

    if row["inner_rmse"] > 1.5 * max(row["outer_rmse"], 1e-9):
        return "INNER_DOMINATED"
    if row["outer_rmse"] > 1.5 * max(row["inner_rmse"], 1e-9):
        return "OUTER_DOMINATED"
    if abs(row["peak_r_shift_frac"]) > 0.35:
        return "TIMING_MISMATCH"
    if row["peak_v_err_frac"] < -0.15:
        return "PEAK_TOO_LOW"
    if row["peak_r_shift_frac"] > 0.15:
        return "PEAK_TOO_LATE"
    return "MIXED"

# ============================================================
# LOAD CATALOGUE
# ============================================================

files = bootstrap_rotmods()
name_to_path = {galaxy_name_from_path(fp): fp for fp in files}

if os.path.exists(OLD_CLASSIFIER_FULL):
    old_df = pd.read_csv(OLD_CLASSIFIER_FULL)
    ordered_names = [n for n in old_df["name"].tolist() if n in name_to_path]
else:
    old_df = None
    ordered_names = sorted(name_to_path.keys())

rows = []
curve_store = {}

for name in ordered_names:
    try:
        rot = read_rotmod(name_to_path[name])
        rec = build_record(rot)
        if rec is None:
            continue

        seg = segment_metrics(rec["r"], rec["vobs"], rec["vpred"])
        cls = classify_row(rec["rmse_backbone"], rec["rmse_best"])

        row = {
            "name": name,
            "class": cls,
            "rmse_backbone": rec["rmse_backbone"],
            "rmse_best": rec["rmse_best"],
            "amp_gap": rec["amp_gap"],
            "Vinf": rec["Vinf"],
            "Mb_max": rec["Mb_max"],
            "Mb_eff": rec["Mb_eff"],
            "psi": rec["psi"],
            "vmax_obs": rec["vmax_obs"],
            "vmax_pred": rec["vmax_pred"],
            "Rs": rec["Rs"],
            "gb_peak": rec["gb_peak"],
            "gb_last": rec["gb_last"],
            "r_bar": rec["r_bar"],
            "r_peak": rec["r_peak"],
            **seg,
        }
        row["residual_bucket"] = residual_bucket(row)

        rows.append(row)
        curve_store[name] = rec

    except Exception as e:
        print(f"[skip] {name}: {e}")

df = pd.DataFrame(rows)

# ============================================================
# GLOBAL SUMMARY
# ============================================================

med = float(np.nanmedian(df["rmse_backbone"]))
p90 = float(np.nanpercentile(df["rmse_backbone"], 90))
mean = float(np.nanmean(df["rmse_backbone"]))
frac20 = float(np.mean(df["rmse_backbone"] <= 20.0))
frac30 = float(np.mean(df["rmse_backbone"] <= 30.0))

print("\n" + "="*100)
print("DEPTH-CONTRAST RESIDUAL CENSUS SUMMARY")
print("="*100)
print(f"N galaxies            : {len(df)}")
print(f"median RMSE           : {med:.6f}")
print(f"p90 RMSE              : {p90:.6f}")
print(f"mean RMSE             : {mean:.6f}")
print(f"fraction <= 20 km/s   : {frac20:.6f}")
print(f"fraction <= 30 km/s   : {frac30:.6f}")

print("\n" + "="*100)
print("CLASS COUNTS")
print("="*100)
print(df["class"].value_counts(dropna=False).to_string())

print("\n" + "="*100)
print("RESIDUAL BUCKET COUNTS")
print("="*100)
print(df["residual_bucket"].value_counts(dropna=False).to_string())

# hard pool
hard_df = df[df["rmse_backbone"] >= HARD_RMSE_MIN].copy()
hard_df = hard_df.sort_values(["rmse_backbone", "rmse_best"], ascending=[False, False])

print("\n" + "="*100)
print("HARD POOL (RMSE >= 40)")
print("="*100)
if len(hard_df):
    print(hard_df[[
        "name", "class", "residual_bucket", "rmse_backbone", "rmse_best", "amp_gap",
        "inner_rmse", "outer_rmse", "peak_r_shift_frac", "peak_v_err_frac",
        "psi", "Vinf", "vmax_obs", "vmax_pred"
    ]].to_string(index=False))
else:
    print("None")

# top successes
good_df = df.sort_values("rmse_backbone", ascending=True).head(25)

print("\n" + "="*100)
print("TOP SUCCESSES")
print("="*100)
print(good_df[[
    "name", "class", "rmse_backbone", "rmse_best", "psi", "Vinf", "vmax_obs", "vmax_pred"
]].to_string(index=False))

# compare to old classifier if available
if old_df is not None and "class" in old_df.columns and "rmse_backbone" in old_df.columns:
    merged = old_df[["name", "class", "rmse_backbone"]].rename(columns={
        "class": "old_class",
        "rmse_backbone": "old_rmse"
    }).merge(
        df[["name", "class", "rmse_backbone", "residual_bucket", "psi", "Vinf", "vmax_obs", "vmax_pred"]].rename(columns={
            "class": "new_class",
            "rmse_backbone": "new_rmse"
        }),
        on="name",
        how="inner"
    )

    merged["gain"] = merged["old_rmse"] - merged["new_rmse"]
    moved = merged[merged["old_class"] != merged["new_class"]].sort_values(["gain", "new_rmse"], ascending=[False, True])

    print("\n" + "="*100)
    print("MOVED CLASS VS OLD BASELINE CLASSIFIER")
    print("="*100)
    if len(moved):
        print(moved[[
            "name", "old_class", "new_class", "old_rmse", "new_rmse", "gain",
            "residual_bucket", "psi", "Vinf", "vmax_obs", "vmax_pred"
        ]].to_string(index=False))
    else:
        print("None")
else:
    merged = None

print("\n" + "="*100)
print("FOCUS OBJECTS")
print("="*100)
focus_df = df[df["name"].isin(FOCUS)].sort_values(["rmse_backbone", "rmse_best"], ascending=[False, False])
if len(focus_df):
    print(focus_df[[
        "name", "class", "residual_bucket", "rmse_backbone", "rmse_best", "amp_gap",
        "inner_rmse", "outer_rmse", "peak_r_shift_frac", "peak_v_err_frac",
        "psi", "Vinf", "vmax_obs", "vmax_pred"
    ]].to_string(index=False))
else:
    print("No focus objects found.")

# ============================================================
# SAVE OUTPUTS
# ============================================================

full_csv = os.path.join(OUTDIR, "depth_contrast_residual_census_full.csv")
hard_csv = os.path.join(OUTDIR, "depth_contrast_residual_census_hard_pool.csv")
good_csv = os.path.join(OUTDIR, "depth_contrast_residual_census_top_successes.csv")
focus_csv = os.path.join(OUTDIR, "depth_contrast_residual_census_focus.csv")

df.to_csv(full_csv, index=False)
hard_df.to_csv(hard_csv, index=False)
good_df.to_csv(good_csv, index=False)
focus_df.to_csv(focus_csv, index=False)

if merged is not None:
    moved_csv = os.path.join(OUTDIR, "depth_contrast_residual_census_vs_old_classifier.csv")
    merged.to_csv(moved_csv, index=False)
else:
    moved_csv = None

# plots
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df["rmse_backbone"], bins=30)
ax.set_xlabel("RMSE [km/s]")
ax.set_ylabel("Count")
ax.set_title("Depth-contrast baseline RMSE distribution")
plt.tight_layout()
hist_png = os.path.join(OUTDIR, "depth_contrast_residual_census_rmse_hist.png")
plt.savefig(hist_png, dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
cls_counts = df["class"].value_counts()
ax.bar(cls_counts.index.astype(str), cls_counts.values)
ax.set_ylabel("Count")
ax.set_title("Depth-contrast baseline class counts")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
class_png = os.path.join(OUTDIR, "depth_contrast_residual_census_class_counts.png")
plt.savefig(class_png, dpi=170)
plt.close(fig)

print("\nSaved:")
print(" -", full_csv)
print(" -", hard_csv)
print(" -", good_csv)
print(" -", focus_csv)
if moved_csv:
    print(" -", moved_csv)
print(" -", hist_png)
print(" -", class_png)

print("\nCELL MTS_DEPTH_CONTRAST_RESIDUAL_CENSUS_v1 complete.")

CELL: MTS_DEPTH_CONTRAST_RESIDUAL_CENSUS_v1

DEPTH-CONTRAST RESIDUAL CENSUS SUMMARY
N galaxies            : 191
median RMSE           : 20.713453
p90 RMSE              : 105.999487
mean RMSE             : 44.924953
fraction <= 20 km/s   : 0.481675
fraction <= 30 km/s   : 0.617801

CLASS COUNTS
class
BACKBONE_SUCCESS       92
STRUCTURAL_SURVIVOR    38
MIXED_FAILURE          36
AMPLITUDE_LIMITED      25

RESIDUAL BUCKET COUNTS
residual_bucket
INNER_DOMINATED    105
UNCLASSIFIED        26
TIMING_MISMATCH     18
MIXED               14
OUTER_DOMINATED     14
PEAK_TOO_LOW        10
PEAK_TOO_LATE        4

HARD POOL (RMSE >= 40)
       name               class residual_bucket  rmse_backbone  rmse_best   amp_gap  inner_rmse  outer_rmse  peak_r_shift_frac  peak_v_err_frac      psi       Vinf  vmax_obs  vmax_pred
    NGC3998 STRUCTURAL_SURVIVOR    UNCLASSIFIED     316.804773 307.591450  9.213323         NaN         NaN                NaN              NaN 4.884594 219.477396     435.0 138.736174
